# Master CrossTab Notebook

**Single notebook replacing 16 individual crosstab files.**  
**Input:** `ML_230426_Exc_V(230426)_2.xlsx`  
**Output:** One Excel file per section, all crosstabs stacked in a single sheet.

---
| Section | Original File | Filter | Output File |
|---------|--------------|--------|-------------|
| 1 | All_MCD_before_exclusion | Full data | `CT_all_before_exclusion.xlsx` |
| 2 | crossTAB_included_excluded_all | Full data | `CT_included_excluded_all.xlsx` |
| 3 | crossTAB_learners_V1 | Deduped by learner | `CT_learners.xlsx` |
| 4 | All_MCD_after_exclusion | Included only | `CT_all_after_exclusion.xlsx` |
| 5 | crossTAB_Only_ANC_before_exclusion | ANC only (all) | `CT_ANC_before_exclusion.xlsx` |
| 6 | crossTAB_PNCL5M_before_exclusion | PNCL5M only (all) | `CT_PNCL5M_before_exclusion.xlsx` |
| 7 | crossTAB_PNCG5M_before_exclusion | PNCG5M only (all) | `CT_PNCG5M_before_exclusion.xlsx` |
| 8 | crossTAB_Only_ANC_after_exclusion_V1 | ANC + Included | `CT_ANC_after_exclusion.xlsx` |
| 9 | crossTAB_Only_ANC_after_exclusion_V1-Copy1 | ANC + Included | `CT_ANC_after_exclusion_copy1.xlsx` |
| 10 | crossTAB_PNCL5M_after_exclusion_V1 | PNCL5M + Included | `CT_PNCL5M_after_exclusion.xlsx` |
| 11 | crossTAB_PNCL5M_after_exclusion_V1-Copy1 | PNCL5M + Included | `CT_PNCL5M_after_exclusion_copy1.xlsx` |
| 12 | crossTAB_PNCG5M_after_exclusion_V1 | PNCG5M + Included | `CT_PNCG5M_after_exclusion.xlsx` |
| 13 | crossTAB_PNCG5M_after_exclusion_V1-Copy1 | PNCG5M + Included | `CT_PNCG5M_after_exclusion_copy1.xlsx` |
| 14 | crossTAB_ANC+PNCL5M_after_exclusion | ANC+PNCL5M + Included | `CT_ANC_PNCL5M_after_exclusion.xlsx` |
| 15 | crossTAB_Proxy_PNC_ANC_col_after_exclusion | PNCL5M+PNCG5M + Included (PNC_2) | `CT_Proxy_PNC_ANC_after_exclusion.xlsx` |
| 16 | All_for_activity_and_outcome_after_exclusion | Multiple subsets | `CT_activity_outcome_after_exclusion.xlsx` |

---
## Section 0 — Config & Imports

In [1]:
import pandas as pd
import numpy as np
import warnings
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import Font, Alignment
import os

warnings.filterwarnings('ignore')

# ==============================================================================
# CONFIGURATION
# ==============================================================================
INPUT_FILE = "JL_140626_Exc_V(170626)_2.xlsx"

OUTPUT_DIR = "C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/outputs"

# NOTE: outputs/ is created below by the mutually-exclusive routing guard
# (only when this is NOT a Jalna run); see should_write_dir().

# ==============================================================================
# STEP 1: LOAD DATA
# ==============================================================================
print("Loading data...")
data1 = pd.read_excel(INPUT_FILE)
print(f"  Loaded: {data1.shape[0]} rows x {data1.shape[1]} columns")

# ==============================================================================
# STEP 2: CORE HELPER FUNCTIONS
# ==============================================================================

def generate_specific_crosstabs(data, crosstab_pairs, row_vars=None,
                                 row_order=None, col_order=None,
                                 row_category_labels=None,
                                 column_category_labels=None):
    crosstab_pairs = apply_geo_level(apply_report_order(add_learner_cat3_pairs(crosstab_pairs)))  # ADD-ONs: 'Learner Category 3' twin pairs, Word-report table order, then district-level 'Blocks' swap (inert at state level)
    cross_tabs = {}

    for row_col, col_col in crosstab_pairs:
        if row_col not in data.columns or col_col not in data.columns:
            continue

        display_name = row_vars.get(row_col, row_col) if row_vars else row_col

        clean_df = data.copy()

        # ---------------- RAW CROSSTAB (NO TOTAL) ----------------
        count_table = pd.crosstab(
            clean_df[row_col],
            clean_df[col_col],
            dropna=True,
            margins=True,
            margins_name="Total"
        )
        if count_table.empty or count_table.shape[1] == 0:
            final_table = pd.DataFrame()
            final_table.index.name = display_name
            cross_tabs[f"{display_name} vs {col_col}"] = final_table
            continue

        # ---------------- APPLY ROW ORDER ----------------
        if row_order and row_col in row_order:
            ordered_rows = (
                [x for x in row_order[row_col] if x in count_table.index]
                + [x for x in count_table.index if x not in row_order[row_col]]
            )
            count_table = count_table.loc[ordered_rows]

        # ---------------- APPLY COLUMN ORDER ----------------
        if col_order and col_col in col_order:
            ordered_cols = (
                [x for x in col_order[col_col] if x in count_table.columns]
                + [x for x in count_table.columns if x not in col_order[col_col]]
            )
            count_table = count_table[ordered_cols]

        # ---------------- PERCENT TABLE ----------------
        percent_table = (
            count_table
            .iloc[:-1, :-1]
            .div(count_table.loc["Total"].iloc[:-1], axis=1)
            .replace([np.inf, -np.inf], 0)
            .fillna(0)
            * 100
        ).round(2)
        
        # Add Total row and column back (Hardcoded to match notebook)

        grand_total = count_table.loc["Total", "Total"]
        if grand_total != 0:
            percent_table["Total"] = (count_table["Total"] / grand_total * 100).round(2)
        else:
            percent_table["Total"] = 0.0

        percent_table.loc["Total"] = 100.0

        # ---------------- DROP z_NA FROM DISPLAY (still counted in Totals) ----------------
        zna_rows = [x for x in count_table.index
                    if ' '.join(str(x).strip().lower().replace('_', ' ').split()) in ('z na', 'zna')]
        zna_cols = [c for c in count_table.columns
                    if ' '.join(str(c).strip().lower().replace('_', ' ').split()) in ('z na', 'zna')]
        if zna_rows or zna_cols:
            count_table = count_table.drop(index=zna_rows, columns=zna_cols)
            percent_table = percent_table.drop(index=zna_rows, columns=zna_cols, errors='ignore')

        # ---------------- LABEL MAPPING ----------------
        if row_category_labels and row_col in row_category_labels:
            count_table.index = [
                row_category_labels[row_col].get(x, x) for x in count_table.index
            ]
            percent_table.index = count_table.index

        if column_category_labels and col_col in column_category_labels:
            count_table.columns = [
                column_category_labels[col_col].get(x, x) for x in count_table.columns
            ]
            percent_table.columns = count_table.columns

        # ---------------- FINAL TABLE ----------------
        final_table = (
            count_table.astype(int).astype(str)
            + " ("
            + percent_table.astype(str)
            + "%)"
        )

        final_table.index.name = display_name
        cross_tabs[f"{display_name} vs {col_col}"] = final_table

    return cross_tabs


def generate_specific_crosstabs_v2(data, crosstab_pairs, row_vars=None,
                                    row_order=None, col_order=None,
                                    row_category_labels=None,
                                    column_category_labels=None):
    crosstab_pairs = apply_geo_level(apply_report_order(add_learner_cat3_pairs(crosstab_pairs)))  # ADD-ONs: 'Learner Category 3' twin pairs, Word-report table order, then district-level 'Blocks' swap (inert at state level)
    cross_tabs = {}
    
    # Exact set of invalids from the notebook
    invalid_vals = {np.nan, None, 'nan', 'null', 'na', 'zna', 'z_na', 'z_NA'}

    for row_col, col_col in crosstab_pairs:
        if row_col not in data.columns or col_col not in data.columns:
            continue

        display_name = row_vars.get(row_col, row_col) if row_vars else row_col

        # ---------------- CLEAN DATA FOR COUNTING ----------------
        clean_df = data.copy()
        clean_df[row_col] = clean_df[row_col].apply(
            lambda x: np.nan if str(x).strip().lower() in invalid_vals else x
        )

        # ---------------- RAW CROSSTAB (NO TOTAL) ----------------
        count_table = pd.crosstab(
            clean_df[row_col],
            clean_df[col_col],
            dropna=True,
            margins=True,
            margins_name="Total"
        )
        if count_table.empty or count_table.shape[1] == 0:
            continue

        # ---------------- APPLY ROW ORDER ----------------
        if row_order and row_col in row_order:
            ordered_rows = (
                [x for x in row_order[row_col] if x in count_table.index]
                + [x for x in count_table.index if x not in row_order[row_col]]
            )
            count_table = count_table.loc[ordered_rows]

        # ---------------- APPLY COLUMN ORDER ----------------
        if col_order and col_col in col_order:
            ordered_cols = (
                [x for x in col_order[col_col] if x in count_table.columns]
                + [x for x in count_table.columns if x not in col_order[col_col]]
            )
            count_table = count_table[ordered_cols]

        # ---------------- PERCENT TABLE ----------------
        percent_table = (
            count_table
            .iloc[:-1, :-1]
            .div(count_table.loc["Total"].iloc[:-1], axis=1)
            .replace([np.inf, -np.inf], 0)
            .fillna(0)
            * 100
        ).round(2)
        
        # Add Total row and column back (Hardcoded to match notebook)
        percent_table.loc["Total"] = 100.0
        percent_table["Total"] = 100.0

        # ---------------- DROP z_NA FROM DISPLAY (still counted in Totals) ----------------
        zna_rows = [x for x in count_table.index
                    if ' '.join(str(x).strip().lower().replace('_', ' ').split()) in ('z na', 'zna')]
        zna_cols = [c for c in count_table.columns
                    if ' '.join(str(c).strip().lower().replace('_', ' ').split()) in ('z na', 'zna')]
        if zna_rows or zna_cols:
            count_table = count_table.drop(index=zna_rows, columns=zna_cols)
            percent_table = percent_table.drop(index=zna_rows, columns=zna_cols, errors='ignore')

        # ---------------- LABEL MAPPING ----------------
        if row_category_labels and row_col in row_category_labels:
            count_table.index = [
                row_category_labels[row_col].get(x, x) for x in count_table.index
            ]
            percent_table.index = count_table.index

        if column_category_labels and col_col in column_category_labels:
            count_table.columns = [
                column_category_labels[col_col].get(x, x) for x in count_table.columns
            ]
            percent_table.columns = count_table.columns

        # ---------------- FINAL TABLE ----------------
        final_table = (
            count_table.astype(int).astype(str)
            + " ("
            + percent_table.astype(str)
            + "%)"
        )

        final_table.index.name = display_name
        cross_tabs[f"{display_name} vs {col_col}"] = final_table

    return cross_tabs


def estimate_column_width(text, font_size=24):
    """Estimate column width for openpyxl formatting."""
    if not text:
        return 0
    base_width = len(str(text)) * 0.9
    scale_factor = font_size / 11
    return base_width * scale_factor + 2


def export_tables_to_single_sheet(tables_dict, file_name, sheet_title="CombinedTables"):
    """
    Export crosstab dict to Excel with Calibri 24pt formatting.
    Matches the exact formatting from all 16 individual scripts.
    """
    import re
    
    def clean_label(label):
        if not isinstance(label, str):
            return label
        
        # If the label represents a relationship (X vs Y), clean each side separately
        if " vs " in label:
            parts = label.split(" vs ")
            return " vs ".join(clean_label(part) for part in parts)
        
        # Strip suffixes like _M, _P, _C, _CR, _YN at the end of the string
        label = re.sub(r'_(?:[mMPpCc]|CR|cr|Cr|YN|yn|Yn)$', '', label)
        
        # Replace underscores with spaces
        label = label.replace('_', ' ')
        
        # Replace multiple spaces with a single space
        label = re.sub(r'\s+', ' ', label)
        
        label = label.strip()
        return DISPLAY_LABEL_RENAMES.get(label, label)  # ADD-ON: display rename ({} -> no-op)

    wb = Workbook()
    ws = wb.active
    ws.title = sheet_title

    current_row = 1

    for table_name, df in tables_dict.items():
        # Clean table name
        table_name = clean_label(table_name)
        
        # Clean dataframe columns, index, and index name
        df = df.copy()
        df.columns = [clean_label(col) for col in df.columns]
        df.index = [clean_label(idx) for idx in df.index]
        if df.index.name:
            df.index.name = clean_label(df.index.name)

        # Add table title (font size 24, bold, and centrally aligned)
        title_cell = ws.cell(row=current_row, column=1, value=table_name)
        title_cell.font = Font(name='Calibri', size=24, bold=True)
        title_cell.alignment = Alignment(horizontal='center')
        current_row += 1


        data_rows = list(dataframe_to_rows(df, index=True, header=True))

        for r_idx, row in enumerate(data_rows):
            for c_idx, value in enumerate(row):
                cell = ws.cell(row=current_row + r_idx, column=c_idx + 1, value=value)

                is_header = r_idx == 0
                is_last_row = r_idx == len(data_rows) - 1
                cell.font = Font(name='Calibri', size=24, bold=is_header or is_last_row)

                # Centrally align headers (r_idx == 0) and row names/index (c_idx == 0)
                if r_idx == 0 or c_idx == 0:
                    cell.alignment = Alignment(horizontal='center')
                else:
                    cell.alignment = Alignment(horizontal='right')


        current_row += len(data_rows) + 2  # space between tables

    # Adjust column widths
    col_max_widths = {}
    for row in ws.iter_rows():
        for cell in row:
            if cell.value:
                col_letter = cell.column_letter
                width = estimate_column_width(cell.value, font_size=24)
                col_max_widths[col_letter] = max(
                    col_max_widths.get(col_letter, 0), width)

    for col_letter, width in col_max_widths.items():
        ws.column_dimensions[col_letter].width = width

    file_name = restamp_output_filename(file_name)  # ADD-ON: dynamic naming from INPUT_FILE

    full_path = os.path.join(OUTPUT_DIR, file_name)

    if not should_write_dir(OUTPUT_DIR):  # ADD-ON: mutually-exclusive Jalna/normal routing
        print(f"  [skip] Jalna run -> normal-folder export not generated: {full_path}")
        return

    wb.save(full_path)
    print(f"  Exported: {full_path} ({len(tables_dict)} tables)")


# ==============================================================================
# ADD-ON CONFIGURATION & DATA STEPS  (situational -- normal pipeline unchanged)
# ==============================================================================
import re as _re_addon

# ---- Analysis-level flag -----------------------------------------------------
# "district" -> single-district project (e.g. Jalna): every crosstab that uses
#               'District' as a row/column variable uses the 'Blocks' variable
#               instead (applies to BOTH output types: column-wise & row-wise %).
# "state"    -> state-level project (e.g. Meghalaya): 'District' kept as-is
#               (default original behaviour, nothing changes).
ANALYSIS_LEVEL = "district"        # <- set to "state" for state-level projects
IS_DISTRICT_LEVEL = ANALYSIS_LEVEL.strip().lower() == "district"
BLOCK_COL = "Blocks"

def apply_geo_level(pairs):
    """District-level projects analyse at Block level: swap 'District' for the
    Blocks variable in every crosstab pair. Inert when ANALYSIS_LEVEL='state'."""
    if not IS_DISTRICT_LEVEL:
        return list(pairs)
    return [(BLOCK_COL if r == "District" else r,
             BLOCK_COL if c == "District" else c) for r, c in pairs]

# ---- Jalna detection (input file name carries the "JL" code) -------------------
IS_JALNA = "JL" in os.path.basename(INPUT_FILE).upper()
print(f"  Analysis level: {ANALYSIS_LEVEL} | Jalna (JL) input detected: {IS_JALNA}")

# ---- Mutually-exclusive output routing (requested Jul 2026) -------------------
# A Jalna (JL) input writes ONLY the Jalna folders (output_jalna / output_jalna_2);
# a non-Jalna input writes ONLY the normal folders (outputs / output_2 -- the
# Jalna pipeline is already skipped for non-JL input). One guard at the single
# export choke point plus the few direct-write sites enforce this; no normal
# crosstab logic is otherwise touched. Set to False to restore the old behaviour
# (a Jalna run also fills outputs / output_2).
MUTUALLY_EXCLUSIVE_OUTPUTS = True
_NORMAL_OUTPUT_DIRS = set()

def _register_normal_dir(d):
    """Record a folder as a normal (non-Jalna) output folder."""
    if d:
        _NORMAL_OUTPUT_DIRS.add(os.path.normpath(str(d)))

def should_write_dir(target_dir):
    """False when a Jalna run targets a normal folder (skip that write / folder),
    keeping the two output sets mutually exclusive. Always True for non-Jalna
    runs and for the Jalna folders."""
    if MUTUALLY_EXCLUSIVE_OUTPUTS and IS_JALNA:
        return os.path.normpath(str(target_dir)) not in _NORMAL_OUTPUT_DIRS
    return True

_register_normal_dir(OUTPUT_DIR)          # outputs/  (output_2/ registered where it is defined)
if should_write_dir(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- Display-label renames (export-time only; {} -> no-op) -------------------
# Requested Jul 2026: show 'visit category' as 'PNC visit category' in every
# exported table (visit counts here are PNC/child-monitoring visits). Applied
# ONLY when a title / axis label is written to Excel; column names, pair lists
# and category orders in the code stay untouched. Empty dict restores the
# original labels byte-for-byte.
DISPLAY_LABEL_RENAMES = {
    "visit category": "PNC visit category",
    # Sir's wording (Jul 2026) for the adoption-visit data gap: verified in the
    # raw extract that every such row HAS birth details (DOB + birth visit)
    # while the entire adoption-visit block (date + Wt/Ht/zscores) is empty.
    "Adoption visit anthropometry data(Wt,Ht,zscore) not available":
        "Birth details available but no adoption details available",
    # Sir's wording (Jul 2026): verified for every row carrying this label that
    # Last Weight/Height and both z-scores are present; only the date is blank.
    "Last visit Date not available":
        "Last visit anthropometry details available but date absent",
}

# ---- Dynamic output naming (derived from INPUT_FILE; naming only) -------------
# Every exported file name carries a dataset code (XX_DDMMYY) and a version
# stamp (V(DDMMYY)). Both are re-stamped with the CURRENT input file's own
# code/version, so outputs are always named after the input they came from
# (works for any district/state: ML / JL / UJ / ...). Applied at export time
# only -- table contents are untouched.
_dyn_base = os.path.basename(str(INPUT_FILE))
_dyn_code = _re_addon.search(r'([A-Za-z]{2}_\d{6})', _dyn_base)
_dyn_ver = _re_addon.search(r'V\((\d{6})\)', _dyn_base)

def restamp_output_filename(fname):
    """Re-stamp the dataset code and version stamp inside an output file name
    with the input file's own (no-op for names that carry neither)."""
    out = str(fname)
    if _dyn_code:
        out = _re_addon.sub(r'[A-Za-z]{2}_\d{6}', _dyn_code.group(1).upper(), out)
    if _dyn_ver:
        out = _re_addon.sub(r'V\(\d{6}\)', f'V({_dyn_ver.group(1)})', out)
    return out

print(f"  Dynamic output naming: code={_dyn_code.group(1) if _dyn_code else 'n/a'}"
      f" | version={'V(' + _dyn_ver.group(1) + ')' if _dyn_ver else 'n/a'}")

# ---- Word-report table order (requested Jul 2026) ------------------------------
# Inside EVERY exported sheet the crosstab tables follow the order in which
# they appear in the report Word file ('HST MP report state level
# descriptives'). Pairs the report does not contain keep their existing
# relative order and are written after the matched ones. Only the ORDER of the
# tables inside a sheet changes -- each table's content stays identical.
REPORT_TABLE_ORDER = [
    ('Department', 'District'),
    ('Role Group', 'District'),
    ('Learner Category 2', 'District'),
    ('Learner Category 2', 'Role Group'),
    ('Primary Exclusion PNC 2', 'District'),
    ('Primary Exclusion PNC 2', 'Department'),
    ('Primary Exclusion PNC 2', 'Learner Category 2'),
    ('Department', 'Learner Category 2'),
    ('Total Adoptions group', 'District'),
    ('Type of adoption', 'District'),
    ('Mother Age Group', 'District'),
    ("Is this the mother's first pregnancy?_C", 'Mother Age Group'),
    ("Is this the mother's first pregnancy?_C", 'District'),
    ("Mother's education level_M", 'District'),
    ("Color of mother's family ration card_M", 'District'),
    ("Mother's family type_M", 'District'),
    ("Mother's social category_M", 'District'),
    ("Color of mother's family ration card_M", "Mother's social category_M"),
    ('Location of Delivery category', 'District'),
    ('Location of Delivery category', "Mother's education level_M"),
    ('Location of Delivery category', "Color of mother's family ration card_M"),
    ("Mother's education level_M", "Color of mother's family ration card_M"),
    ("Mother's education level_M", "Mother's social category_M"),
    ("Mother's family type_M", "Color of mother's family ration card_M"),
    ('Method of delivery_C', 'District'),
    ('Method of delivery_C', "Mother's education level_M"),
    ('Method of delivery_C', "Color of mother's family ration card_M"),
    ('Method of delivery_C', "Mother's social category_M"),
    ('birthweight category 2', 'District'),
    ('birthweight category 2', 'Type of adoption'),
    ('birthweight category 2', 'Mother Age Group'),
    ('birthweight category 2', "Mother's education level_M"),
    ('birthweight category 2', "Color of mother's family ration card_M"),
    ('birthweight category 2', "Mother's social category_M"),
    ('birthweight category 2', 'ANC protein category'),
    ('birthweight group', 'ANC protein category'),
    ('ANC 60d 3visit', 'Learner Category 2'),
    ('birthweight category 2', 'Learner Category 2'),
    ('birthweight group', 'Learner Category 2'),
    ('birthweight category 2', 'ANC 60d 3visit'),
    ('birthweight group', 'ANC 60d 3visit'),
    ('adoption age classification', 'District'),
    ('adoption age classification', 'Role Group'),
    ('adoption age classification', 'Learner Category 2'),
    ('visit category', 'District'),
    ('visit category', 'Role Group'),
    ('visit category', 'Learner Category 2'),
    ('visit category', 'adoption age classification'),
    ('visit category', 'adoption duration baby'),
    ('visit category', 'birthweight category 2'),
    ('bf assessment category', 'District'),
    ('bf assessment category', 'Role Group'),
    ('bf assessment category', 'Learner Category 2'),
    ('bf assessment category', 'adoption age classification'),
    ('bf assessment category', 'adoption duration baby'),
    ('bf assessment category', 'birthweight category 2'),
    ('bf assessment category', 'visit category'),
    ('visit category', 'birthweight group'),
    ('bf assessment category', 'birthweight group'),
    ('bf assessment category 2', 'District'),
    ('bf assessment category 2', 'Role Group'),
    ('bf assessment category 2', 'Learner Category 2'),
    ('bf assessment category 2', 'adoption age classification'),
    ('bf assessment category 2', 'adoption duration baby'),
    ('bf assessment category 2', 'visit category'),
    ('cf assessment category 2', 'District'),
    ('cf assessment category 2', 'Role Group'),
    ('cf assessment category 2', 'Learner Category 2'),
    ('cf assessment category 2', 'adoption age classification'),
    ('cf assessment category 2', 'adoption duration baby'),
    ('cf assessment category 2', 'visit category'),
    ('adoption age classification', 'Type of adoption'),
    ('visit category', 'Type of adoption'),
]

_REPORT_ORDER_INDEX = {p: i for i, p in enumerate(REPORT_TABLE_ORDER)}

def _report_order_name(name):
    """Normalise a variable name for report-order matching ('Blocks' stands in
    for 'District' at district level; Jalna twin columns map to their base)."""
    n = str(name)
    if n == BLOCK_COL:
        return 'District'
    if n.endswith(' jalna'):
        n = n[:-len(' jalna')]
    if n == LC3_COL:
        n = LC2_COL   # LC3 twin sorts with its LC2 original; the stable sort keeps it right after
    return n

def apply_report_order(pairs):
    """Stable-order crosstab pairs by the Word-report sequence: matched pairs
    first (in report order), everything else after in its original order."""
    pairs = list(pairs)
    def _pos(rc):
        return _REPORT_ORDER_INDEX.get((_report_order_name(rc[0]),
                                        _report_order_name(rc[1])))
    matched = [p for p in pairs if _pos(p) is not None]
    unmatched = [p for p in pairs if _pos(p) is None]
    matched.sort(key=_pos)
    return matched + unmatched

# ---- ADD-ON: 'Learner Category 3' twin crosstabs (requested Jul 2026) ----------
# The derived sheet now carries 'Learner Category 3'. Every crosstab pair that
# uses 'Learner Category 2' (as row OR column) is generated a SECOND time with
# 'Learner Category 3' in its place -- same partner variable, same percentage
# direction, ordering and formatting. The twin pair is inserted straight after
# its original, and _report_order_name() maps LC3 to LC2 so the twin table also
# lands right after its original in the Word-report table order. Covers every
# section and every pipeline (column-wise, row-wise, Jalna) for any district/
# state. Inert when the input has no 'Learner Category 3' column (missing
# columns are skipped by every generator) or when LEARNER_CAT3_TWIN = False.
LEARNER_CAT3_TWIN = True
LC2_COL = "Learner Category 2"
LC3_COL = "Learner Category 3"

def add_learner_cat3_pairs(pairs):
    """After every pair involving 'Learner Category 2', insert the same pair
    with 'Learner Category 3'. No-op when LEARNER_CAT3_TWIN is False."""
    if not LEARNER_CAT3_TWIN:
        return list(pairs)
    out = []
    for r, c in pairs:
        out.append((r, c))
        if LC2_COL in (r, c):
            out.append((LC3_COL if r == LC2_COL else r,
                        LC3_COL if c == LC2_COL else c))
    return out

JALNA_COL_MAP = {
    "Total Adoptions group": "Total Adoptions group jalna",
    "Total Adoptions group after exclusion": "Total Adoptions group after exclusion jalna",
}

def jalnify_pairs(pairs):
    """Swap the two adoption-group columns for their Jalna twins in every pair."""
    return [(JALNA_COL_MAP.get(r, r), JALNA_COL_MAP.get(c, c)) for r, c in pairs]

# ---- ADD-ON DATA STEP 1: merge Medical Officers into 'Other' --------------------
# (requested for Jalna AND for all other districts/states)
if 'Role Group' in data1.columns:
    _mo_mask = (data1['Role Group'].astype(str).str.strip().str.upper()
                .isin(['MO', 'MEDICAL OFFICER']))
    print(f"  Role Group: merging {int(_mo_mask.sum())} 'MO' (Medical Officer) row(s) into 'Other'")
    data1.loc[_mo_mask, 'Role Group'] = 'Other'

# ---- Unify the catch-all bucket to 'Other' (02-08-2026) -----------------------
# The Role sheet renamed the group 'Others' -> 'Other'. Older derived files (or
# any future regression) may still carry 'Others'; left alone that shows up as a
# SECOND separate row/column beside 'Other' in every table. Merge it here, right
# after load, so the split can never appear regardless of the input's vintage.
for _uo_col in ['Role Group', 'Learner Category 3']:
    if _uo_col in data1.columns:
        _uo = data1[_uo_col].astype(str).str.strip().eq('Others')
        if _uo.any():
            print(f"  Unified {int(_uo.sum())} 'Others' value(s) -> 'Other' in '{_uo_col}'")
            data1.loc[_uo, _uo_col] = 'Other'

# ---- ADD-ON DATA STEP 2: derive the learner 'Blocks' variable -------------------
# Learner block = PHC taluka from the case-registration sheet ('Phc Taluka_CR'),
# normalised (case / suffix variants like 'JALNA', 'AMBAD-1', 'JALNA-2 ' cleaned).
# Spelling variants of the SAME block, confirmed 29-07-2026: the raw exports
# write 'Ghatia' and 'Ghatiya' for one block and 'Khacharod' and 'Khachrod' for
# another, and MASD / DOCS each use a different one again. Left alone they show
# up as separate columns and split the same block's learners in two. The map
# below is applied inside _normalise_block, so EVERY source -- case data, the
# MASD fill and the DOCS fill -- is canonicalised the same way before anything
# is counted. Keys are matched after the existing strip/Title-case step.
# Canonical labels chosen to match the actual Ujjain tehsil names, which are
# also the majority spelling in the case data.
# Add a pair here if another district shows the same problem; empty the dict to
# restore the previous behaviour exactly.
BLOCK_NAME_ALIASES = {
    'Ghatia': 'Ghatiya',
    'Khachrod': 'Khacharod',
    # 03-08-2026 (project lead): rural sub-area folded into the Ujjain block
    'Ujjain - Rural': 'Ujjain',
    'Ujjain-Rural': 'Ujjain',
    'Ujjain Rural': 'Ujjain',
}

def _normalise_block(v):
    if pd.isna(v):
        return 'z_NA'
    s = str(v).strip()
    if s == 'z_NA':                                # already normalised
        return s
    s = _re_addon.sub(r'[\s\-_]*\d+$', '', s)      # strip trailing '-1' / '-2'
    s = _re_addon.sub(r'\s+', ' ', s).strip().title()
    s = BLOCK_NAME_ALIASES.get(s, s)               # merge spelling variants
    return s if s else 'z_NA'

# 29-07-2026: the derived sheet now builds and fills 'Blocks' itself (stage 2),
# so plots made from the derived sheet match these tables. When that column is
# present it is the single source of truth and is used as-is -- the derivation
# and BOTH gap-fills below are skipped. The old path is kept for older derived
# files that predate the change.
BLOCKS_FROM_DERIVED = (BLOCK_COL in data1.columns
                       and not data1[BLOCK_COL].astype(str).eq('z_NA').all())
if BLOCKS_FROM_DERIVED:
    data1[BLOCK_COL] = data1[BLOCK_COL].apply(_normalise_block)
    print(f"  '{BLOCK_COL}' taken from the derived sheet (built and filled in stage 2): "
          f"{dict(data1[BLOCK_COL].value_counts())}")
elif 'Phc Taluka_CR' in data1.columns:
    data1[BLOCK_COL] = data1['Phc Taluka_CR'].apply(_normalise_block)
    print(f"  Derived '{BLOCK_COL}' from 'Phc Taluka_CR': "
          f"{dict(data1[BLOCK_COL].value_counts())}")
else:
    data1[BLOCK_COL] = 'z_NA'


# ---- ADD-ON DATA STEP 2b: fill missing blocks from the MASD combined sheet ----
# Requested 29-07-2026. Case registration leaves 'Phc Taluka_CR' blank for a
# sizeable share of cases, so those cases had no block (BLOCK_COL = 'z_NA') and
# were invisible in every block-wise table while still sitting inside the Total.
# The MASD combined sheet (MASD_<District>_<DDMMYY>_combined.xlsx) carries the
# Block/Taluk of every learner keyed on 'User Account ID', so the missing blocks
# are looked up from it here, learner by learner. Only rows whose block is
# MISSING are touched -- a block already present in the case data is never
# overwritten.
# The sheet is a REQUIRED pipeline input: the master pipeline injects its path
# as MASD_BLOCK_SHEET; run standalone, it is auto-discovered next to INPUT_FILE
# (same naming scheme for every district: Ujjain / Jalna / Meghalaya).
# Set FILL_BLOCKS_FROM_MASD = False to restore the previous behaviour exactly.
FILL_BLOCKS_FROM_MASD = True
MASD_ID_COL = 'User Account ID'
MASD_BLOCK_COL_NAME = 'Block/Taluk'
MASD_DISTRICT_BY_CODE = {'UJ': 'Ujjain', 'JL': 'Jalna', 'ML': 'Meghalaya'}

MASD_BLOCK_SHEET = globals().get('MASD_BLOCK_SHEET', None)   # injected by master_pipeline
MASD_FILL_STATS = {}

if BLOCKS_FROM_DERIVED:
    print('  MASD block fill skipped -> blocks already filled in the derived sheet (stage 2)')
elif not FILL_BLOCKS_FROM_MASD:
    print('  FILL_BLOCKS_FROM_MASD is False -> missing blocks left as z_NA')
else:
    import glob as _glob_masd
    # ---- locate the sheet -----------------------------------------------------
    if not MASD_BLOCK_SHEET or not os.path.exists(str(MASD_BLOCK_SHEET)):
        _msd_code = _re_addon.search(r'([A-Za-z]{2})_\d{6}', os.path.basename(str(INPUT_FILE)))
        _msd_dist = MASD_DISTRICT_BY_CODE.get(_msd_code.group(1).upper()) if _msd_code else None
        _msd_in = os.path.dirname(os.path.abspath(str(INPUT_FILE)))
        _msd_dirs = [_msd_in, os.path.dirname(_msd_in),
                     os.path.join(os.path.dirname(_msd_in), 'Inputs'), os.getcwd()]
        _msd_hits = []
        for _d in _msd_dirs:
            if _msd_dist:
                _msd_hits += _glob_masd.glob(os.path.join(_d, f'MASD_{_msd_dist}_*_combined.xlsx'))
        _msd_hits = [h for h in _msd_hits if os.path.isfile(h)]
        MASD_BLOCK_SHEET = max(_msd_hits, key=os.path.getmtime) if _msd_hits else None

    if not MASD_BLOCK_SHEET:
        print('  !! MASD combined sheet NOT FOUND -> missing blocks left as z_NA.'
              ' Place MASD_<District>_<DDMMYY>_combined.xlsx in the Inputs folder.')
    else:
        print(f'  MASD block sheet: {MASD_BLOCK_SHEET}')
        _msd = pd.read_excel(MASD_BLOCK_SHEET)
        if MASD_ID_COL not in _msd.columns or MASD_BLOCK_COL_NAME not in _msd.columns:
            print(f"  !! MASD sheet lacks '{MASD_ID_COL}' / '{MASD_BLOCK_COL_NAME}'"
                  f" (has: {list(_msd.columns)[:8]}...) -> fill skipped")
        else:
            def _msd_key(s):
                return (s.astype(str).str.strip()
                        .str.replace(r'\.0$', '', regex=True))

            _msd_lut = {}
            for _k, _v in zip(_msd_key(_msd[MASD_ID_COL]), _msd[MASD_BLOCK_COL_NAME]):
                if isinstance(_v, str) and _v.strip():
                    _msd_lut[_k] = _normalise_block(_v)
            print(f'  MASD sheet: {len(_msd)} learner row(s),'
                  f' {len(_msd_lut)} with a usable {MASD_BLOCK_COL_NAME}')

            _msd_idcol = 'User Acc ID all' if 'User Acc ID all' in data1.columns else None
            if _msd_idcol is None:
                print("  !! 'User Acc ID all' not in the data -> MASD fill skipped")
            else:
                _msd_miss = data1[BLOCK_COL].astype(str).eq('z_NA')
                _msd_rows0 = int(_msd_miss.sum())
                _msd_lrn0 = data1.loc[_msd_miss, _msd_idcol].astype(str).nunique()
                # MASD is a GAP-FILLER ONLY. It must never introduce a block
                # label that the case data did not already use (its spelling can
                # differ, e.g. 'Ghatia' vs 'Ghatiya'), so the block categories in
                # every table stay exactly the ones that existed before MASD
                # arrived. A proposed label that is unknown to the case data is
                # REJECTED and the case is left as z_NA rather than creating a
                # new column. Existing (non-missing) blocks are never touched.
                MASD_ALLOW_NEW_BLOCKS = False
                _msd_known = set(data1.loc[~_msd_miss, BLOCK_COL].astype(str).unique())
                _msd_found = (data1.loc[_msd_miss, _msd_idcol]
                              .astype(str).str.strip()
                              .str.replace(r'\.0$', '', regex=True).map(_msd_lut))
                if not MASD_ALLOW_NEW_BLOCKS:
                    _msd_new = _msd_found[_msd_found.notna() & ~_msd_found.isin(_msd_known)]
                    if len(_msd_new):
                        print(f'  MASD proposed a block label the case data never used ->'
                              f' REJECTED, left as z_NA ({len(_msd_new)} row(s)):'
                              f' {dict(_msd_new.value_counts())}')
                        _msd_found = _msd_found.where(_msd_found.isin(_msd_known))
                    else:
                        print(f'  All MASD block labels already exist in the case data'
                              f' -> no new block category introduced')
                _msd_hit = _msd_found.notna()
                data1.loc[_msd_found[_msd_hit].index, BLOCK_COL] = _msd_found[_msd_hit]

                _msd_left = data1[BLOCK_COL].astype(str).eq('z_NA')
                _msd_rows1 = int(_msd_left.sum())
                _msd_lrn1 = data1.loc[_msd_left, _msd_idcol].astype(str).nunique()
                MASD_FILL_STATS = {
                    'rows missing before': _msd_rows0,
                    'learners missing before': int(_msd_lrn0),
                    'rows filled': _msd_rows0 - _msd_rows1,
                    'learners resolved': int(_msd_lrn0) - int(_msd_lrn1),
                    'rows still missing': _msd_rows1,
                    'learners still missing': int(_msd_lrn1),
                }
                print(f'  Block fill from MASD: {_msd_rows0 - _msd_rows1} of {_msd_rows0}'
                      f' missing case row(s) filled'
                      f' ({int(_msd_lrn0) - int(_msd_lrn1)} of {int(_msd_lrn0)} learner(s) resolved)')
                if _msd_rows1:
                    print(f'  STILL missing a block: {_msd_rows1} case row(s),'
                          f' {_msd_lrn1} learner(s) -- not present in the MASD sheet'
                          f' or blank there')
                if _msd_rows0 - _msd_rows1:
                    print(f'  Filled block values: '
                          f'{dict(_msd_found[_msd_hit].value_counts())}')
                print(f'  {BLOCK_COL} values after fill: {dict(data1[BLOCK_COL].value_counts())}')

# ---- ONE-TIME DOCS BLOCK FILL (29-07-2026) ------------------------------------
# Second gap-filler, run AFTER the MASD fill for the cases MASD could not
# resolve (learner absent from MASD, or blank Block/Taluk there). Source is the
# DOCS skill-refreshment sheet, prepared as Inputs\DOCS_onetime_fill_<District>.xlsx.
# Same rules as MASD: fills ONLY where the block is still missing, never
# overwrites, and never introduces a block label the case data does not already
# use. If the file is absent this block does nothing, so later datasets are
# unaffected. Set DOCS_ONE_TIME_BLOCK_FILL = False to disable it outright.
DOCS_ONE_TIME_BLOCK_FILL = True
DOCS_BLOCK_FILL_STATS = {}

if BLOCKS_FROM_DERIVED:
    print('  DOCS block fill skipped -> blocks already filled in the derived sheet (stage 2)')
elif not DOCS_ONE_TIME_BLOCK_FILL:
    print('  DOCS_ONE_TIME_BLOCK_FILL is False -> no second block fill')
elif not IS_DISTRICT_LEVEL:
    print('  DOCS block fill: not a district-level run -> skipped')
else:
    import glob as _glob_db
    # district of THIS dataset -- never fall back to a wildcard, the other
    # district's DOCS sheet lives in the same folder
    _dc = _re_addon.search(r'([A-Za-z]{2})_\d{6}', os.path.basename(str(INPUT_FILE)))
    _msd_dist_docs = MASD_DISTRICT_BY_CODE.get(_dc.group(1).upper()) if _dc else None
    if not _msd_dist_docs:
        print('  District could not be determined from INPUT_FILE ->'
              ' DOCS block fill skipped (never guess the district)')
    _db_dirs = [os.path.dirname(os.path.abspath(str(INPUT_FILE))),
                os.path.dirname(os.path.dirname(os.path.abspath(str(INPUT_FILE)))),
                os.getcwd()]
    if MASD_BLOCK_SHEET:
        _db_dirs.insert(0, os.path.dirname(os.path.abspath(str(MASD_BLOCK_SHEET))))
    _db_hits = []
    for _d in _db_dirs:
        if _msd_dist_docs:
            _db_hits += _glob_db.glob(
                os.path.join(_d, f'DOCS_onetime_fill_{_msd_dist_docs}.xlsx'))
    _db_hits = [h for h in _db_hits if os.path.isfile(h)]
    if not _db_hits:
        print('  DOCS one-time fill sheet not found -> second block fill skipped'
              ' (normal for a new dataset)')
    elif 'User Acc ID all' not in data1.columns:
        print("  DOCS block fill: 'User Acc ID all' not present -> skipped")
    else:
        _db_path = max(_db_hits, key=os.path.getmtime)
        _db = pd.read_excel(_db_path)
        _db_lut = {str(k).strip(): _normalise_block(v)
                   for k, v in zip(_db['User Acc ID'], _db['Block'])
                   if isinstance(v, str) and v.strip()}
        _db_miss = data1[BLOCK_COL].astype(str).eq('z_NA')
        _db_rows0 = int(_db_miss.sum())
        _db_lrn0 = data1.loc[_db_miss, 'User Acc ID all'].astype(str).nunique()
        _db_known = set(data1.loc[~_db_miss, BLOCK_COL].astype(str).unique())
        _db_found = (data1.loc[_db_miss, 'User Acc ID all'].astype(str).str.strip()
                     .str.replace(r'\.0$', '', regex=True).map(_db_lut))
        if not MASD_ALLOW_NEW_BLOCKS:
            _db_new = _db_found[_db_found.notna() & ~_db_found.isin(_db_known)]
            if len(_db_new):
                print(f'  DOCS proposed a block label the case data never used ->'
                      f' REJECTED ({len(_db_new)} row(s)): {dict(_db_new.value_counts())}')
                _db_found = _db_found.where(_db_found.isin(_db_known))
        _db_hit = _db_found.notna()
        data1.loc[_db_found[_db_hit].index, BLOCK_COL] = _db_found[_db_hit]
        _db_left = data1[BLOCK_COL].astype(str).eq('z_NA')
        _db_rows1 = int(_db_left.sum())
        _db_lrn1 = data1.loc[_db_left, 'User Acc ID all'].astype(str).nunique()
        DOCS_BLOCK_FILL_STATS = {
            'rows missing before DOCS': _db_rows0,
            'learners missing before DOCS': int(_db_lrn0),
            'rows filled by DOCS': _db_rows0 - _db_rows1,
            'learners resolved by DOCS': int(_db_lrn0) - int(_db_lrn1),
            'rows still missing': _db_rows1,
            'learners still missing': int(_db_lrn1),
        }
        print(f'  DOCS block fill from {os.path.basename(_db_path)}:'
              f' {_db_rows0 - _db_rows1} of {_db_rows0} remaining case row(s) filled'
              f' ({int(_db_lrn0) - int(_db_lrn1)} of {int(_db_lrn0)} learner(s) resolved)')
        print(f'  After BOTH fills -> still missing: {_db_rows1} case row(s),'
              f' {_db_lrn1} learner(s)')
        print(f'  {BLOCK_COL} values after DOCS fill: {dict(data1[BLOCK_COL].value_counts())}')


# ---- Pre-analysis removal via the derived sheet's reason column (02-08-2026) --
# When the derived sheet carries 'Reason for pre-analysis removal' (built in
# stage 2: HST / no training batch / no role / no block / Nursing staff / zero
# adoptions in MASD), that column IS the analysis definition: every row with a
# non-blank reason is removed here in ONE step and the individual filters below
# are skipped, so the crosstab N always equals the derived sheet's 'Removal
# flowchart'. Older derived files without the column use the original filters.
REASON_COL = 'pre_analysis_exclusion'
if REASON_COL not in data1.columns and 'Reason for pre-analysis removal' in data1.columns:
    REASON_COL = 'Reason for pre-analysis removal'   # derived files from before 02-08-2026
REASON_FROM_DERIVED = REASON_COL in data1.columns
_PR_HST = _PR_NOROLE = _PR_NOBLOCK = None
if REASON_FROM_DERIVED:
    _pr = data1[REASON_COL].fillna('').astype(str).str.strip()
    _pr_n0 = len(data1)
    _pr_l0 = data1['User Acc ID all'].astype(str).nunique() \
        if 'User Acc ID all' in data1.columns else None
    print(f"  Pre-analysis removal (reason column from stage 2):"
          f" start {_pr_n0} case row(s), {_pr_l0} learner(s)")
    for _r, _n in _pr[~_pr.isin(['', 'Included'])].value_counts().items():
        _lr = (data1.loc[_pr.eq(_r), 'User Acc ID all'].astype(str).nunique()
               if 'User Acc ID all' in data1.columns else '?')
        print(f"    - {_r}: {_n} case row(s), {_lr} learner(s) affected")
    _PR_HST = data1[_pr.str.contains('HST', case=False, na=False)].copy()
    _PR_NOROLE = data1[_pr.str.startswith('No role')].copy()
    _PR_NOBLOCK = data1[_pr.str.startswith('No block')].copy()
    data1 = data1[_pr.isin(['', 'Included'])].copy()
    _pr_l1 = data1['User Acc ID all'].astype(str).nunique() \
        if 'User Acc ID all' in data1.columns else None
    print(f"  Analysis set: {len(data1)} case row(s), {_pr_l1} learner(s)")

# ---- ADD-ON DATA STEP 3: role-group analysis filters (requested Jul 2026) -------
# (a) Cases with NO role / role group defined (upstream 'Role Group' = 'z_NA':
#     all three raw role columns empty) are REMOVED from the analysis dataset
#     BEFORE any crosstab is built, so every count, percentage and Total in
#     EVERY table (all sections; column-wise, row-wise and Jalna pipelines)
#     uses the reduced N. Until now these cases were only hidden from display
#     but still counted inside Totals. The removed cases are kept aside in
#     NO_ROLE_CASES and written out as a separate list AFTER the analysis ends
#     (see the final NO-ROLE-GROUP CASE LIST cell).
# (b) The role groups in EXCLUDED_ROLE_GROUPS -- currently 'Nursing staff'
#     (the mapping sheet renamed 'Staff nurse' -> 'Nursing staff' on
#     29-07-2026; it carries Nursing Officer / Nursing Sister / Staff Nurse /
#     Male Staff nurse / Nursing Supervisor / the PHNs / District Nurse
#     Instructor) -- are likewise removed from the analysis for now.
# Set EXCLUDE_UNDEFINED_ROLE_GROUP = False and EXCLUDED_ROLE_GROUPS = [] to
# restore the previous behaviour exactly.
EXCLUDE_UNDEFINED_ROLE_GROUP = True
EXCLUDED_ROLE_GROUPS = ['Nursing staff']

# (c) HST-project staff -- roles such as 'HST Trainer', 'HST Trainer + Calling
#     Team Leader' and 'HST Trainer + Calling Team Member' -- run the programme;
#     they are not learners under study and must never appear in the analysis.
#     These roles are absent from the mapping sheet, so they arrive as Role
#     Group 'Other' and CANNOT be caught by EXCLUDED_ROLE_GROUPS above; they
#     are matched here on the RAW role columns instead. A case is dropped when
#     ANY of the raw role columns matches ANY pattern in EXCLUDED_ROLE_PATTERNS.
#     This runs BEFORE (a) and (b) so HST cases never reach the no-role list.
#     Set EXCLUDE_HST_ROLES = False to restore the previous behaviour exactly.
EXCLUDE_HST_ROLES = True
EXCLUDED_ROLE_PATTERNS = [r'\bHST\b', r'calling\s*team']
RAW_ROLE_COLS = ['User Role', 'User Role_CR', 'Role_M']

HST_ROLE_CASES = data1.iloc[0:0].copy()   # kept aside for the run log
if not REASON_FROM_DERIVED and EXCLUDE_HST_ROLES and EXCLUDED_ROLE_PATTERNS:
    _hst_cols = [c for c in RAW_ROLE_COLS if c in data1.columns]
    if _hst_cols:
        _hst_pat = '|'.join(EXCLUDED_ROLE_PATTERNS)
        _hst_mask = pd.Series(False, index=data1.index)
        for _hc in _hst_cols:
            _hst_mask |= data1[_hc].astype(str).str.contains(
                _hst_pat, case=False, na=False, regex=True)
        if _hst_mask.any():
            _hst_n0 = len(data1)
            HST_ROLE_CASES = data1[_hst_mask].copy()
            data1 = data1[~_hst_mask].copy()
            _hst_l = (HST_ROLE_CASES['User Acc ID all'].astype(str).nunique()
                      if 'User Acc ID all' in HST_ROLE_CASES.columns
                      else len(HST_ROLE_CASES))
            print(f"  HST role filter: removed {len(HST_ROLE_CASES)} case row(s)"
                  f" ({_hst_l} unique learner(s)) with an HST-project role"
                  f" -> analysis N: {_hst_n0} -> {len(data1)}")
            for _hc in _hst_cols:
                _hv = HST_ROLE_CASES.loc[
                    HST_ROLE_CASES[_hc].astype(str).str.contains(
                        _hst_pat, case=False, na=False, regex=True), _hc]
                if len(_hv):
                    print(f"    matched in '{_hc}': {dict(_hv.value_counts())}")
            _hst_show = [c for c in ['User Acc ID all', 'User Name all', 'Case ID',
                                     'User Role', 'User Role_CR', 'Role_M']
                         if c in HST_ROLE_CASES.columns]
            if _hst_show:
                print("    excluded HST cases:")
                print(HST_ROLE_CASES[_hst_show].to_string(index=False))
        else:
            print("  HST role filter: no HST-project role found in"
                  f" {_hst_cols} -> nothing removed")
    else:
        print(f"  HST role filter: none of {RAW_ROLE_COLS} present -> skipped")
else:
    print("  EXCLUDE_HST_ROLES is False -> HST-project roles NOT removed")

NO_ROLE_CASES = data1.iloc[0:0].copy()   # empty fallback so the list cell always runs
if REASON_FROM_DERIVED:
    print('  Role-group filters skipped -> the reason column was already applied')
elif 'Role Group' in data1.columns:
    _rg3_norm = (data1['Role Group'].astype(str).str.strip().str.lower()
                 .str.replace('_', ' ', regex=False).str.split().str.join(' '))
    _rg3_undef = data1['Role Group'].isna() | _rg3_norm.isin(['z na', 'zna', 'nan', 'na', 'none', ''])
    if EXCLUDE_UNDEFINED_ROLE_GROUP and _rg3_undef.any():
        _rg3_n0 = len(data1)
        NO_ROLE_CASES = data1[_rg3_undef].copy()
        data1 = data1[~_rg3_undef].copy()
        _rg3_l = (NO_ROLE_CASES['User Acc ID all'].astype(str).nunique()
                  if 'User Acc ID all' in NO_ROLE_CASES.columns else len(NO_ROLE_CASES))
        print(f"  Role Group filter: removed {len(NO_ROLE_CASES)} case row(s) with no role/role group defined"
              f" ({_rg3_l} unique learner(s)) -> analysis N: {_rg3_n0} -> {len(data1)}")
    if EXCLUDED_ROLE_GROUPS:
        _rg3_excl = (data1['Role Group'].astype(str).str.strip().str.lower()
                     .isin([g.strip().lower() for g in EXCLUDED_ROLE_GROUPS]))
        if _rg3_excl.any():
            _rg3_n1 = len(data1)
            _rg3_le = (data1.loc[_rg3_excl, 'User Acc ID all'].astype(str).nunique()
                       if 'User Acc ID all' in data1.columns else int(_rg3_excl.sum()))
            data1 = data1[~_rg3_excl].copy()
            print(f"  Role Group filter: excluded role group(s) {EXCLUDED_ROLE_GROUPS} ->"
                  f" removed {_rg3_n1 - len(data1)} case row(s) ({_rg3_le} unique learner(s))"
                  f" -> analysis N: {_rg3_n1} -> {len(data1)}")
    print(f"  Role Group values in analysis: {dict(data1['Role Group'].value_counts())}")
else:
    print("  'Role Group' column not found -> role-group analysis filters skipped.")


# ---- ADD-ON DATA STEP 3 (d): drop cases with NO block defined ------------------
# Requested 29-07-2026. BLOCK_COL ('Blocks') is derived from 'Phc Taluka_CR' in
# ADD-ON DATA STEP 2 and is 'z_NA' when case registration carries no PHC taluka.
# Until now those cases were hidden from display (the crosstab generator drops
# z_NA rows/columns) but still counted inside every Total, so the block columns
# never added up to the Total. They are now REMOVED from the analysis dataset
# before any crosstab is built -- exactly the treatment part (a) gives no-role
# cases -- kept aside in NO_BLOCK_CASES and written out as a separate list AFTER
# the analysis ends (see the NO-BLOCK CASE LIST cell).
# Guarded by IS_DISTRICT_LEVEL: at state level (ANALYSIS_LEVEL = 'state') the
# Blocks variable is not used by any table, so nothing is removed there.
# Set EXCLUDE_UNDEFINED_BLOCK = False to restore the previous behaviour exactly.
# 29-07-2026: set to False -- missing blocks are now FILLED from the MASD sheet
# (ADD-ON DATA STEP 2b) instead of having their cases removed.
# 29-07-2026 (confirmed by the project lead): missing blocks are FIRST filled
# from the MASD sheet (ADD-ON DATA STEP 2b) and then from the DOCS sheet; the
# handful of cases neither source can resolve are REMOVED from the analysis
# here. The lead's instruction was that losing learners with no block or no
# cadre is acceptable, and that tables must not be left with a totalling gap
# because of one or two incomplete records. Set to False to keep them in (they
# would reappear as a hidden z_NA column and the block columns would again fall
# short of the Total).
EXCLUDE_UNDEFINED_BLOCK = True

NO_BLOCK_CASES = data1.iloc[0:0].copy()   # empty fallback so the list cell always runs
if REASON_FROM_DERIVED:
    print('  Block filter skipped -> the reason column was already applied')
elif not EXCLUDE_UNDEFINED_BLOCK:
    print("  EXCLUDE_UNDEFINED_BLOCK is False -> cases with no block kept in the analysis")
elif not IS_DISTRICT_LEVEL:
    print(f"  Block filter: ANALYSIS_LEVEL is '{ANALYSIS_LEVEL}' -> '{BLOCK_COL}' unused, nothing removed")
elif BLOCK_COL not in data1.columns:
    print(f"  Block filter: '{BLOCK_COL}' column not found -> skipped")
else:
    _blk_norm = (data1[BLOCK_COL].astype(str).str.strip().str.lower()
                 .str.replace('_', ' ', regex=False).str.split().str.join(' '))
    _blk_undef = data1[BLOCK_COL].isna() | _blk_norm.isin(['z na', 'zna', 'nan', 'na', 'none', ''])
    if _blk_undef.any():
        _blk_n0 = len(data1)
        _blk_l0 = (data1['User Acc ID all'].astype(str).nunique()
                   if 'User Acc ID all' in data1.columns else _blk_n0)
        NO_BLOCK_CASES = data1[_blk_undef].copy()
        data1 = data1[~_blk_undef].copy()
        _blk_laff = (NO_BLOCK_CASES['User Acc ID all'].astype(str).nunique()
                     if 'User Acc ID all' in NO_BLOCK_CASES.columns else len(NO_BLOCK_CASES))
        _blk_l1 = (data1['User Acc ID all'].astype(str).nunique()
                   if 'User Acc ID all' in data1.columns else len(data1))
        print(f"  Block filter: removed {len(NO_BLOCK_CASES)} case row(s) with no block defined"
              f" ({_blk_laff} learner(s) had at least one such case)"
              f" -> analysis N: {_blk_n0} -> {len(data1)}"
              f" | learners: {_blk_l0} -> {_blk_l1}")
        print(f"  {BLOCK_COL} values in analysis: {dict(data1[BLOCK_COL].value_counts())}")
    else:
        print(f"  Block filter: no case with an undefined '{BLOCK_COL}' -> nothing removed")

# restore the reason-based case lists for the export cells (the legacy filter
# sections above reset them to empty frames when they are skipped)
if REASON_FROM_DERIVED:
    HST_ROLE_CASES = _PR_HST
    NO_ROLE_CASES = _PR_NOROLE
    NO_BLOCK_CASES = _PR_NOBLOCK


Loading data...


  Loaded: 590 rows x 1693 columns
  Analysis level: district | Jalna (JL) input detected: True
  Role Group: merging 1 'MO' (Medical Officer) row(s) into 'Others'
  Derived 'Blocks' from 'Phc Taluka_CR': {'Jalna': np.int64(197), 'Bhokardan': np.int64(77), 'Ambad': np.int64(75), 'Ghansawangi': np.int64(54), 'Jafrabad': np.int64(48), 'Badnapur': np.int64(45), 'Partur': np.int64(39), 'Mantha': np.int64(32), 'z_NA': np.int64(23)}


ROW ORDER DECLARATION

In [2]:
row_order = {
    "Department": ['HFW', 'WCD', 'MSRLS'],
    "District" :['East Garo Hills','East Jaintia Hills','East Khasi Hills','Eastern West Khasi Hills','North Garo Hills','Ri Bhoi','South Garo Hills','South West Garo Hills','South West Khasi Hills','West Garo Hills','West Jaintia Hills','West Khasi Hills'],
    # 03-08-2026 (project lead): fixed display order for Role Group in every
    # crosstab -- display only, no count or percentage is affected. A role group
    # not listed here is appended after 'Other' (a visible signal that a new
    # group has appeared in the data).
    "Role Group": ['AWW', 'ASHA', 'ASHASup', 'ANM', 'CHO', 'Other'],
    "Member Tag_CR": ['Master Trainer', 'Facilitator', 'Other','Under evaluation','Under Evaluation'],
    "Total Adoptions group": ['01_to_03', '04_to_06', '07_to_09', '10_or_more'],
    "Total Adoptions group after exclusion": ['01_to_03', '04_to_06', '07_to_09', '10_or_more'],
    "Type of adoption": ['ANC', 'PNCL5M','PNCG5M','Data to calculate child age of adoption is not available','Invalid difference: Baby Adoption date earlier than Mother adoption date','Mother adoption date is missing'],
    "Mother Age Group": ['15_to_18_years_old', '19_to_24_years_old', '25_to_36_years_old', '37_to_46_years_old','47_to_50_years_old'],
    "Is this the mother's first pregnancy_C": ["Yes", "No"],
    "Mother's education level_M": ['Illiterate','Class 5', 'Class 8', 'Class 10', 'Class 12','Vocational education', 'Graduate', 'Post-graduate'],
    "Color of mother's family ration card_M": ['Yellow','Orange', 'White','Pink','No ration card','Other'],
    "Mother's family type_M": ['Joint family', 'Nuclear family'],
    "Mother's social category_M": ['General', 'Other Backward Class (OBC)', 'Scheduled Caste (SC)', 'Scheduled Tribe (ST)'],
    "Location of Delivery category": ['Government', 'Private', 'Home'],
    "Method of delivery_C": ['Normal', 'Cesarean Section', 'Assisted delivery'],
    "birthweight category": ['01.5_or_less_kg','01.51_to_02.5_kg','02.51_to_02.7_kg','02.71_to_03.0_kg','03.01_to_03.5_kg','More_than_03.5_kg'],
    "birthweight category 2": ['Less_than_01.5_kg','01.5_to_02.49_kg','02.5_to_02.69_kg','02.7_to_02.99_kg','03.0_to_03.49_kg','03.5_or_more_kg','Above_+6SD'],
    "birthweight group":['Low Birth Weight','Normal Birth Weight','Overweight','Birth Weight is not available'],
    
    "adoption age classification": ['invalid_age','d000','d001_to_d015', 'd016_to_d030', 'd031_to_d060', 'd061_to_d090', 'd091_to_d120', 'd121_to_d150', 'd151_to_d180',
                                        'd181_to_d210',
    'd211_to_d240',
    'd241_to_d270',
    'd271_to_d300',
    'd301_to_d330',
    'd331_to_d365',
    'd366_plus'],
    "last visit age classification 1": ['d001_to_d015', 'd016_to_d030', 'd031_to_d060', 'd061_to_d090', 'd091_to_d120', 'd121_to_d150', 'd151_to_d180',
                                        'd181_to_d210',
    'd211_to_d240',
    'd241_to_d270',
    'd271_to_d300',
    'd301_to_d330',
    'd331_to_d365',
    'd366_plus'],
    "last visit age classification 2": ['d001_to_d060','d061_to_d120','d121_to_d180',
                                        'd181_to_d210',
    'd211_to_d240',
    'd241_to_d270',
    'd271_to_d300',
    'd301_to_d330',
    'd331_to_d360',
    'd361_plus'],
    "adoption duration baby": ['Only birth data available','01_to_30_days','31_to_60_days', '61_to_90_days','91_days_or_more', 'negatives'],
    "visit category": ['01_visit','02_visits','03_visits', '04_to_05_visits', '06_to_07_visits', '08_to_09_visits', '10_or_more_visits'],
    "bf assessment category": ['00_assessment','01_assessment',
 '02_assessments',
 '03_assessments',
 '04_to_05_assessments',
 '06_to_07_assessments',
 '08_to_09_assessments',
 '10_or_more_assessments'],
    "cf assessment category": ['00_assessment','01_assessment',
 '02_assessments',
 '03_assessments',
 '04_to_05_assessments',
 '06_to_07_assessments',
 '08_to_09_assessments',
 '10_or_more_assessments'],
    "bf assessment category 2": ['00_assessment','01_assessment',
 '02_assessments',
 '03_assessments',
 '04_to_05_assessments',
 '06_or_more_assessments'],
    "cf assessment category 2": ['00_assessment','01_assessment',
 '02_assessments',
 '03_assessments',
 '04_to_05_assessments',
 '06_or_more_assessments'],


    "Activity Score": ['00','01_to_02', '03_to_10', '11_to_20', '21_to_30', '31_or_above'],
    "WFA status at BV 2": ['Normal','MUW','SUW','Zscore not available'],
    "WFA status at AV 2": ['Normal','MUW','SUW','Zscore not available'],
    "WFA status at LV 2": ['Normal','MUW','SUW','Zscore not available'],
    "HFA status at BV 2": ['Normal','MST','SST','Zscore not available'],
    "HFA status at AV 2": ['Normal','MST','SST','Zscore not available'],
    "HFA status at LV 2": ['Normal','MST','SST','Zscore not available'],
    "WFH status at BV 2": ['Normal','MAM','SAM','Zscore not available'],
    "WFH status at AV 2": ['Normal','MAM','SAM','Zscore not available'],
    "WFH status at LV 2": ['Normal','MAM','SAM','Zscore not available'],
    "WFA at LV/Change zscore LV AV": ['Normal/Catchup', 'Normal/nan', 'None/nan', 'Normal/0.00 to 0.67',
       'MUW/0.00 to 0.67', 'MUW/Faltering', 'Mild/0.00 to 0.67',
       'Mild/-0.67 to -0.01', 'Mild/Catchup', 'Normal/-0.67 to -0.01',
       'Mild/Faltering', 'Normal/Faltering', 'SUW/Faltering',
       'MUW/-0.67 to -0.01', 'MUW/Catchup', 'Mild/nan',
       'SUW/0.00 to 0.67', 'SUW/Catchup', 'MUW/nan', 'SUW/-0.67 to -0.01',
       'SUW/nan'],
    "number of protein assessment category": ['00_assessment','01_assessment',
 '02_assessments',
 '03_assessments',
 '04_to_05_assessments',
 '06_to_07_assessments',
 '08_to_09_assessments',
 '10_or_more_assessments'],
    "gestational week classification": ['20 weeks or less', '20.1 to 28 weeks','28.1 to 34 weeks','34.1 to 36.9 weeks', '37 to 41.9 weeks', '42 weeks or more'],
    "Mother adoption till birth": ["PNC", "001_to_030", "031_to_060", "061_to_090", "091_to_120", 
    "121_to_150", "151_to_180", "181_to_210", "211_to_240", "241_to_270", "271_to_300",'DOB is missing','Mother adoption date is missing',
       'DOB and Mother adoption both missing','Mother adoption during pregnancy is More than 300 days- Suggestive of data entry error'],
    "Adoption type 1": ['ANC>=60D', 'ANC<60D','PNCL5M','PNCG5M'],
    'ANC 60d 3visit':['Yes','No'],
    'PNC lt5 60d 8visit':['Yes','No'],
    'PNC gt5 60d 4visit':['Yes','No'],
    'Included excluded' : ['Included','Excluded'],
    'Primary Exclusion PNC 2' : [
    "No Child ID - Child not born yet/child not followed up",
    "Birth date not available",
    "Adoption visit anthropometry data(Wt,Ht,zscore) not available",
    "Mother adoption date is after last visit date of the child - suggestive of data entry error",
    "Birth weight category is below -6SD or above +6SD",
    "Birth Weight in CR sheet and Visit 1 Weight in CM sheet do not match",
    "Date of birth in CR sheets and Visit Date 1 in CM sheet do not match",
    "Adoption date in CR sheet and Visit Date 2 in CM sheet do not match",
    "Last visit dates do not match in CR and CM sheets",
    "Wt Gain per day ≥ 180gm between Adoption and Last visit",
    "Only adoption visit occurred",
    "Only birth anthropometry details available",
    "mother_AD_DOB > 300",
    "Birth date, Adoption date, and Last visit date are the same",
    'User role belongs to excluded category',
    'User Role is not available',"Weight Zscore at LV/AV/BV is missing",
    "Height Zscore at LV/AV/BV is missing",
    "WFH Zscore at LV/AV/BV is missing",
    'Invalid difference: Baby Adoption date earlier than Mother adoption date',"Follow-up category is Unclassified ","Last visit Date not available",
    'Included'],
    'Primary Exclusion ANC' : [
    "No Child ID - Child not born yet/child not followed up",
    "Birth date not available",
    "Birth weight category is below -6SD or above +6SD",
    "Birth Weight in CR sheet and Visit 1 Weight in CM sheet do not match",
    "Date of birth in CR sheets and Visit Date 1 in CM sheet do not match",
    'User role belongs to excluded category',
    'User Role is not available',"Weight Zscore at LV/AV/BV is missing",
    "Height Zscore at LV/AV/BV is missing",
    "WFH Zscore at LV/AV/BV is missing",'Follow-up category is Unclassified',
    'Invalid difference: Baby Adoption date earlier than Mother adoption date',"Follow-up category is Unclassified ",
    "Last visit Date not available",'Included'],
    "Avg Weight Gain Category LV AV 2": ['0 or less',
 '0.1–5 g/d',
 '5.1–10 g/d',
 '10.1–15 g/d',
 '15.1–17 g/d',
 '17.1–20 g/d',
 '20.1–25 g/d',
 '25.1–28 g/d',
 '28.1–30 g/d',
 '30–35 g/d',
 '35.1–40 g/d',
 'More than 40 g/d'],
    "Avg Weight Gain Category LV AV" : [
 'Less than 0g/d','0.1_to_10g/d',
 '10.1_to_20g/d',
 '20.1_to_30g/d',
 '30.1_to_40g/d',
 '40.1_to_50g/d',
 'More than 50g/d'],
    "Learner Normal 60pct" : ["Yes","No"],
    "Improvement in WFA Zscore btw AV LV":["Yes","No"],
    "Catchup YN":["Yes","No"],
    "avg weight gain 17g 60pct YN" : ['Yes','No'],
    "avg weight gain 28g 60pct YN" : ['Yes','No'],
}

column_order = row_order  # Same as all individual scripts


# ==============================================================================
# ADD-ON orders (situational variables -- inert unless these variables are used)
# ==============================================================================
row_order["Blocks"] = ['Ambad', 'Badnapur', 'Bhokardan', 'Ghansawangi',
                       'Jafrabad', 'Jalna', 'Mantha', 'Partur']
row_order["Total Adoptions group jalna"] = ['01', '02', '03', 'More_than_03']
row_order["Total Adoptions group after exclusion jalna"] = ['01', '02', '03', 'More_than_03']

# LC3 twin variable: learner-type tags first, 'Under Evaluation' last (values a
# dataset does not contain are skipped; unknown ones are appended after these)
row_order["Learner Category 3"] = ['MT + FL', 'Learner', 'Other',
                                   'Under Evaluation', 'Under evaluation']


HELPER TO RUN A SECTION

In [3]:
def run_section(section_name, data, pairs, output_file):
    """Run one section: generate crosstabs and export."""
    print(f"\n--- {section_name} ---")
    print(f"  Data shape: {data.shape}")
    print(f"  Pairs: {len(pairs)}")
    results = generate_specific_crosstabs(
        data=data,
        crosstab_pairs=pairs,
        row_order=row_order,
        col_order=column_order
    )
    print(f"  Generated: {len(results)} tables")
    export_tables_to_single_sheet(results, output_file)
    return results

---
## Section 1 — All MCD Before Exclusion
`All_MCD_before_exclusion_CT_V1.ipynb` | Filter: full data

In [4]:
print("\n" + "="*60)
print("SECTION 1: All MCD Before Exclusion")
print("="*60)

s1_pairs = [
    ("Department", "District"),
    ("Department", "District"),
    ("Role Group", "District"),
    ("Role Group", "District"),
    ("Learner Category 2", "District"),
    ("Learner Category 2", "District"),
    ("Learner Category 2", "Role Group"),
    ("Learner Category 2", "Role Group"),
    ("Role Group", "Learner Category 2"),
    ("Role Group", "Learner Category 2"),
    ("Department", "Learner Category 2"),
    ("Department", "Learner Category 2"),

    ("Primary Exclusion PNC 2", "District"),
    ("Primary Exclusion PNC 2", "Department"),
    ("Primary Exclusion PNC 2", "Role Group"),
    ("Primary Exclusion PNC 2", "Learner Category 2"),
    ("Primary Exclusion PNC 2", "Type of adoption"),
    ("Primary Exclusion PNC 2", "adoption age classification"),
    ("Primary Exclusion PNC 2", "adoption duration baby"),
    ("Primary Exclusion PNC 2", "visit category"),
    ("Primary Exclusion PNC 2", "Activity Score"),
]

run_section("S1: All MCD Before Exclusion", data1, s1_pairs,
            "crosstab_all_data_ML_230426_V(230426).xlsx")



SECTION 1: All MCD Before Exclusion

--- S1: All MCD Before Exclusion ---
  Data shape: (590, 1694)
  Pairs: 21


  Generated: 15 tables
  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/outputs\crosstab_all_data_ML_230426_V(230426).xlsx (15 tables)


{'Department vs Blocks': Blocks            Ambad     Badnapur    Bhokardan  Ghansawangi     Jafrabad  \
 Department                                                                    
 HFW         64 (85.33%)  45 (100.0%)  77 (100.0%)  42 (77.78%)  45 (93.75%)   
 WCD         11 (14.67%)     0 (0.0%)     0 (0.0%)  12 (22.22%)    3 (6.25%)   
 Total       75 (100.0%)  45 (100.0%)  77 (100.0%)  54 (100.0%)  48 (100.0%)   
 
 Blocks             Jalna       Mantha       Partur         Total  
 Department                                                        
 HFW         156 (79.19%)  29 (90.62%)  35 (89.74%)  509 (86.27%)  
 WCD          36 (18.27%)    3 (9.38%)   4 (10.26%)   71 (12.03%)  
 Total       197 (100.0%)  32 (100.0%)  39 (100.0%)  590 (100.0%)  ,
 'Role Group vs Blocks': Blocks             Ambad     Badnapur    Bhokardan  Ghansawangi     Jafrabad  \
 Role Group                                                                     
 AWW          11 (14.67%)     0 (0.0%)     0 (0

---
## Section 2 — Included / Excluded All
`crossTAB_included_excluded_all.ipynb` | Filter: full data

In [5]:
print("\n" + "="*60)
print("SECTION 2: Included/Excluded All")
print("="*60)

# Build pairs: many columns vs 'Included excluded'
s2_cols = [
    "District",
    "Role Group",
    "Department",
    "Learner Category 2",
    "Total Adoptions group",
    "Total Adoptions group after exclusion",
    "Primary Exclusion ANC",
    "Primary Exclusion PNC 2",
    "Type of adoption",
    "Adoption type 1",
    "Mother Age Group",
    "Is this the mother's first pregnancy?_C",
    "Number of child's siblings_C",
    "Mother's education level_M",
    "Color of mother's family ration card_M",
    "Mother's family type_M",
    "Mother's social category_M",
    "Location of Delivery category",
    "Method of delivery_C",
    "Mother adoption till birth",
    "gestational week classification",
    "adoption age classification",
    "last visit age classification 1",
    "last visit age classification 2",
    "adoption duration baby",
    "visit category",
    "bf assessment category",
    "cf assessment category",
    "Activity Score",
    "birthweight category",
    "birthweight category 2",
    "WFA status at BV 2",
    "WFA status at AV 2",
    "WFA status at LV 2",
    "WFA at LV/Change zscore LV AV",
    "HFA status at BV 2",
    "HFA status at AV 2",
    "HFA status at LV 2",
    "WFH status at BV 2",
    "WFH status at AV 2",
    "WFH status at LV 2",
    "Member Tag_CR",
    'Followup Category',
    "avg weight gain 28g 60pct YN",
    "avg weight gain 17g 60pct YN",
    "Catchup YN",
    "Avg Weight Gain Category LV AV 2",
    "Avg Weight Gain Category LV AV",
    "Learner Normal 60pct" ,
    "Improvement in WFA Zscore btw AV LV",
]
s2_pairs = [(col, "Included excluded") for col in s2_cols]

print(f"  Data shape: {data1.shape}")
print(f"  Pairs: {len(s2_pairs)}")
s2_results = generate_specific_crosstabs_v2(
    data=data1, crosstab_pairs=s2_pairs,
    row_order=row_order, col_order=column_order)
print(f"  Generated: {len(s2_results)} tables")
export_tables_to_single_sheet(s2_results,
    "included_excluded_CT_ML_100526_V(120526).xlsx")


SECTION 2: Included/Excluded All
  Data shape: (590, 1694)
  Pairs: 50


  Generated: 49 tables
  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/outputs\included_excluded_CT_ML_100526_V(120526).xlsx (49 tables)


---
## Section 3 — Learners (Deduplicated)
`crossTAB_learners_V1.ipynb` | Two sheets: all learners + included learners

In [6]:
print("\n" + "="*60)
print("SECTION 3: Learners")
print("="*60)

df_learners = data1.drop_duplicates(subset=['User Acc ID all'])
df_learners = df_learners[df_learners['User Acc ID all'].notna()]

s3_pairs = [
    ("Department", "District"),
    ("Role Group", "District"),
    ("Learner Category 2", "District"),
    ("Learner Category 2", "Role Group"),
    ("Role Group", "Learner Category 2"),
    ("Department", "Learner Category 2"),
    ("Total Adoptions group", "District"),
    ("Total Adoptions group", "Department"),
    ("Total Adoptions group", "Role Group"),
    ("Total Adoptions group", "Learner Category 2"),
    ("Total Adoptions group after exclusion", "District"),
    ("Total Adoptions group after exclusion", "Department"),
    ("Total Adoptions group after exclusion", "Role Group"),
    ("Total Adoptions group after exclusion", "Learner Category 2"),
]

# Before exclusion
run_section("S3a: Learners Before Exclusion", df_learners, s3_pairs,
            "learner_based_ct_before_exclusion_ML_100526_V(120526).xlsx")

# After exclusion (matching Script 3: Primary Exclusion_PNC_2)
df_learners_incl = df_learners[df_learners['Primary Exclusion PNC 2'] == 'Included']
run_section("S3b: Learners After Exclusion", df_learners_incl, s3_pairs,
            "learner_based_ct_after_exclusion_ML_100526_V(120526).xlsx")


SECTION 3: Learners

--- S3a: Learners Before Exclusion ---
  Data shape: (192, 1694)
  Pairs: 14


  Generated: 14 tables
  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/outputs\learner_based_ct_before_exclusion_ML_100526_V(120526).xlsx (14 tables)

--- S3b: Learners After Exclusion ---
  Data shape: (50, 1694)
  Pairs: 14


  Generated: 14 tables
  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/outputs\learner_based_ct_after_exclusion_ML_100526_V(120526).xlsx (14 tables)


{'Department vs Blocks': Blocks           Ambad    Badnapur   Bhokardan Ghansawangi    Jafrabad  \
 Department                                                               
 HFW         5 (100.0%)  5 (100.0%)  8 (100.0%)  4 (100.0%)  5 (100.0%)   
 WCD           0 (0.0%)    0 (0.0%)    0 (0.0%)    0 (0.0%)    0 (0.0%)   
 Total       5 (100.0%)  5 (100.0%)  8 (100.0%)  4 (100.0%)  5 (100.0%)   
 
 Blocks            Jalna      Mantha      Partur        Total  
 Department                                                    
 HFW         14 (82.35%)  2 (100.0%)  4 (100.0%)   47 (94.0%)  
 WCD          2 (11.76%)    0 (0.0%)    0 (0.0%)     2 (4.0%)  
 Total       17 (100.0%)  2 (100.0%)  4 (100.0%)  50 (100.0%)  ,
 'Role Group vs Blocks': Blocks            Ambad    Badnapur   Bhokardan Ghansawangi    Jafrabad  \
 Role Group                                                                
 AWW            0 (0.0%)    0 (0.0%)    0 (0.0%)    0 (0.0%)    0 (0.0%)   
 ANM           3 (60.0%)  

---
## Section 4 — All MCD After Exclusion
`All_MCD_after_exclusion_CT_V1.ipynb` | Filter: Included only

In [7]:
print("\n" + "="*60)
print("SECTION 4: All MCD After Exclusion")
print("="*60)

all_data_s4 = data1[data1['Exclusion Reason PNC 2'] == 'Included']

s4_pairs = [
    ("Department", "District"),
    ("Role Group", "District"),
    ("Learner Category 2", "District"),
    
    ("Learner Category 2", "Role Group"),
    
    ("Role Group", "Learner Category 2"),
    ("Department", "Learner Category 2"),
    
    ("Type of adoption", "District"),
    ("Type of adoption", "Department"),
    ("Type of adoption", "Role Group"),
    ("Type of adoption", "Learner Category 2"),
    
    ("Mother Age Group", "District"),
    ("Is this the mother's first pregnancy?_C", "District"),
    ("Mother's education level_M", "District"),
    ("Color of mother's family ration card_M", "District"),
    ("Mother's family type_M", "District"),
    ("Mother's social category_M", "District"),
    
    ("Mother's education level_M", "Role Group"),
    ("Color of mother's family ration card_M", "Role Group"),
    ("Mother's family type_M", "Role Group"),
    ("Mother's social category_M", "Role Group"),
    
    ("Mother's education level_M", "Learner Category 2"),
    ("Color of mother's family ration card_M", "Learner Category 2"),
    ("Mother's family type_M", "Learner Category 2"),
    ("Mother's social category_M", "Learner Category 2"),
    
    ("Mother's education level_M", "Type of adoption"),
    ("Color of mother's family ration card_M", "Type of adoption"),
    ("Mother's family type_M", "Type of adoption"),
    ("Mother's social category_M", "Type of adoption"),
    
    # Between mother variables (using exact names)
    ("Mother's education level_M", "Color of mother's family ration card_M"),
    ("Mother's family type_M", "Color of mother's family ration card_M"),
    ("Mother's social category_M", "Color of mother's family ration card_M"),
    
    ("Mother's education level_M", "Mother's social category_M"),
    ("Color of mother's family ration card_M", "Mother's social category_M"),
    ("Mother's family type_M", "Mother's social category_M"),
    
    ("Is this the mother's first pregnancy?_C", "Mother Age Group"),
    
    # Location of Delivery
    ("Location of Delivery category", "District"),
    ("Location of Delivery category", "Role Group"),
    ("Location of Delivery category", "Learner Category 2"),
    ("Location of Delivery category", "Type of adoption"),
    ("Location of Delivery category", "Mother's education level_M"),
    ("Location of Delivery category", "Color of mother's family ration card_M"),
    ("Location of Delivery category", "Mother's social category_M"),
    ("Location of Delivery category", "gestational week classification"),
    ("Location of Delivery category", "birthweight category"),
    ("Location of Delivery category", "birthweight category 2"),
    
    # Method of delivery
    ("Method of delivery_C", "District"),
    ("Method of delivery_C", "Role Group"),
    ("Method of delivery_C", "Learner Category 2"),
    ("Method of delivery_C", "Type of adoption"),
    ("Method of delivery_C", "Mother Age Group"),
    ("Method of delivery_C", "Mother's education level_M"),
    ("Method of delivery_C", "Color of mother's family ration card_M"),
    ("Method of delivery_C", "Mother's social category_M"),
    ("Method of delivery_C", "gestational week classification"),
    ("Method of delivery_C", "birthweight category"),
    ("Method of delivery_C", "birthweight category 2"),
    
    # birthweight
    ("birthweight category", "District"),
    ("birthweight category 2", "District"),
    
    ("birthweight category", "Role Group"),
    ("birthweight category 2", "Role Group"),
    
    ("birthweight category", "Learner Category 2"),
    ("birthweight category 2", "Learner Category 2"),
    
    ("birthweight category", "Type of adoption"),
    ("birthweight category 2", "Type of adoption"),
    
    ("birthweight category", "Mother Age Group"),
    ("birthweight category 2", "Mother Age Group"),
    
    ("birthweight category", "Mother's education level_M"),
    ("birthweight category 2", "Mother's education level_M"),
    
    ("birthweight category", "Color of mother's family ration card_M"),
    ("birthweight category 2", "Color of mother's family ration card_M"),
    
    ("birthweight category", "Mother's social category_M"),
    ("birthweight category 2", "Mother's social category_M"),
    
    ("birthweight category", "gestational week classification"),
    ("birthweight category 2", "gestational week classification"),
    
    ("birthweight category", "adoption age classification"),
    ("birthweight category 2", "adoption age classification"),
]

run_section("S4: All MCD After Exclusion", all_data_s4, s4_pairs,
            "crosstab_after_exclusion_ML_230426_V(230426).xlsx")


SECTION 4: All MCD After Exclusion

--- S4: All MCD After Exclusion ---
  Data shape: (216, 1694)
  Pairs: 76


  Generated: 76 tables
  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/outputs\crosstab_after_exclusion_ML_230426_V(230426).xlsx (76 tables)


{'Department vs Blocks': Blocks            Ambad     Badnapur    Bhokardan  Ghansawangi     Jafrabad  \
 Department                                                                    
 HFW         22 (84.62%)  20 (100.0%)  28 (100.0%)  16 (66.67%)  18 (100.0%)   
 WCD          4 (15.38%)     0 (0.0%)     0 (0.0%)   8 (33.33%)     0 (0.0%)   
 Total       26 (100.0%)  20 (100.0%)  28 (100.0%)  24 (100.0%)  18 (100.0%)   
 
 Blocks            Jalna      Mantha       Partur         Total  
 Department                                                      
 HFW         62 (78.48%)  9 (100.0%)  12 (100.0%)  187 (86.57%)  
 WCD         15 (18.99%)    0 (0.0%)     0 (0.0%)    27 (12.5%)  
 Total       79 (100.0%)  9 (100.0%)  12 (100.0%)  216 (100.0%)  ,
 'Role Group vs Blocks': Blocks             Ambad     Badnapur    Bhokardan  Ghansawangi     Jafrabad  \
 Role Group                                                                     
 AWW           4 (15.38%)     0 (0.0%)     0 (0.0%)   8 (

---
## Section 5 — ANC Before Exclusion
`crossTAB_Only_ANC_before_exclusion_V1.ipynb` | Filter: ANC (full data)

In [8]:
print("\n" + "="*60)
print("SECTION 5: ANC Before Exclusion")
print("="*60)

df_anc_all = data1[data1['Type of adoption'] == 'ANC']

s5_pairs = [
    ("Primary Exclusion ANC", "District"),
    ("Primary Exclusion ANC", "Department"),
    ("Primary Exclusion ANC", "Role Group"),
    ("Primary Exclusion ANC", "Learner Category 2"),
    ("Primary Exclusion ANC", "birthweight category 2"),
    ("Primary Exclusion ANC", "ANC protein category"),
    ("Primary Exclusion ANC", "PNC protein category"),
    
    ("Primary Exclusion ANC", "visit category"),

    # Primary Exclusion PNC 2
    ("Primary Exclusion PNC 2", "District"),
    ("Primary Exclusion PNC 2", "Department"),
    ("Primary Exclusion PNC 2", "Role Group"),
    ("Primary Exclusion PNC 2", "Learner Category 2"),
    ("Primary Exclusion PNC 2", "Type of adoption"),
    ("Primary Exclusion PNC 2", "adoption age classification"),
    ("Primary Exclusion PNC 2", "adoption duration baby"),
    ("Primary Exclusion PNC 2", "visit category"),
    ("Primary Exclusion PNC 2", "Activity Score"),
]

run_section("S5: ANC Before Exclusion", df_anc_all, s5_pairs,
            "only_anc_ct_before_exclusion_ML_230426_V(230426).xlsx")



SECTION 5: ANC Before Exclusion

--- S5: ANC Before Exclusion ---
  Data shape: (3, 1694)
  Pairs: 17


  Generated: 17 tables
  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/outputs\only_anc_ct_before_exclusion_ML_230426_V(230426).xlsx (17 tables)


{'Primary Exclusion ANC vs Blocks': Blocks                      Jalna       Total
 Primary Exclusion ANC                        
 Included               3 (100.0%)  3 (100.0%)
 Total                  3 (100.0%)  3 (100.0%),
 'Primary Exclusion ANC vs Department': Department                    WCD       Total
 Primary Exclusion ANC                        
 Included               3 (100.0%)  3 (100.0%)
 Total                  3 (100.0%)  3 (100.0%),
 'Primary Exclusion ANC vs Role Group': Role Group                    AWW       Total
 Primary Exclusion ANC                        
 Included               3 (100.0%)  3 (100.0%)
 Total                  3 (100.0%)  3 (100.0%),
 'Primary Exclusion ANC vs Learner Category 2': Learner Category 2          Other       Total
 Primary Exclusion ANC                        
 Included               3 (100.0%)  3 (100.0%)
 Total                  3 (100.0%)  3 (100.0%),
 'Primary Exclusion ANC vs birthweight category 2': birthweight category 2 02.7_to_0

---
## Section 6 — PNCL5M Before Exclusion
`crossTAB_PNCL5M_before_exclusion_V1.ipynb` | Filter: PNCL5M (full data)

In [9]:
print("\n" + "="*60)
print("SECTION 6: PNCL5M Before Exclusion")
print("="*60)

df_pncl5m_all = data1[data1['Type of adoption'] == 'PNCL5M']

s6_pairs = [
    ("Primary Exclusion PNC 2", "District"),
    ("Primary Exclusion PNC 2", "Department"),
    ("Primary Exclusion PNC 2", "Role Group"),
    ("Primary Exclusion PNC 2", "Learner Category 2"),
    ("Primary Exclusion PNC 2", "Type of adoption"),
    ("Primary Exclusion PNC 2", "adoption age classification"),
    ("Primary Exclusion PNC 2", "adoption duration baby"),
    ("Primary Exclusion PNC 2", "visit category"),
    ("Primary Exclusion PNC 2", "Activity Score"),
]

run_section("S6: PNCL5M Before Exclusion", df_pncl5m_all, s6_pairs,
            "pncl5m_ct_before_exclusion_ML_050426_V(150426).xlsx")


SECTION 6: PNCL5M Before Exclusion

--- S6: PNCL5M Before Exclusion ---
  Data shape: (194, 1694)
  Pairs: 9


  Generated: 9 tables
  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/outputs\pncl5m_ct_before_exclusion_ML_050426_V(150426).xlsx (9 tables)


{'Primary Exclusion PNC 2 vs Blocks': Blocks                                                    Ambad     Badnapur  \
 Primary Exclusion PNC 2                                                        
 Mother adoption date is after last visit date o...     0 (0.0%)     0 (0.0%)   
 Birth weight category is below -6SD or above +6SD      0 (0.0%)     0 (0.0%)   
 Birth Weight in CR sheet and Visit 1 Weight in ...    3 (10.0%)     0 (0.0%)   
 Adoption date in CR sheet and Visit Date 2 in C...    1 (3.33%)   3 (23.08%)   
 Only adoption visit occurred                          2 (6.67%)     0 (0.0%)   
 Weight Zscore at LV/AV/BV is missing                   0 (0.0%)     0 (0.0%)   
 Last visit Date not available                         9 (30.0%)     0 (0.0%)   
 Included                                             15 (50.0%)  10 (76.92%)   
 Total                                               30 (100.0%)  13 (100.0%)   
 
 Blocks                                                Bhokardan  Ghan

---
## Section 7 — PNCG5M Before Exclusion
`crossTAB_PNCG5M_before_exclusion_V1.ipynb` | Filter: PNCG5M (full data)

In [10]:
print("\n" + "="*60)
print("SECTION 7: PNCG5M Before Exclusion")
print("="*60)

df_pncg5m_all = data1[data1['Type of adoption'] == 'PNCG5M']

s7_pairs = s6_pairs  # Same pairs as Script 6

run_section("S7: PNCG5M Before Exclusion", df_pncg5m_all, s7_pairs,
            "pncg5m_ct_before_exclusion_ML_050426_V(150426).xlsx")



SECTION 7: PNCG5M Before Exclusion

--- S7: PNCG5M Before Exclusion ---
  Data shape: (133, 1694)
  Pairs: 9


  Generated: 9 tables
  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/outputs\pncg5m_ct_before_exclusion_ML_050426_V(150426).xlsx (9 tables)


{'Primary Exclusion PNC 2 vs Blocks': Blocks                                                    Ambad     Badnapur  \
 Primary Exclusion PNC 2                                                        
 Birth weight category is below -6SD or above +6SD      0 (0.0%)     0 (0.0%)   
 Birth Weight in CR sheet and Visit 1 Weight in ...     0 (0.0%)    1 (7.14%)   
 Adoption date in CR sheet and Visit Date 2 in C...    1 (5.88%)    1 (7.14%)   
 Only adoption visit occurred                         5 (29.41%)    1 (7.14%)   
 Weight Zscore at LV/AV/BV is missing                   0 (0.0%)    1 (7.14%)   
 Included                                            11 (64.71%)  10 (71.43%)   
 Total                                               17 (100.0%)  14 (100.0%)   
 
 Blocks                                                Bhokardan  Ghansawangi  \
 Primary Exclusion PNC 2                                                        
 Birth weight category is below -6SD or above +6SD      0 (0.0%)    1 

---
## Section 8 — ANC After Exclusion (V1)
`crossTAB_Only_ANC_after_exclusion_V1.ipynb` | Filter: ANC + Included

In [11]:
print("\n" + "="*60)
print("SECTION 8: ANC After Exclusion")
print("="*60)

df_anc_incl = data1[(data1['Type of adoption'] == 'ANC') &
                     (data1['Exclusion Reason PNC 2'] == 'Included')]

s8_pairs = [
    ("birthweight category", "ANC protein category"),
    ("birthweight category", "PNC protein category"),
    ("birthweight category", "Mother adoption till birth"),
    ("birthweight category", "Mother adoption till birth 2"),
    ("birthweight category", "Learner Category 2"),
    ("birthweight category", "ANC 60d 3visit"),


    ("birthweight category 2", "ANC protein category"),
    ("birthweight category 2", "PNC protein category"),
    ("birthweight category 2", "Mother adoption till birth"),
    ("birthweight category 2", "Mother adoption till birth 2"),
    ("birthweight category 2", "Learner Category 2"),
    ("birthweight category 2", "ANC 60d 3visit"),
    
    
    ("birthweight group", "ANC protein category"),
    ("birthweight group", "PNC protein category"),
    ("birthweight group", "Mother adoption till birth"),
    ("birthweight group", "Mother adoption till birth 2"),
    ("birthweight group", "Learner Category 2"),
    ("birthweight group", "ANC 60d 3visit"),
    

    ("adoption age classification", "District"),
    ("adoption age classification", "Role Group"),
    ("adoption age classification", "Learner Category 2"),
    ("adoption age classification", "Type of adoption"),
    ("last visit age classification 1", "adoption age classification"),
    ("last visit age classification 2", "adoption age classification"),
    ("adoption duration baby", "adoption age classification"),

    ("visit category", "District"),
    ("visit category", "Role Group"),
    ("visit category", "Learner Category 2"),
    ("visit category", "Type of adoption"),
    ("visit category", "gestational week classification"),
    ("visit category", "adoption age classification"),
    ("visit category", "adoption duration baby"),
    ("visit category", "birthweight category"),
    ("visit category", "birthweight category 2"),
    ("visit category", "birthweight group"),
    

    ("bf assessment category", "District"),
    ("bf assessment category", "Role Group"),
    ("bf assessment category", "Learner Category 2"),
    ("bf assessment category", "Type of adoption"),
    ("bf assessment category", "adoption age classification"),
    ("bf assessment category", "adoption duration baby"),
    ("bf assessment category", "birthweight category"),
    ("bf assessment category", "birthweight category 2"),
    ("bf assessment category", "birthweight group"),
    ("bf assessment category", "visit category"),

    ("cf assessment category", "District"),
    ("cf assessment category", "Role Group"),
    ("cf assessment category", "Learner Category 2"),
    ("cf assessment category", "Type of adoption"),
    ("cf assessment category", "adoption age classification"),
    ("cf assessment category", "adoption duration baby"),

    ("Activity Score", "District"),
    ("Activity Score", "Role Group"),
    ("Activity Score", "Learner Category 2"),
    ("Activity Score", "Type of adoption"),
    ("Activity Score", "adoption age classification"),
    ("Activity Score", "adoption duration baby"),

    ("WFA at LV/Change zscore LV AV", "District"),
    ("WFA at LV/Change zscore LV AV", "Role Group"),
    ("WFA at LV/Change zscore LV AV", "Learner Category 2"),
    ("WFA at LV/Change zscore LV AV", "Type of adoption"),
    ("WFA at LV/Change zscore LV AV", "Adoption type 1"),
    ("WFA at LV/Change zscore LV AV", "Mother Age Group"),
    ("WFA at LV/Change zscore LV AV", "Number of child's siblings_C"),
    ("WFA at LV/Change zscore LV AV", "Mother's education level_M"),
    ("WFA at LV/Change zscore LV AV", "Color of mother's family ration card_M"),
    ("WFA at LV/Change zscore LV AV", "Mother's social category_M"),
    ("WFA at LV/Change zscore LV AV", "gestational week classification"),
    ("WFA at LV/Change zscore LV AV", "adoption age classification"),
    ("WFA at LV/Change zscore LV AV", "adoption duration baby"),
    ("WFA at LV/Change zscore LV AV", "visit category"),
    ("WFA at LV/Change zscore LV AV", "bf assessment category"),
    ("WFA at LV/Change zscore LV AV", "cf assessment category"),
    ("WFA at LV/Change zscore LV AV", "Activity Score"),

    ("ANC 60d 3visit","Learner Category 2"),
]

run_section("S8: ANC After Exclusion", df_anc_incl, s8_pairs,
            "only_anc_ct_after_exclusion_ML_230426_V(230426).xlsx")


SECTION 8: ANC After Exclusion

--- S8: ANC After Exclusion ---
  Data shape: (2, 1694)
  Pairs: 75


  Generated: 75 tables
  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/outputs\only_anc_ct_after_exclusion_ML_230426_V(230426).xlsx (75 tables)


{'birthweight category vs ANC protein category': ANC protein category    00_visit       Total
 birthweight category                        
 02.71_to_03.0_kg      2 (100.0%)  2 (100.0%)
 Total                 2 (100.0%)  2 (100.0%),
 'birthweight category vs PNC protein category': PNC protein category    00_visit       Total
 birthweight category                        
 02.71_to_03.0_kg      2 (100.0%)  2 (100.0%)
 Total                 2 (100.0%)  2 (100.0%),
 'birthweight category vs Mother adoption till birth': Mother adoption till birth  001_to_030       Total
 birthweight category                              
 02.71_to_03.0_kg            2 (100.0%)  2 (100.0%)
 Total                       2 (100.0%)  2 (100.0%),
 'birthweight category vs Mother adoption till birth 2': Mother adoption till birth 2  001_to_030       Total
 birthweight category                                
 02.71_to_03.0_kg              2 (100.0%)  2 (100.0%)
 Total                         2 (100.0%)  2 (100.0%)

---
## Section 9 — ANC After Exclusion (Copy1 — bf_assessment_category_2)
`crossTAB_Only_ANC_after_exclusion_V1-Copy1.ipynb` | Filter: ANC + Included

In [12]:
print("\n" + "="*60)
print("SECTION 9: ANC After Exclusion - BF Assess Cat 2")
print("="*60)

s9_pairs = [
    ("bf assessment category 2", "District"),
    ("bf assessment category 2", "Role Group"),
    ("bf assessment category 2", "Learner Category 2"),
    ("bf assessment category 2", "Type of adoption"),
    ("bf assessment category 2", "adoption age classification"),
    ("bf assessment category 2", "adoption duration baby"),
    ("bf assessment category 2", "birthweight category"),
    ("bf assessment category 2", "birthweight category 2"),
    ("bf assessment category 2", "birthweight group"),
    ("bf assessment category 2", "visit category"),
]

run_section("S9: ANC After Excl (bf_cat_2)", df_anc_incl, s9_pairs,
            "BF_Assess_only_anc_ct_after_exclusion_ML_230426_V(230426).xlsx")


SECTION 9: ANC After Exclusion - BF Assess Cat 2

--- S9: ANC After Excl (bf_cat_2) ---
  Data shape: (2, 1694)
  Pairs: 10


  Generated: 10 tables
  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/outputs\BF_Assess_only_anc_ct_after_exclusion_ML_230426_V(230426).xlsx (10 tables)


{'bf assessment category 2 vs Blocks': Blocks                         Jalna       Total
 bf assessment category 2                        
 00_assessment              1 (50.0%)   1 (50.0%)
 04_to_05_assessments       1 (50.0%)   1 (50.0%)
 Total                     2 (100.0%)  2 (100.0%),
 'bf assessment category 2 vs Role Group': Role Group                       AWW       Total
 bf assessment category 2                        
 00_assessment              1 (50.0%)   1 (50.0%)
 04_to_05_assessments       1 (50.0%)   1 (50.0%)
 Total                     2 (100.0%)  2 (100.0%),
 'bf assessment category 2 vs Learner Category 2': Learner Category 2             Other       Total
 bf assessment category 2                        
 00_assessment              1 (50.0%)   1 (50.0%)
 04_to_05_assessments       1 (50.0%)   1 (50.0%)
 Total                     2 (100.0%)  2 (100.0%),
 'bf assessment category 2 vs Type of adoption': Type of adoption                 ANC       Total
 bf assessment cate

---
## Section 10 — PNCL5M After Exclusion (V1)
`crossTAB_PNCL5M_after_exclusion_V1.ipynb` | Filter: PNCL5M + Included

In [13]:
print("\n" + "="*60)
print("SECTION 10: PNCL5M After Exclusion")
print("="*60)

df_pncl5m_incl = data1[(data1['Type of adoption'] == 'PNCL5M') &
                        (data1['Exclusion Reason PNC 2'] == 'Included')]

s10_pairs = [
    ("adoption age classification", "District"),
    ("adoption age classification", "Role Group"),
    ("adoption age classification", "Learner Category 2"),
    ("adoption age classification", "Type of adoption"),
    ("last visit age classification 1", "adoption age classification"),
    ("last visit age classification 2", "adoption age classification"),
    ("adoption duration baby", "adoption age classification"),

    ("visit category", "District"),
    ("visit category", "Role Group"),
    ("visit category", "Learner Category 2"),
    ("visit category", "Type of adoption"),
    ("visit category", "gestational week classification"),
    ("visit category", "adoption age classification"),
    ("visit category", "adoption duration baby"),
    ("visit category", "birthweight category"),
    ("visit category", "birthweight category 2"),

    ("bf assessment category", "District"),
    ("bf assessment category", "Role Group"),
    ("bf assessment category", "Learner Category 2"),

    ("cf assessment category", "District"),
    ("cf assessment category", "Role Group"),
    ("cf assessment category", "Learner Category 2"),

    ("bf assessment category", "Type of adoption"),
    ("cf assessment category", "Type of adoption"),

    ("bf assessment category", "adoption age classification"),
    ("cf assessment category", "adoption age classification"),

    ("bf assessment category", "adoption duration baby"),
    ("cf assessment category", "adoption duration baby"),

    ("bf assessment category", "birthweight category"),
    ("bf assessment category", "birthweight category 2"),

    ("bf assessment category", "visit category"),

    ("Activity Score", "District"),
    ("Activity Score", "Role Group"),
    ("Activity Score", "Learner Category 2"),
    ("Activity Score", "Type of adoption"),
    ("Activity Score", "adoption age classification"),
    ("Activity Score", "adoption duration baby"),

    ("WFA at LV/Change zscore LV AV", "District"),
    ("WFA at LV/Change zscore LV AV", "Role Group"),
    ("WFA at LV/Change zscore LV AV", "Learner Category 2"),
    ("WFA at LV/Change zscore LV AV", "Type of adoption"),
    ("WFA at LV/Change zscore LV AV", "Adoption type 1"),
    ("WFA at LV/Change zscore LV AV", "Mother Age Group"),
    ("WFA at LV/Change zscore LV AV", "Number of child's siblings_C"),
    ("WFA at LV/Change zscore LV AV", "Mother's education level_M"),
    ("WFA at LV/Change zscore LV AV", "Color of mother's family ration card_M"),
    ("WFA at LV/Change zscore LV AV", "Mother's social category_M"),
    ("WFA at LV/Change zscore LV AV", "gestational week classification"),
    ("WFA at LV/Change zscore LV AV", "adoption age classification"),
    ("WFA at LV/Change zscore LV AV", "adoption duration baby"),
    ("WFA at LV/Change zscore LV AV", "visit category"),
    ("WFA at LV/Change zscore LV AV", "bf assessment category"),
    ("WFA at LV/Change zscore LV AV", "cf assessment category"),
    ("WFA at LV/Change zscore LV AV", "Activity Score"),
]

run_section("S10: PNCL5M After Exclusion", df_pncl5m_incl, s10_pairs,
            "pncl5m_ct_after_exclusion_ML_050426_V(150426).xlsx")


SECTION 10: PNCL5M After Exclusion

--- S10: PNCL5M After Exclusion ---
  Data shape: (125, 1694)
  Pairs: 54


  Generated: 54 tables
  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/outputs\pncl5m_ct_after_exclusion_ML_050426_V(150426).xlsx (54 tables)


{'adoption age classification vs Blocks': Blocks                             Ambad     Badnapur    Bhokardan  \
 adoption age classification                                          
 d001_to_d015                  4 (26.67%)    6 (60.0%)  10 (58.82%)   
 d016_to_d030                   6 (40.0%)    2 (20.0%)   3 (17.65%)   
 d031_to_d060                  2 (13.33%)    2 (20.0%)   3 (17.65%)   
 d061_to_d090                  2 (13.33%)     0 (0.0%)     0 (0.0%)   
 d091_to_d120                   1 (6.67%)     0 (0.0%)     0 (0.0%)   
 d121_to_d150                    0 (0.0%)     0 (0.0%)    1 (5.88%)   
 Total                        15 (100.0%)  10 (100.0%)  17 (100.0%)   
 
 Blocks                       Ghansawangi     Jafrabad        Jalna  \
 adoption age classification                                          
 d001_to_d015                  2 (16.67%)    3 (30.0%)  21 (44.68%)   
 d016_to_d030                  4 (33.33%)    2 (20.0%)   8 (17.02%)   
 d031_to_d060                   3 

---
## Section 11 — PNCL5M After Exclusion (Copy1 — bf_assessment_category_2)
`crossTAB_PNCL5M_after_exclusion_V1-Copy1.ipynb` | Filter: PNCL5M + Included

In [14]:
print("\n" + "="*60)
print("SECTION 11: PNCL5M After Exclusion - BF Assess Cat 2")
print("="*60)

s11_pairs = [
    ("bf assessment category 2", "District"),
    ("bf assessment category 2", "Role Group"),
    ("bf assessment category 2", "Learner Category 2"),
    ("bf assessment category 2", "adoption age classification"),
    ("bf assessment category 2", "adoption duration baby"),
    ("bf assessment category 2", "birthweight category"),
    ("bf assessment category 2", "birthweight category 2"),
    ("bf assessment category 2", "birthweight group"),
    ("bf assessment category 2", "visit category"),
]

run_section("S11: PNCL5M After Excl (bf_cat_2)", df_pncl5m_incl, s11_pairs,
            "BF_Assess_pncl5m_ct_after_exclusion_ML_050426_V(150426).xlsx")


SECTION 11: PNCL5M After Exclusion - BF Assess Cat 2

--- S11: PNCL5M After Excl (bf_cat_2) ---
  Data shape: (125, 1694)
  Pairs: 9


  Generated: 9 tables
  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/outputs\BF_Assess_pncl5m_ct_after_exclusion_ML_050426_V(150426).xlsx (9 tables)


{'bf assessment category 2 vs Blocks': Blocks                          Ambad     Badnapur    Bhokardan  Ghansawangi  \
 bf assessment category 2                                                       
 00_assessment               1 (6.67%)    1 (10.0%)   2 (11.76%)    1 (8.33%)   
 01_assessment              4 (26.67%)    2 (20.0%)   5 (29.41%)   5 (41.67%)   
 02_assessments             2 (13.33%)    4 (40.0%)   2 (11.76%)    1 (8.33%)   
 03_assessments             4 (26.67%)     0 (0.0%)     0 (0.0%)    1 (8.33%)   
 04_to_05_assessments       4 (26.67%)    1 (10.0%)   5 (29.41%)   2 (16.67%)   
 06_or_more_assessments       0 (0.0%)    2 (20.0%)   3 (17.65%)   2 (16.67%)   
 Total                     15 (100.0%)  10 (100.0%)  17 (100.0%)  12 (100.0%)   
 
 Blocks                       Jafrabad        Jalna      Mantha      Partur  \
 bf assessment category 2                                                     
 00_assessment               1 (10.0%)   5 (10.64%)  2 (33.33%)   1 (12.5

---
## Section 12 — PNCG5M After Exclusion (V1)
`crossTAB_PNCG5M_after_exclusion_V1.ipynb` | Filter: PNCG5M + Included

In [15]:
print("\n" + "="*60)
print("SECTION 12: PNCG5M After Exclusion")
print("="*60)

df_pncg5m_incl = data1[(data1['Type of adoption'] == 'PNCG5M') &
                        (data1['Exclusion Reason PNC 2'] == 'Included')]

# PNCG5M after exclusion uses same pairs as PNCL5M after exclusion
run_section("S12: PNCG5M After Exclusion", df_pncg5m_incl, s10_pairs,
            "pncg5m_ct_after_exclusion_ML_050426_V(150426).xlsx")



SECTION 12: PNCG5M After Exclusion

--- S12: PNCG5M After Exclusion ---
  Data shape: (89, 1694)
  Pairs: 54


  Generated: 54 tables
  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/outputs\pncg5m_ct_after_exclusion_ML_050426_V(150426).xlsx (54 tables)


{'adoption age classification vs Blocks': Blocks                             Ambad     Badnapur    Bhokardan  \
 adoption age classification                                          
 d151_to_d180                  4 (36.36%)    5 (50.0%)   2 (18.18%)   
 d181_to_d210                  6 (54.55%)    5 (50.0%)   8 (72.73%)   
 d211_to_d240                   1 (9.09%)     0 (0.0%)    1 (9.09%)   
 d241_to_d270                    0 (0.0%)     0 (0.0%)     0 (0.0%)   
 d271_to_d300                    0 (0.0%)     0 (0.0%)     0 (0.0%)   
 Total                        11 (100.0%)  10 (100.0%)  11 (100.0%)   
 
 Blocks                       Ghansawangi    Jafrabad        Jalna      Mantha  \
 adoption age classification                                                     
 d151_to_d180                  2 (16.67%)   2 (25.0%)   5 (16.67%)  1 (33.33%)   
 d181_to_d210                   9 (75.0%)   6 (75.0%)  19 (63.33%)  2 (66.67%)   
 d211_to_d240                   1 (8.33%)    0 (0.0%)    3 (1

---
## Section 13 — PNCG5M After Exclusion (Copy1 — bf/cf_assessment_category_2)
`crossTAB_PNCG5M_after_exclusion_V1-Copy1.ipynb` | Filter: PNCG5M + Included

In [16]:
print("\n" + "="*60)
print("SECTION 13: PNCG5M After Exclusion - BF/CF Assess Cat 2")
print("="*60)

s13_pairs = [
    ("bf assessment category 2", "District"),
    ("bf assessment category 2", "Role Group"),
    ("bf assessment category 2", "Learner Category 2"),

    ("cf assessment category 2", "District"),
    ("cf assessment category 2", "Role Group"),
    ("cf assessment category 2", "Learner Category 2"),

    ("bf assessment category 2", "Type of adoption"),
    ("cf assessment category 2", "Type of adoption"),

    ("bf assessment category 2", "adoption age classification"),
    ("cf assessment category 2", "adoption age classification"),

    ("bf assessment category 2", "adoption duration baby"),
    ("cf assessment category 2", "adoption duration baby"),

    ("bf assessment category 2", "birthweight category"),
    ("bf assessment category 2", "birthweight category 2"),
    
    ("bf assessment category 2", "visit category"),
    ("cf assessment category 2", "visit category"),

    ("bf assessment category", "visit category"),
    ("cf assessment category", "visit category"),
]

run_section("S13: PNCG5M After Excl (bf/cf_cat_2)", df_pncg5m_incl, s13_pairs,
            "BF_CF_Assess_pncg5m_ct_after_exclusion_ML_230426_V(230426).xlsx")


SECTION 13: PNCG5M After Exclusion - BF/CF Assess Cat 2

--- S13: PNCG5M After Excl (bf/cf_cat_2) ---
  Data shape: (89, 1694)
  Pairs: 18


  Generated: 18 tables
  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/outputs\BF_CF_Assess_pncg5m_ct_after_exclusion_ML_230426_V(230426).xlsx (18 tables)


{'bf assessment category 2 vs Blocks': Blocks                          Ambad     Badnapur    Bhokardan  Ghansawangi  \
 bf assessment category 2                                                       
 00_assessment              2 (18.18%)    1 (10.0%)   2 (18.18%)   5 (41.67%)   
 01_assessment              5 (45.45%)    4 (40.0%)   4 (36.36%)    3 (25.0%)   
 02_assessments              1 (9.09%)    4 (40.0%)   2 (18.18%)    1 (8.33%)   
 03_assessments              1 (9.09%)    1 (10.0%)     0 (0.0%)   2 (16.67%)   
 04_to_05_assessments       2 (18.18%)     0 (0.0%)    1 (9.09%)     0 (0.0%)   
 06_or_more_assessments       0 (0.0%)     0 (0.0%)   2 (18.18%)    1 (8.33%)   
 Total                     11 (100.0%)  10 (100.0%)  11 (100.0%)  12 (100.0%)   
 
 Blocks                      Jafrabad        Jalna      Mantha      Partur  \
 bf assessment category 2                                                    
 00_assessment              1 (12.5%)  10 (33.33%)  2 (66.67%)    0 (0.0%) 

---
## Section 14 — ANC + PNCL5M After Exclusion
`crossTAB_ANC+PNCL5M_after_exclusion_V1.ipynb` | Filter: ANC or PNCL5M + Included (Exclusion Reason_ANC)

In [17]:
print("\n" + "="*60)
print("SECTION 14: ANC+PNCL5M After Exclusion")
print("="*60)

df_anc_pncl5m = data1[
    ((data1['Type of adoption'] == 'ANC') |
     (data1['Type of adoption'] == 'PNCL5M'))]
df_anc_pncl5m_incl = df_anc_pncl5m[df_anc_pncl5m['Exclusion Reason ANC 2'] == 'Included']

s14_pairs = [
    ("adoption age classification", "District"),
    ("adoption age classification", "Role Group"),
    ("adoption age classification", "Learner Category 2"),
    ("adoption age classification", "Type of adoption"),

    ("last visit age classification 1", "adoption age classification"),
    ("last visit age classification 2", "adoption age classification"),
    ("adoption duration baby", "adoption age classification"),

    ("visit category", "District"),
    ("visit category", "Role Group"),
    ("visit category", "Learner Category 2"),
    ("visit category", "Type of adoption"),
    ("visit category", "gestational week classification"),
    ("visit category", "adoption age classification"),
    ("visit category", "adoption duration baby"),
    ("visit category", "birthweight category"),
    ("visit category", "birthweight category 2"),

    ("bf assessment category", "District"),
    ("bf assessment category", "Role Group"),
    ("bf assessment category", "Learner Category 2"),
    ("bf assessment category", "Type of adoption"),
    ("bf assessment category", "adoption age classification"),
    ("bf assessment category", "adoption duration baby"),
    ("bf assessment category", "birthweight category"),
    ("bf assessment category", "birthweight category 2"),
    ("bf assessment category", "visit category"),

    ("Activity Score", "District"),
    ("Activity Score", "Role Group"),
    ("Activity Score", "Learner Category 2"),
    ("Activity Score", "Type of adoption"),
    ("Activity Score", "adoption age classification"),
    ("Activity Score", "adoption duration baby"),

    ("WFA at LV/Change zscore LV AV", "District"),
    ("WFA at LV/Change zscore LV AV", "Role Group"),
    ("WFA at LV/Change zscore LV AV", "Learner Category 2"),
    ("WFA at LV/Change zscore LV AV", "Type of adoption"),
    ("WFA at LV/Change zscore LV AV", "Adoption type 1"),
    ("WFA at LV/Change zscore LV AV", "Mother Age Group"),
    ("WFA at LV/Change zscore LV AV", "Number of child's siblings_C"),
    ("WFA at LV/Change zscore LV AV", "Mother's education level_M"),
    ("WFA at LV/Change zscore LV AV", "Color of mother's family ration card_M"),
    ("WFA at LV/Change zscore LV AV", "Mother's social category_M"),
    ("WFA at LV/Change zscore LV AV", "gestational week classification"),
    ("WFA at LV/Change zscore LV AV", "adoption age classification"),
    ("WFA at LV/Change zscore LV AV", "adoption duration baby"),
    ("WFA at LV/Change zscore LV AV", "visit category"),
    ("WFA at LV/Change zscore LV AV", "bf assessment category"),
    ("WFA at LV/Change zscore LV AV", "Activity Score"),
]

run_section("S14: ANC+PNCL5M After Exclusion", df_anc_pncl5m_incl, s14_pairs,
            "anc_pncl5m_ct_after_exclusion_ML_230426_V(230426).xlsx")



SECTION 14: ANC+PNCL5M After Exclusion

--- S14: ANC+PNCL5M After Exclusion ---
  Data shape: (183, 1694)
  Pairs: 47


  Generated: 47 tables


  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/outputs\anc_pncl5m_ct_after_exclusion_ML_230426_V(230426).xlsx (47 tables)


{'adoption age classification vs Blocks': Blocks                             Ambad     Badnapur    Bhokardan  \
 adoption age classification                                          
 d000                          9 (33.33%)     0 (0.0%)     1 (5.0%)   
 d001_to_d015                  6 (22.22%)   6 (46.15%)   12 (60.0%)   
 d016_to_d030                  6 (22.22%)   3 (23.08%)    3 (15.0%)   
 d031_to_d060                   2 (7.41%)   4 (30.77%)    3 (15.0%)   
 d061_to_d090                  3 (11.11%)     0 (0.0%)     0 (0.0%)   
 d091_to_d120                    1 (3.7%)     0 (0.0%)     0 (0.0%)   
 d121_to_d150                    0 (0.0%)     0 (0.0%)     1 (5.0%)   
 Total                        27 (100.0%)  13 (100.0%)  20 (100.0%)   
 
 Blocks                       Ghansawangi     Jafrabad        Jalna  \
 adoption age classification                                          
 d000                            0 (0.0%)     0 (0.0%)    7 (9.59%)   
 d001_to_d015                  2 (

---
## Section 15 — Proxy PNC ANC After Exclusion
`crossTAB_Proxy_PNC_ANC_col_after_exclusion_V1.ipynb` | Filter: PNCL5M or PNCG5M + Primary Exclusion_PNC_2 == Included

In [18]:
print("\n" + "="*60)
print("SECTION 15: Proxy PNC ANC Birthweight")
print("="*60)

df_pnc_both = data1[
    (data1['Type of adoption'] == 'PNCL5M') |
    (data1['Type of adoption'] == 'PNCG5M')]
df_pnc_both_incl = df_pnc_both[df_pnc_both['Primary Exclusion PNC 2'] == 'Included']

s15_pairs = [
    ("birthweight category", "District"),
    ("birthweight category 2", "District"),
    ("birthweight group", "District"),          # ← added

    ("birthweight category", "Role Group"),
    ("birthweight category 2", "Role Group"),
    ("birthweight group", "Role Group"),        # ← added

    ("birthweight category", "Learner Category 2"),
    ("birthweight category 2", "Learner Category 2"),
    ("birthweight group", "Learner Category 2"), # ← added

    ("birthweight category", "Type of adoption"),
    ("birthweight category 2", "Type of adoption"),
    ("birthweight group", "Type of adoption"),   # ← added

    ("birthweight category", "Mother Age Group"),
    ("birthweight category 2", "Mother Age Group"),
    ("birthweight group", "Mother Age Group"),   # ← added

    ("birthweight category", "Mother's education level_M"),
    ("birthweight category 2", "Mother's education level_M"),
    ("birthweight group", "Mother's education level_M"),  # ← added

    ("birthweight category", "Color of mother's family ration card_M"),
    ("birthweight category 2", "Color of mother's family ration card_M"),
    ("birthweight group", "Color of mother's family ration card_M"),  # ← added 
    
    ("birthweight category", "Mother's social category_M"),
    ("birthweight category 2", "Mother's social category_M"),
    ("birthweight group", "Mother's social category_M"),  # ← added

    ("birthweight category", "gestational week classification"),
    ("birthweight category 2", "gestational week classification"),
    ("birthweight group", "gestational week classification"),  # ← added

    ("birthweight category", "adoption age classification"),
    ("birthweight category 2", "adoption age classification"),
    ("birthweight group", "adoption age classification"),  # ← added

    ("birthweight category", "ANC protein category"),
    ("birthweight category 2", "ANC protein category"),
    ("birthweight group", "ANC protein category"),  # ← added

    ("birthweight category", "Mother adoption till birth"),
    ("birthweight category 2", "Mother adoption till birth"),
    ("birthweight group", "Mother adoption till birth"),  # ← added

    ("Type of adoption", "ANC 60d 3visit"),
    ("Type of adoption", "PNC lt5 60d 8visit"),
]

run_section("S15: Proxy PNC ANC Birthweight", df_pnc_both_incl, s15_pairs,
            "BW_proxy_ct_after_primexc_ML_050426_V(150426).xlsx")


SECTION 15: Proxy PNC ANC Birthweight

--- S15: Proxy PNC ANC Birthweight ---
  Data shape: (214, 1694)
  Pairs: 38


  Generated: 38 tables
  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/outputs\BW_proxy_ct_after_primexc_ML_050426_V(150426).xlsx (38 tables)


{'birthweight category vs Blocks': Blocks                      Ambad     Badnapur    Bhokardan  Ghansawangi  \
 birthweight category                                                       
 01.51_to_02.5_kg       5 (19.23%)    4 (20.0%)   4 (14.29%)   7 (29.17%)   
 02.51_to_02.7_kg       4 (15.38%)    7 (35.0%)    2 (7.14%)    1 (4.17%)   
 02.71_to_03.0_kg      10 (38.46%)    5 (25.0%)  11 (39.29%)  10 (41.67%)   
 03.01_to_03.5_kg       3 (11.54%)     1 (5.0%)   8 (28.57%)   4 (16.67%)   
 More_than_03.5_kg      4 (15.38%)    3 (15.0%)   3 (10.71%)    2 (8.33%)   
 Total                 26 (100.0%)  20 (100.0%)  28 (100.0%)  24 (100.0%)   
 
 Blocks                   Jafrabad        Jalna      Mantha       Partur  \
 birthweight category                                                      
 01.51_to_02.5_kg       4 (22.22%)  22 (28.57%)  2 (22.22%)   4 (33.33%)   
 02.51_to_02.7_kg        1 (5.56%)  12 (15.58%)  2 (22.22%)    1 (8.33%)   
 02.71_to_03.0_kg       6 (33.33%)  18 (23.3

---
## Section 16 — Activity & Outcome After Exclusion (Multiple Subsets)
`All_for_activity_and_outcome_after_exclusion_CT_V1-all_sort.ipynb`  
Five subsets: all_data, anc_pncl5m, anc, pncl5m, pncg5m — each as a separate sheet

In [19]:
print("\n" + "="*60)
print("SECTION 16: Activity & Outcome After Exclusion (5 subsets)")
print("="*60)

all_incl = data1[data1['Exclusion Reason PNC 2'] == 'Included']
anc_pncl5m_s16 = all_incl[(all_incl['Type of adoption'] == 'ANC') |
                           (all_incl['Type of adoption'] == 'PNCL5M')]
anc_s16 = all_incl[all_incl['Type of adoption'] == 'ANC']
pncl5m_s16 = all_incl[all_incl['Type of adoption'] == 'PNCL5M']
pncg5m_s16 = all_incl[all_incl['Type of adoption'] == 'PNCG5M']

s16_pairs = [
    ('adoption age classification', 'District'),
    ('adoption age classification', 'Role Group'),
    ('adoption age classification', 'Member Tag_CR'),
    ('adoption age classification', 'Type of adoption'),
    ('last visit age classification 1', 'adoption age classification'),
    ('last visit age classification 2', 'adoption age classification'),
    ('adoption duration baby', 'adoption age classification'),
    ('visit category', 'District'),
    ('visit category', 'Role Group'),
    ('visit category', 'Member Tag_CR'),
    ('visit category', 'Type of adoption'),
    ('visit category', 'gestational week classification'),
    ('visit category', 'adoption age classification'),
    ('visit category', 'adoption duration baby'),
    ('visit category', 'birthweight category'),
    ('visit category', 'birthweight category 2'),
    ('bf assessment category', 'District'),
    ('bf assessment category', 'Role Group'),
    ('bf assessment category', 'Member Tag_CR'),
    ('bf assessment category', 'visit category'),
    ('Activity Score', 'District'),
    ('Activity Score', 'Role Group'),
    ('Activity Score', 'Member Tag_CR'),
    ('Activity Score', 'Type of adoption'),
    ('Activity Score', 'adoption age classification'),
    ('Activity Score', 'adoption duration baby'),
    ('WFA status at BV 2', 'District'),
    ('WFA status at AV 2', 'District'),
    ('WFA status at LV 2', 'District'),
    ('WFA at LV/Change zscore LV AV', 'District'),
    ('WFA status at BV 2', 'Role Group'),
    ('WFA status at AV 2', 'Role Group'),
    ('WFA status at LV 2', 'Role Group'),
    ('WFA at LV/Change zscore LV AV', 'Role Group'),
    ('WFA status at BV 2', 'Member Tag_CR'),
    ('WFA status at AV 2', 'Member Tag_CR'),
    ('WFA status at LV 2', 'Member Tag_CR'),
    ('WFA at LV/Change zscore LV AV', 'Member Tag_CR'),
    ('WFA status at BV 2', 'Type of adoption'),
    ('WFA status at AV 2', 'Type of adoption'),
    ('WFA status at LV 2', 'Type of adoption'),
    ('WFA at LV/Change zscore LV AV', 'Type of adoption'),
    ('WFA status at BV 2', 'Adoption type 1'),
    ('WFA status at AV 2', 'Adoption type 1'),
    ('WFA status at LV 2', 'Adoption type 1'),
    ('WFA at LV/Change zscore LV AV', 'Adoption type 1'),
    ('WFA status at BV 2', 'Mother Age Group'),
    ('WFA status at AV 2', 'Mother Age Group'),
    ('WFA status at LV 2', 'Mother Age Group'),
    ('WFA at LV/Change zscore LV AV', 'Mother Age Group'),
    
    # Fixed mother-related columns
    ("WFA at LV/Change zscore LV AV", "Number of child's siblings_C"),
    ("WFA status at BV 2", "Mother's education level_M"),
    ("WFA status at AV 2", "Mother's education level_M"),
    ("WFA status at LV 2", "Mother's education level_M"),
    ("WFA at LV/Change zscore LV AV", "Mother's education level_M"),
    ("WFA status at BV 2", "Color of mother's family ration card_M"),
    ("WFA status at AV 2", "Color of mother's family ration card_M"),
    ("WFA status at LV 2", "Color of mother's family ration card_M"),
    ("WFA at LV/Change zscore LV AV", "Color of mother's family ration card_M"),
    ("WFA status at BV 2", "Mother's social category_M"),
    ("WFA status at AV 2", "Mother's social category_M"),
    ("WFA status at LV 2", "Mother's social category_M"),
    ("WFA at LV/Change zscore LV AV", "Mother's social category_M"),
    
    ('WFA status at BV 2', 'gestational week classification'),
    ('WFA status at AV 2', 'gestational week classification'),
    ('WFA status at LV 2', 'gestational week classification'),
    ('WFA at LV/Change zscore LV AV', 'gestational week classification'),
    ('WFA status at BV 2', 'adoption age classification'),
    ('WFA status at AV 2', 'adoption age classification'),
    ('WFA status at LV 2', 'adoption age classification'),
    ('WFA at LV/Change zscore LV AV', 'adoption age classification'),
    ('WFA status at BV 2', 'adoption duration baby'),
    ('WFA status at AV 2', 'adoption duration baby'),
    ('WFA status at LV 2', 'adoption duration baby'),
    ('WFA at LV/Change zscore LV AV', 'adoption duration baby'),
    ('WFA status at BV 2', 'visit category'),
    ('WFA status at AV 2', 'visit category'),
    ('WFA status at LV 2', 'visit category'),
    ('WFA at LV/Change zscore LV AV', 'visit category'),
    ('WFA status at BV 2', 'bf assessment category'),
    ('WFA status at AV 2', 'bf assessment category'),
    ('WFA status at LV 2', 'bf assessment category'),
    ('WFA at LV/Change zscore LV AV', 'bf assessment category'),
    ('WFA status at BV 2', 'Activity Score'),
    ('WFA status at AV 2', 'Activity Score'),
    ('WFA status at LV 2', 'Activity Score'),
    ('WFA at LV/Change zscore LV AV', 'Activity Score'),
]

run_section("S16a: All Included", all_incl, s16_pairs,
            "all_data_activity_and_outcome_CT_after_exclusion_ML_230426_V(230426).xlsx")
run_section("S16b: ANC+PNCL5M", anc_pncl5m_s16, s16_pairs,
            "anc_pncl5m_activity_and_outcome_CT_after_exclusion_ML_230426_V(230426).xlsx")
run_section("S16c: ANC Only", anc_s16, s16_pairs,
            "anc_activity_and_outcome_CT_after_exclusion_ML_230426_V(230426).xlsx")
run_section("S16d: PNCL5M Only", pncl5m_s16, s16_pairs,
            "pncl5m_activity_and_outcome_CT_after_exclusion_ML_230426_V(230426).xlsx")
run_section("S16e: PNCG5M Only", pncg5m_s16, s16_pairs,
            "pncg5m_activity_and_outcome_CT_after_exclusion_ML_230426_V(230426).xlsx")


# ==============================================================================
# SECTION EXTRA: ANC_60d Multi-Level Crosstab (from Scripts 8/9)
# This is a special analysis NOT captured in the original combined script
# ==============================================================================
print("\n" + "="*60)
print("SECTION EXTRA: ANC_60d by Birthweight Multi-Level Crosstab")
print("="*60)

try:
    ct = pd.crosstab(
        index=df_anc_incl['birthweight category 2'],
        columns=[df_anc_incl['Learner Category 2'], df_anc_incl['ANC 60d 3visit']],
        margins=False
    )

    desired_cols = [
        ('MT + FL', 'Yes'), ('MT + FL', 'No'),
        ('Other', 'Yes'), ('Other', 'No'),
    ]
    available_cols = [c for c in desired_cols if c in ct.columns]
    ct = ct[available_cols]

    ct[('Overall', 'Yes')] = ct.get(('MT + FL', 'Yes'), 0) + ct.get(('Other', 'Yes'), 0)
    ct[('Overall', 'No')] = ct.get(('MT + FL', 'No'), 0) + ct.get(('Other', 'No'), 0)
    ct['Total'] = ct.sum(axis=1)

    totals = ct.sum().to_frame().T
    totals.index = ['Total']
    ct = pd.concat([ct, totals])

    bw_row_order = ['Less_than_01.5_kg', '01.5_to_02.49_kg', '02.5_to_02.69_kg',
                    '02.7_to_02.99_kg', '03.0_to_03.49_kg', '03.5_or_more_kg', 'Total']
    ct = ct.reindex(bw_row_order).fillna(0).astype(int)
    ct.columns = pd.MultiIndex.from_tuples(ct.columns)

    # Birthweight group version
    ct_group = pd.crosstab(
        df_anc_incl['birthweight group'],
        [df_anc_incl['Learner Category 2'], df_anc_incl['ANC 60d 3visit']],
    )
    available_cols_g = [c for c in desired_cols if c in ct_group.columns]
    ct_group = ct_group[available_cols_g]
    ct_group[('Overall', 'Yes')] = ct_group.get(('MT + FL', 'Yes'), 0) + ct_group.get(('Other', 'Yes'), 0)
    ct_group[('Overall', 'No')] = ct_group.get(('MT + FL', 'No'), 0) + ct_group.get(('Other', 'No'), 0)
    ct_group['Total'] = ct_group.sum(axis=1)
    totals_g = ct_group.sum().to_frame().T
    totals_g.index = ['Total']
    ct_group = pd.concat([ct_group, totals_g])
    bw_grp_order = ['Low Birth Weight', 'Normal Birth Weight', 'Overweight', 'Total']
    ct_group = ct_group.reindex(bw_grp_order).fillna(0).astype(int)
    ct_group.columns = pd.MultiIndex.from_tuples(ct_group.columns)

    anc_file = os.path.join(
    OUTPUT_DIR,
    'ANC_60d_by_birthweight.xlsx'
    )

    # Clean index and columns to remove underscores and suffixes like _M, _P
    def clean_label_local(label):
        if not isinstance(label, str):
            return label
        import re
        if " vs " in label:
            parts = label.split(" vs ")
            return " vs ".join(clean_label_local(part) for part in parts)
        label = re.sub(r'_(?:[mMPpCc]|CR|cr|Cr|YN|yn|Yn)$', '', label)
        label = label.replace('_', ' ')
        label = re.sub(r'\s+', ' ', label)
        return label.strip()

    def clean_index_local(idx):
        if isinstance(idx, pd.MultiIndex):
            new_tuples = [tuple(clean_label_local(x) for x in t) for t in idx]
            new_names = [clean_label_local(name) if name else name for name in idx.names]
            return pd.MultiIndex.from_tuples(new_tuples, names=new_names)
        else:
            new_values = [clean_label_local(x) for x in idx]
            new_name = clean_label_local(idx.name) if idx.name else idx.name
            return pd.Index(new_values, name=new_name)

    ct.index = clean_index_local(ct.index)
    ct.columns = clean_index_local(ct.columns)
    ct_group.index = clean_index_local(ct_group.index)
    ct_group.columns = clean_index_local(ct_group.columns)

    if should_write_dir(OUTPUT_DIR):  # ADD-ON: mutually-exclusive Jalna/normal routing
        with pd.ExcelWriter(anc_file, engine='openpyxl') as writer:
            ct.to_excel(writer, sheet_name='BW_Categories')
            ct_group.to_excel(writer, sheet_name='LBW Normal Overweight')
        print(f"  Exported: {anc_file} (2 sheets)")
    else:
        print(f"  [skip] Jalna run -> ANC_60d not written to normal folder: {anc_file}")
except Exception as e:
    print(f"  Warning: Could not generate ANC_60d multi-level crosstab: {e}")


print("\n" + "="*60)
print("PIPELINE COMPLETE")
print("="*60)
print(f"Total output files: ~18 Excel files")



SECTION 16: Activity & Outcome After Exclusion (5 subsets)



--- S16a: All Included ---
  Data shape: (216, 1694)
  Pairs: 87


  Generated: 87 tables


  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/outputs\all_data_activity_and_outcome_CT_after_exclusion_ML_230426_V(230426).xlsx (87 tables)

--- S16b: ANC+PNCL5M ---
  Data shape: (127, 1694)
  Pairs: 87


  Generated: 87 tables


...

  Generated: 87 tables
  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/outputs\anc_activity_and_outcome_CT_after_exclusion_ML_230426_V(230426).xlsx (87 tables)

--- S16d: PNCL5M Only ---
  Data shape: (125, 1694)
  Pairs: 87


  Generated: 87 tables


  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/outputs\pncl5m_activity_and_outcome_CT_after_exclusion_ML_230426_V(230426).xlsx (87 tables)

--- S16e: PNCG5M Only ---
  Data shape: (89, 1694)
  Pairs: 87


  Generated: 87 tables


  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/outputs\pncg5m_activity_and_outcome_CT_after_exclusion_ML_230426_V(230426).xlsx (87 tables)

SECTION EXTRA: ANC_60d by Birthweight Multi-Level Crosstab
  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/outputs\ANC_60d_by_birthweight.xlsx (2 sheets)

PIPELINE COMPLETE
Total output files: ~18 Excel files


---
## ROW-WISE VERSION → `output_2`

Regenerates the **exact same set of crosstabs, in the exact same order**, with **identical counts, structure, labels and Calibri-24pt formatting** as the `outputs` folder.

The **only** difference: the percentage inside every `count (percent%)` cell is computed **row-wise** (each cell ÷ its **row** total) instead of column-wise (each cell ÷ its column total).

All files are written to the `output_2` folder using the same file names as `outputs`.

In [20]:
# ==============================================================================
# ROW-WISE PERCENTAGE PIPELINE  ->  output_2
# Same crosstabs, same order, same counts / labels / formatting.
# ONLY the percentage direction changes:
#   column-wise  = cell / COLUMN total   (original -> outputs)
#   row-wise     = cell / ROW total      (this block -> output_2)
# ==============================================================================

OUTPUT_DIR_2 = "C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/output_2"
_register_normal_dir(OUTPUT_DIR_2)        # mutually-exclusive routing: mark output_2/ as a normal folder
if should_write_dir(OUTPUT_DIR_2):
    os.makedirs(OUTPUT_DIR_2, exist_ok=True)


def generate_specific_crosstabs_rowwise(data, crosstab_pairs, row_vars=None,
                                        row_order=None, col_order=None,
                                        row_category_labels=None,
                                        column_category_labels=None):
    """Row-wise twin of generate_specific_crosstabs().
    Identical in every respect EXCEPT percentages are cell / ROW total."""
    crosstab_pairs = apply_geo_level(apply_report_order(add_learner_cat3_pairs(crosstab_pairs)))  # ADD-ONs: 'Learner Category 3' twin pairs, Word-report table order, then district-level 'Blocks' swap (inert at state level)
    cross_tabs = {}

    for row_col, col_col in crosstab_pairs:
        if row_col not in data.columns or col_col not in data.columns:
            continue

        display_name = row_vars.get(row_col, row_col) if row_vars else row_col

        clean_df = data.copy()

        # ---------------- RAW CROSSTAB (NO TOTAL) ----------------
        count_table = pd.crosstab(
            clean_df[row_col],
            clean_df[col_col],
            dropna=True,
            margins=True,
            margins_name="Total"
        )
        if count_table.empty or count_table.shape[1] == 0:
            final_table = pd.DataFrame()
            final_table.index.name = display_name
            cross_tabs[f"{display_name} vs {col_col}"] = final_table
            continue

        # ---------------- APPLY ROW ORDER ----------------
        if row_order and row_col in row_order:
            ordered_rows = (
                [x for x in row_order[row_col] if x in count_table.index]
                + [x for x in count_table.index if x not in row_order[row_col]]
            )
            count_table = count_table.loc[ordered_rows]

        # ---------------- APPLY COLUMN ORDER ----------------
        if col_order and col_col in col_order:
            ordered_cols = (
                [x for x in col_order[col_col] if x in count_table.columns]
                + [x for x in count_table.columns if x not in col_order[col_col]]
            )
            count_table = count_table[ordered_cols]

        # ---------------- PERCENT TABLE (ROW-WISE) ----------------
        percent_table = (
            count_table
            .iloc[:-1, :-1]
            .div(count_table["Total"].iloc[:-1], axis=0)   # <-- each cell / its ROW total
            .replace([np.inf, -np.inf], 0)
            .fillna(0)
            * 100
        ).round(2)

        # Total column: each row divided by its own row total = 100%
        percent_table["Total"] = 100.0

        # Total row (bottom margin): each column total / grand total
        grand_total = count_table.loc["Total", "Total"]
        if grand_total != 0:
            percent_table.loc["Total"] = (count_table.loc["Total"] / grand_total * 100).round(2)
        else:
            percent_table.loc["Total"] = 0.0

        # ---------------- DROP z_NA FROM DISPLAY (still counted in Totals) ----------------
        zna_rows = [x for x in count_table.index
                    if ' '.join(str(x).strip().lower().replace('_', ' ').split()) in ('z na', 'zna')]
        zna_cols = [c for c in count_table.columns
                    if ' '.join(str(c).strip().lower().replace('_', ' ').split()) in ('z na', 'zna')]
        if zna_rows or zna_cols:
            count_table = count_table.drop(index=zna_rows, columns=zna_cols)
            percent_table = percent_table.drop(index=zna_rows, columns=zna_cols, errors='ignore')

        # ---------------- LABEL MAPPING ----------------
        if row_category_labels and row_col in row_category_labels:
            count_table.index = [
                row_category_labels[row_col].get(x, x) for x in count_table.index
            ]
            percent_table.index = count_table.index

        if column_category_labels and col_col in column_category_labels:
            count_table.columns = [
                column_category_labels[col_col].get(x, x) for x in count_table.columns
            ]
            percent_table.columns = count_table.columns

        # ---------------- FINAL TABLE ----------------
        final_table = (
            count_table.astype(int).astype(str)
            + " ("
            + percent_table.astype(str)
            + "%)"
        )

        final_table.index.name = display_name
        cross_tabs[f"{display_name} vs {col_col}"] = final_table

    return cross_tabs


def generate_specific_crosstabs_v2_rowwise(data, crosstab_pairs, row_vars=None,
                                           row_order=None, col_order=None,
                                           row_category_labels=None,
                                           column_category_labels=None):
    """Row-wise twin of generate_specific_crosstabs_v2().
    Same invalid-value cleaning and margin handling; only the percentage
    direction changes to cell / ROW total."""
    crosstab_pairs = apply_geo_level(apply_report_order(add_learner_cat3_pairs(crosstab_pairs)))  # ADD-ONs: 'Learner Category 3' twin pairs, Word-report table order, then district-level 'Blocks' swap (inert at state level)
    cross_tabs = {}

    # Exact set of invalids from the notebook
    invalid_vals = {np.nan, None, 'nan', 'null', 'na', 'zna', 'z_na', 'z_NA'}

    for row_col, col_col in crosstab_pairs:
        if row_col not in data.columns or col_col not in data.columns:
            continue

        display_name = row_vars.get(row_col, row_col) if row_vars else row_col

        # ---------------- CLEAN DATA FOR COUNTING ----------------
        clean_df = data.copy()
        clean_df[row_col] = clean_df[row_col].apply(
            lambda x: np.nan if str(x).strip().lower() in invalid_vals else x
        )

        # ---------------- RAW CROSSTAB (NO TOTAL) ----------------
        count_table = pd.crosstab(
            clean_df[row_col],
            clean_df[col_col],
            dropna=True,
            margins=True,
            margins_name="Total"
        )
        if count_table.empty or count_table.shape[1] == 0:
            continue

        # ---------------- APPLY ROW ORDER ----------------
        if row_order and row_col in row_order:
            ordered_rows = (
                [x for x in row_order[row_col] if x in count_table.index]
                + [x for x in count_table.index if x not in row_order[row_col]]
            )
            count_table = count_table.loc[ordered_rows]

        # ---------------- APPLY COLUMN ORDER ----------------
        if col_order and col_col in col_order:
            ordered_cols = (
                [x for x in col_order[col_col] if x in count_table.columns]
                + [x for x in count_table.columns if x not in col_order[col_col]]
            )
            count_table = count_table[ordered_cols]

        # ---------------- PERCENT TABLE (ROW-WISE) ----------------
        percent_table = (
            count_table
            .iloc[:-1, :-1]
            .div(count_table["Total"].iloc[:-1], axis=0)   # <-- each cell / its ROW total
            .replace([np.inf, -np.inf], 0)
            .fillna(0)
            * 100
        ).round(2)

        # Margins hardcoded to 100 (mirrors generate_specific_crosstabs_v2)
        percent_table["Total"] = 100.0
        percent_table.loc["Total"] = 100.0

        # ---------------- DROP z_NA FROM DISPLAY (still counted in Totals) ----------------
        zna_rows = [x for x in count_table.index
                    if ' '.join(str(x).strip().lower().replace('_', ' ').split()) in ('z na', 'zna')]
        zna_cols = [c for c in count_table.columns
                    if ' '.join(str(c).strip().lower().replace('_', ' ').split()) in ('z na', 'zna')]
        if zna_rows or zna_cols:
            count_table = count_table.drop(index=zna_rows, columns=zna_cols)
            percent_table = percent_table.drop(index=zna_rows, columns=zna_cols, errors='ignore')

        # ---------------- LABEL MAPPING ----------------
        if row_category_labels and row_col in row_category_labels:
            count_table.index = [
                row_category_labels[row_col].get(x, x) for x in count_table.index
            ]
            percent_table.index = count_table.index

        if column_category_labels and col_col in column_category_labels:
            count_table.columns = [
                column_category_labels[col_col].get(x, x) for x in count_table.columns
            ]
            percent_table.columns = count_table.columns

        # ---------------- FINAL TABLE ----------------
        final_table = (
            count_table.astype(int).astype(str)
            + " ("
            + percent_table.astype(str)
            + "%)"
        )

        final_table.index.name = display_name
        cross_tabs[f"{display_name} vs {col_col}"] = final_table

    return cross_tabs


def export_tables_to_single_sheet_to(tables_dict, file_name, output_dir,
                                     sheet_title="CombinedTables"):
    """Same as export_tables_to_single_sheet() (identical formatting) but writes
    into `output_dir` instead of the global OUTPUT_DIR."""
    global OUTPUT_DIR
    _saved = OUTPUT_DIR
    OUTPUT_DIR = output_dir
    try:
        export_tables_to_single_sheet(tables_dict, file_name, sheet_title)
    finally:
        OUTPUT_DIR = _saved


def run_section_rowwise(section_name, data, pairs, output_file):
    """Row-wise twin of run_section(): exports to OUTPUT_DIR_2."""
    print(f"\n--- [ROW-WISE] {section_name} ---")
    print(f"  Data shape: {data.shape}")
    print(f"  Pairs: {len(pairs)}")
    results = generate_specific_crosstabs_rowwise(
        data=data,
        crosstab_pairs=pairs,
        row_order=row_order,
        col_order=column_order
    )
    print(f"  Generated: {len(results)} tables")
    export_tables_to_single_sheet_to(results, output_file, OUTPUT_DIR_2)
    return results

In [21]:
print("\n" + "#"*60)
print("ROW-WISE PIPELINE  ->  output_2  (same crosstabs, row-wise percentages)")
print("#"*60)

# ---- Section 1 : All MCD Before Exclusion ----
print("\n" + "="*60); print("SECTION 1 (row-wise): All MCD Before Exclusion"); print("="*60)
run_section_rowwise("S1: All MCD Before Exclusion", data1, s1_pairs,
                    "crosstab_all_data_ML_230426_V(230426).xlsx")

# ---- Section 2 : Included / Excluded All  (uses the v2 generator) ----
print("\n" + "="*60); print("SECTION 2 (row-wise): Included/Excluded All"); print("="*60)
print(f"  Data shape: {data1.shape}")
print(f"  Pairs: {len(s2_pairs)}")
s2_results_rw = generate_specific_crosstabs_v2_rowwise(
    data=data1, crosstab_pairs=s2_pairs,
    row_order=row_order, col_order=column_order)
print(f"  Generated: {len(s2_results_rw)} tables")
export_tables_to_single_sheet_to(s2_results_rw,
    "included_excluded_CT_ML_100526_V(120526).xlsx", OUTPUT_DIR_2)

# ---- Section 3 : Learners (before + after exclusion) ----
print("\n" + "="*60); print("SECTION 3 (row-wise): Learners"); print("="*60)
run_section_rowwise("S3a: Learners Before Exclusion", df_learners, s3_pairs,
                    "learner_based_ct_before_exclusion_ML_100526_V(120526).xlsx")
run_section_rowwise("S3b: Learners After Exclusion", df_learners_incl, s3_pairs,
                    "learner_based_ct_after_exclusion_ML_100526_V(120526).xlsx")

# ---- Section 4 : All MCD After Exclusion ----
print("\n" + "="*60); print("SECTION 4 (row-wise): All MCD After Exclusion"); print("="*60)
run_section_rowwise("S4: All MCD After Exclusion", all_data_s4, s4_pairs,
                    "crosstab_after_exclusion_ML_230426_V(230426).xlsx")

# ---- Section 5 : ANC Before Exclusion ----
print("\n" + "="*60); print("SECTION 5 (row-wise): ANC Before Exclusion"); print("="*60)
run_section_rowwise("S5: ANC Before Exclusion", df_anc_all, s5_pairs,
                    "only_anc_ct_before_exclusion_ML_230426_V(230426).xlsx")

# ---- Section 6 : PNCL5M Before Exclusion ----
print("\n" + "="*60); print("SECTION 6 (row-wise): PNCL5M Before Exclusion"); print("="*60)
run_section_rowwise("S6: PNCL5M Before Exclusion", df_pncl5m_all, s6_pairs,
                    "pncl5m_ct_before_exclusion_ML_050426_V(150426).xlsx")

# ---- Section 7 : PNCG5M Before Exclusion ----
print("\n" + "="*60); print("SECTION 7 (row-wise): PNCG5M Before Exclusion"); print("="*60)
run_section_rowwise("S7: PNCG5M Before Exclusion", df_pncg5m_all, s7_pairs,
                    "pncg5m_ct_before_exclusion_ML_050426_V(150426).xlsx")

# ---- Section 8 : ANC After Exclusion ----
print("\n" + "="*60); print("SECTION 8 (row-wise): ANC After Exclusion"); print("="*60)
run_section_rowwise("S8: ANC After Exclusion", df_anc_incl, s8_pairs,
                    "only_anc_ct_after_exclusion_ML_230426_V(230426).xlsx")

# ---- Section 9 : ANC After Exclusion (bf_assessment_category_2) ----
print("\n" + "="*60); print("SECTION 9 (row-wise): ANC After Exclusion - BF Assess Cat 2"); print("="*60)
run_section_rowwise("S9: ANC After Excl (bf_cat_2)", df_anc_incl, s9_pairs,
                    "BF_Assess_only_anc_ct_after_exclusion_ML_230426_V(230426).xlsx")

# ---- Section 10 : PNCL5M After Exclusion ----
print("\n" + "="*60); print("SECTION 10 (row-wise): PNCL5M After Exclusion"); print("="*60)
run_section_rowwise("S10: PNCL5M After Exclusion", df_pncl5m_incl, s10_pairs,
                    "pncl5m_ct_after_exclusion_ML_050426_V(150426).xlsx")

# ---- Section 11 : PNCL5M After Exclusion (bf_assessment_category_2) ----
print("\n" + "="*60); print("SECTION 11 (row-wise): PNCL5M After Exclusion - BF Assess Cat 2"); print("="*60)
run_section_rowwise("S11: PNCL5M After Excl (bf_cat_2)", df_pncl5m_incl, s11_pairs,
                    "BF_Assess_pncl5m_ct_after_exclusion_ML_050426_V(150426).xlsx")

# ---- Section 12 : PNCG5M After Exclusion (same pairs as S10) ----
print("\n" + "="*60); print("SECTION 12 (row-wise): PNCG5M After Exclusion"); print("="*60)
run_section_rowwise("S12: PNCG5M After Exclusion", df_pncg5m_incl, s10_pairs,
                    "pncg5m_ct_after_exclusion_ML_050426_V(150426).xlsx")

# ---- Section 13 : PNCG5M After Exclusion (bf/cf_assessment_category_2) ----
print("\n" + "="*60); print("SECTION 13 (row-wise): PNCG5M After Exclusion - BF/CF Assess Cat 2"); print("="*60)
run_section_rowwise("S13: PNCG5M After Excl (bf/cf_cat_2)", df_pncg5m_incl, s13_pairs,
                    "BF_CF_Assess_pncg5m_ct_after_exclusion_ML_230426_V(230426).xlsx")

# ---- Section 14 : ANC + PNCL5M After Exclusion ----
print("\n" + "="*60); print("SECTION 14 (row-wise): ANC+PNCL5M After Exclusion"); print("="*60)
run_section_rowwise("S14: ANC+PNCL5M After Exclusion", df_anc_pncl5m_incl, s14_pairs,
                    "anc_pncl5m_ct_after_exclusion_ML_230426_V(230426).xlsx")

# ---- Section 15 : Proxy PNC ANC Birthweight ----
print("\n" + "="*60); print("SECTION 15 (row-wise): Proxy PNC ANC Birthweight"); print("="*60)
run_section_rowwise("S15: Proxy PNC ANC Birthweight", df_pnc_both_incl, s15_pairs,
                    "BW_proxy_ct_after_primexc_ML_050426_V(150426).xlsx")

# ---- Section 16 : Activity & Outcome After Exclusion (5 subsets) ----
print("\n" + "="*60); print("SECTION 16 (row-wise): Activity & Outcome After Exclusion (5 subsets)"); print("="*60)
run_section_rowwise("S16a: All Included", all_incl, s16_pairs,
                    "all_data_activity_and_outcome_CT_after_exclusion_ML_230426_V(230426).xlsx")
run_section_rowwise("S16b: ANC+PNCL5M", anc_pncl5m_s16, s16_pairs,
                    "anc_pncl5m_activity_and_outcome_CT_after_exclusion_ML_230426_V(230426).xlsx")
run_section_rowwise("S16c: ANC Only", anc_s16, s16_pairs,
                    "anc_activity_and_outcome_CT_after_exclusion_ML_230426_V(230426).xlsx")
run_section_rowwise("S16d: PNCL5M Only", pncl5m_s16, s16_pairs,
                    "pncl5m_activity_and_outcome_CT_after_exclusion_ML_230426_V(230426).xlsx")
run_section_rowwise("S16e: PNCG5M Only", pncg5m_s16, s16_pairs,
                    "pncg5m_activity_and_outcome_CT_after_exclusion_ML_230426_V(230426).xlsx")


############################################################
ROW-WISE PIPELINE  ->  output_2  (same crosstabs, row-wise percentages)
############################################################

SECTION 1 (row-wise): All MCD Before Exclusion

--- [ROW-WISE] S1: All MCD Before Exclusion ---
  Data shape: (590, 1694)
  Pairs: 21


  Generated: 15 tables
  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/output_2\crosstab_all_data_ML_230426_V(230426).xlsx (15 tables)

SECTION 2 (row-wise): Included/Excluded All
  Data shape: (590, 1694)
  Pairs: 50


  Generated: 49 tables
  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/output_2\included_excluded_CT_ML_100526_V(120526).xlsx (49 tables)

SECTION 3 (row-wise): Learners

--- [ROW-WISE] S3a: Learners Before Exclusion ---
  Data shape: (192, 1694)
  Pairs: 14


  Generated: 14 tables
  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/output_2\learner_based_ct_before_exclusion_ML_100526_V(120526).xlsx (14 tables)

--- [ROW-WISE] S3b: Learners After Exclusion ---
  Data shape: (50, 1694)
  Pairs: 14


  Generated: 14 tables
  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/output_2\learner_based_ct_after_exclusion_ML_100526_V(120526).xlsx (14 tables)

SECTION 4 (row-wise): All MCD After Exclusion

--- [ROW-WISE] S4: All MCD After Exclusion ---
  Data shape: (216, 1694)
  Pairs: 76


...

  Generated: 87 tables


  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/output_2\pncl5m_activity_and_outcome_CT_after_exclusion_ML_230426_V(230426).xlsx (87 tables)

--- [ROW-WISE] S16e: PNCG5M Only ---
  Data shape: (89, 1694)
  Pairs: 87


  Generated: 87 tables


  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/output_2\pncg5m_activity_and_outcome_CT_after_exclusion_ML_230426_V(230426).xlsx (87 tables)


{'adoption age classification vs Blocks': Blocks                             Ambad     Badnapur    Bhokardan  \
 adoption age classification                                          
 d151_to_d180                  4 (18.18%)   5 (22.73%)    2 (9.09%)   
 d181_to_d210                  6 (10.71%)    5 (8.93%)   8 (14.29%)   
 d211_to_d240                   1 (12.5%)     0 (0.0%)    1 (12.5%)   
 d241_to_d270                    0 (0.0%)     0 (0.0%)     0 (0.0%)   
 d271_to_d300                    0 (0.0%)     0 (0.0%)     0 (0.0%)   
 Total                        11 (12.36%)  10 (11.24%)  11 (12.36%)   
 
 Blocks                       Ghansawangi    Jafrabad        Jalna     Mantha  \
 adoption age classification                                                    
 d151_to_d180                   2 (9.09%)   2 (9.09%)   5 (22.73%)  1 (4.55%)   
 d181_to_d210                  9 (16.07%)  6 (10.71%)  19 (33.93%)  2 (3.57%)   
 d211_to_d240                   1 (12.5%)    0 (0.0%)    3 (37.5%

In [22]:
# ==============================================================================
# SECTION EXTRA (row-wise pipeline): ANC_60d Multi-Level Crosstab  ->  output_2
# NOTE: this table is COUNTS ONLY (it has no percentages), so it is identical
# to the version in `outputs`; it is regenerated here only so that output_2
# holds the complete set of files.
# ==============================================================================
print("\n" + "="*60)
print("SECTION EXTRA (row-wise): ANC_60d by Birthweight Multi-Level Crosstab")
print("="*60)

try:
    ct = pd.crosstab(
        index=df_anc_incl['birthweight category 2'],
        columns=[df_anc_incl['Learner Category 2'], df_anc_incl['ANC 60d 3visit']],
        margins=False
    )

    desired_cols = [
        ('MT + FL', 'Yes'), ('MT + FL', 'No'),
        ('Other', 'Yes'), ('Other', 'No'),
    ]
    available_cols = [c for c in desired_cols if c in ct.columns]
    ct = ct[available_cols]

    ct[('Overall', 'Yes')] = ct.get(('MT + FL', 'Yes'), 0) + ct.get(('Other', 'Yes'), 0)
    ct[('Overall', 'No')] = ct.get(('MT + FL', 'No'), 0) + ct.get(('Other', 'No'), 0)
    ct['Total'] = ct.sum(axis=1)

    totals = ct.sum().to_frame().T
    totals.index = ['Total']
    ct = pd.concat([ct, totals])

    bw_row_order = ['Less_than_01.5_kg', '01.5_to_02.49_kg', '02.5_to_02.69_kg',
                    '02.7_to_02.99_kg', '03.0_to_03.49_kg', '03.5_or_more_kg', 'Total']
    ct = ct.reindex(bw_row_order).fillna(0).astype(int)
    ct.columns = pd.MultiIndex.from_tuples(ct.columns)

    # Birthweight group version
    ct_group = pd.crosstab(
        df_anc_incl['birthweight group'],
        [df_anc_incl['Learner Category 2'], df_anc_incl['ANC 60d 3visit']],
    )
    available_cols_g = [c for c in desired_cols if c in ct_group.columns]
    ct_group = ct_group[available_cols_g]
    ct_group[('Overall', 'Yes')] = ct_group.get(('MT + FL', 'Yes'), 0) + ct_group.get(('Other', 'Yes'), 0)
    ct_group[('Overall', 'No')] = ct_group.get(('MT + FL', 'No'), 0) + ct_group.get(('Other', 'No'), 0)
    ct_group['Total'] = ct_group.sum(axis=1)
    totals_g = ct_group.sum().to_frame().T
    totals_g.index = ['Total']
    ct_group = pd.concat([ct_group, totals_g])
    bw_grp_order = ['Low Birth Weight', 'Normal Birth Weight', 'Overweight', 'Total']
    ct_group = ct_group.reindex(bw_grp_order).fillna(0).astype(int)
    ct_group.columns = pd.MultiIndex.from_tuples(ct_group.columns)

    anc_file = os.path.join(
        OUTPUT_DIR_2,
        'ANC_60d_by_birthweight.xlsx'
    )

    # Clean index and columns to remove underscores and suffixes like _M, _P
    def clean_label_local(label):
        if not isinstance(label, str):
            return label
        import re
        if " vs " in label:
            parts = label.split(" vs ")
            return " vs ".join(clean_label_local(part) for part in parts)
        label = re.sub(r'_(?:[mMPpCc]|CR|cr|Cr|YN|yn|Yn)$', '', label)
        label = label.replace('_', ' ')
        label = re.sub(r'\s+', ' ', label)
        return label.strip()

    def clean_index_local(idx):
        if isinstance(idx, pd.MultiIndex):
            new_tuples = [tuple(clean_label_local(x) for x in t) for t in idx]
            new_names = [clean_label_local(name) if name else name for name in idx.names]
            return pd.MultiIndex.from_tuples(new_tuples, names=new_names)
        else:
            new_values = [clean_label_local(x) for x in idx]
            new_name = clean_label_local(idx.name) if idx.name else idx.name
            return pd.Index(new_values, name=new_name)

    ct.index = clean_index_local(ct.index)
    ct.columns = clean_index_local(ct.columns)
    ct_group.index = clean_index_local(ct_group.index)
    ct_group.columns = clean_index_local(ct_group.columns)

    if should_write_dir(OUTPUT_DIR_2):  # ADD-ON: mutually-exclusive Jalna/normal routing
        with pd.ExcelWriter(anc_file, engine='openpyxl') as writer:
            ct.to_excel(writer, sheet_name='BW_Categories')
            ct_group.to_excel(writer, sheet_name='LBW Normal Overweight')
        print(f"  Exported: {anc_file} (2 sheets)")
    else:
        print(f"  [skip] Jalna run -> ANC_60d not written to normal folder: {anc_file}")
except Exception as e:
    print(f"  Warning: Could not generate ANC_60d multi-level crosstab: {e}")


print("\n" + "="*60)
print("ROW-WISE PIPELINE COMPLETE  ->  output_2")
print("="*60)
print("Row-wise output files written to:", OUTPUT_DIR_2)


SECTION EXTRA (row-wise): ANC_60d by Birthweight Multi-Level Crosstab
  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/output_2\ANC_60d_by_birthweight.xlsx (2 sheets)

ROW-WISE PIPELINE COMPLETE  ->  output_2
Row-wise output files written to: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/output_2


---
## JALNA PIPELINE (situational add-on) -> `output_jalna` + `output_jalna_2`

Runs **only** when the input file name carries the `JL` code (`IS_JALNA`).
Regenerates **every crosstab of the normal pipeline** with the two Jalna
adoption-group columns substituted in every pair:

| normal column | Jalna column used instead |
|---|---|
| `Total Adoptions group` | `Total Adoptions group jalna` |
| `Total Adoptions group after exclusion` | `Total Adoptions group after exclusion jalna` |

* `output_jalna`   = column-wise % (twin of `outputs`)
* `output_jalna_2` = row-wise % (twin of `output_2`)

Non-Jalna inputs skip this section entirely, so the normal combined script keeps
populating `outputs` / `output_2` exactly as before.

In [23]:
# ==============================================================================
# JALNA PIPELINE (situational add-on)  ->  output_jalna / output_jalna_2
# Runs ONLY when the input file carries the "JL" code (IS_JALNA).
# The normal pipeline above is untouched.
# ==============================================================================
if not IS_JALNA:
    print("Input file is NOT a Jalna (JL) file -> Jalna pipeline skipped.")
else:
    OUTPUT_DIR_JALNA = "C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/output_jalna"
    OUTPUT_DIR_JALNA_2 = "C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/output_jalna_2"
    os.makedirs(OUTPUT_DIR_JALNA, exist_ok=True)
    os.makedirs(OUTPUT_DIR_JALNA_2, exist_ok=True)

    import re as _re_jl
    _base = os.path.basename(INPUT_FILE)
    _m_code = _re_jl.search(r'([A-Za-z]{2})_(\d{6})', _base)
    _m_ver = _re_jl.search(r'V\((\d{6})\)', _base)

    def jl_filename(fname):
        """Re-stamp an ML_xxxxxx / V(xxxxxx) output name with the Jalna input's
        own code, date and version so file names stay traceable to the input."""
        out = fname
        if _m_code:
            out = _re_jl.sub(r'ML_\d{6}', f"{_m_code.group(1).upper()}_{_m_code.group(2)}", out)
        else:
            out = out.replace('_ML_', '_JL_')
        if _m_ver:
            out = _re_jl.sub(r'V\(\d{6}\)', f"V({_m_ver.group(1)})", out)
        return out

    # Every export of the normal pipeline: (section, data, pairs, generator, normal file name)
    jl_sections = [
        ("S1: All MCD Before Exclusion",         data1,              s1_pairs,  'v1', "crosstab_all_data_ML_230426_V(230426).xlsx"),
        ("S2: Included/Excluded All",            data1,              s2_pairs,  'v2', "included_excluded_CT_ML_100526_V(120526).xlsx"),
        ("S3a: Learners Before Exclusion",       df_learners,        s3_pairs,  'v1', "learner_based_ct_before_exclusion_ML_100526_V(120526).xlsx"),
        ("S3b: Learners After Exclusion",        df_learners_incl,   s3_pairs,  'v1', "learner_based_ct_after_exclusion_ML_100526_V(120526).xlsx"),
        ("S4: All MCD After Exclusion",          all_data_s4,        s4_pairs,  'v1', "crosstab_after_exclusion_ML_230426_V(230426).xlsx"),
        ("S5: ANC Before Exclusion",             df_anc_all,         s5_pairs,  'v1', "only_anc_ct_before_exclusion_ML_230426_V(230426).xlsx"),
        ("S6: PNCL5M Before Exclusion",          df_pncl5m_all,      s6_pairs,  'v1', "pncl5m_ct_before_exclusion_ML_050426_V(150426).xlsx"),
        ("S7: PNCG5M Before Exclusion",          df_pncg5m_all,      s7_pairs,  'v1', "pncg5m_ct_before_exclusion_ML_050426_V(150426).xlsx"),
        ("S8: ANC After Exclusion",              df_anc_incl,        s8_pairs,  'v1', "only_anc_ct_after_exclusion_ML_230426_V(230426).xlsx"),
        ("S9: ANC After Excl (bf_cat_2)",        df_anc_incl,        s9_pairs,  'v1', "BF_Assess_only_anc_ct_after_exclusion_ML_230426_V(230426).xlsx"),
        ("S10: PNCL5M After Exclusion",          df_pncl5m_incl,     s10_pairs, 'v1', "pncl5m_ct_after_exclusion_ML_050426_V(150426).xlsx"),
        ("S11: PNCL5M After Excl (bf_cat_2)",    df_pncl5m_incl,     s11_pairs, 'v1', "BF_Assess_pncl5m_ct_after_exclusion_ML_050426_V(150426).xlsx"),
        ("S12: PNCG5M After Exclusion",          df_pncg5m_incl,     s10_pairs, 'v1', "pncg5m_ct_after_exclusion_ML_050426_V(150426).xlsx"),
        ("S13: PNCG5M After Excl (bf/cf_cat_2)", df_pncg5m_incl,     s13_pairs, 'v1', "BF_CF_Assess_pncg5m_ct_after_exclusion_ML_230426_V(230426).xlsx"),
        ("S14: ANC+PNCL5M After Exclusion",      df_anc_pncl5m_incl, s14_pairs, 'v1', "anc_pncl5m_ct_after_exclusion_ML_230426_V(230426).xlsx"),
        ("S15: Proxy PNC ANC Birthweight",       df_pnc_both_incl,   s15_pairs, 'v1', "BW_proxy_ct_after_primexc_ML_050426_V(150426).xlsx"),
        ("S16a: All Included",                   all_incl,           s16_pairs, 'v1', "all_data_activity_and_outcome_CT_after_exclusion_ML_230426_V(230426).xlsx"),
        ("S16b: ANC+PNCL5M",                     anc_pncl5m_s16,     s16_pairs, 'v1', "anc_pncl5m_activity_and_outcome_CT_after_exclusion_ML_230426_V(230426).xlsx"),
        ("S16c: ANC Only",                       anc_s16,            s16_pairs, 'v1', "anc_activity_and_outcome_CT_after_exclusion_ML_230426_V(230426).xlsx"),
        ("S16d: PNCL5M Only",                    pncl5m_s16,         s16_pairs, 'v1', "pncl5m_activity_and_outcome_CT_after_exclusion_ML_230426_V(230426).xlsx"),
        ("S16e: PNCG5M Only",                    pncg5m_s16,         s16_pairs, 'v1', "pncg5m_activity_and_outcome_CT_after_exclusion_ML_230426_V(230426).xlsx"),
    ]

    print("\n" + "#"*60)
    print("JALNA PIPELINE -> output_jalna (column-wise %) + output_jalna_2 (row-wise %)")
    print("#"*60)

    for _name, _data, _pairs, _gen, _fname in jl_sections:
        _jl_pairs = jalnify_pairs(_pairs)
        _out_name = jl_filename(_fname)
        print(f"\n--- [JALNA] {_name} ---")
        print(f"  Data shape: {_data.shape} | Pairs: {len(_jl_pairs)} | File: {_out_name}")

        # -- column-wise % (twin of `outputs`) --
        _gen_col = generate_specific_crosstabs if _gen == 'v1' else generate_specific_crosstabs_v2
        _res_col = _gen_col(data=_data, crosstab_pairs=_jl_pairs,
                            row_order=row_order, col_order=column_order)
        export_tables_to_single_sheet_to(_res_col, _out_name, OUTPUT_DIR_JALNA)

        # -- row-wise % (twin of `output_2`) --
        _gen_row = generate_specific_crosstabs_rowwise if _gen == 'v1' else generate_specific_crosstabs_v2_rowwise
        _res_row = _gen_row(data=_data, crosstab_pairs=_jl_pairs,
                            row_order=row_order, col_order=column_order)
        export_tables_to_single_sheet_to(_res_row, _out_name, OUTPUT_DIR_JALNA_2)

    # ---- SECTION EXTRA: ANC_60d multi-level crosstab (counts only, identical in
    # both jalna folders; regenerated so each folder holds the complete set) ----
    print("\n--- [JALNA] SECTION EXTRA: ANC_60d by Birthweight ---")
    try:
        _ct = pd.crosstab(
            index=df_anc_incl['birthweight category 2'],
            columns=[df_anc_incl['Learner Category 2'], df_anc_incl['ANC 60d 3visit']],
            margins=False
        )
        _desired = [('MT + FL', 'Yes'), ('MT + FL', 'No'), ('Other', 'Yes'), ('Other', 'No')]
        _ct = _ct[[c for c in _desired if c in _ct.columns]]
        _ct[('Overall', 'Yes')] = _ct.get(('MT + FL', 'Yes'), 0) + _ct.get(('Other', 'Yes'), 0)
        _ct[('Overall', 'No')] = _ct.get(('MT + FL', 'No'), 0) + _ct.get(('Other', 'No'), 0)
        _ct['Total'] = _ct.sum(axis=1)
        _tot = _ct.sum().to_frame().T
        _tot.index = ['Total']
        _ct = pd.concat([_ct, _tot])
        _ct = _ct.reindex(['Less_than_01.5_kg', '01.5_to_02.49_kg', '02.5_to_02.69_kg',
                           '02.7_to_02.99_kg', '03.0_to_03.49_kg', '03.5_or_more_kg',
                           'Total']).fillna(0).astype(int)
        _ct.columns = pd.MultiIndex.from_tuples(_ct.columns)

        _ctg = pd.crosstab(
            df_anc_incl['birthweight group'],
            [df_anc_incl['Learner Category 2'], df_anc_incl['ANC 60d 3visit']],
        )
        _ctg = _ctg[[c for c in _desired if c in _ctg.columns]]
        _ctg[('Overall', 'Yes')] = _ctg.get(('MT + FL', 'Yes'), 0) + _ctg.get(('Other', 'Yes'), 0)
        _ctg[('Overall', 'No')] = _ctg.get(('MT + FL', 'No'), 0) + _ctg.get(('Other', 'No'), 0)
        _ctg['Total'] = _ctg.sum(axis=1)
        _totg = _ctg.sum().to_frame().T
        _totg.index = ['Total']
        _ctg = pd.concat([_ctg, _totg])
        _ctg = _ctg.reindex(['Low Birth Weight', 'Normal Birth Weight', 'Overweight',
                             'Total']).fillna(0).astype(int)
        _ctg.columns = pd.MultiIndex.from_tuples(_ctg.columns)

        def _clean_label_jl(label):
            if not isinstance(label, str):
                return label
            if " vs " in label:
                return " vs ".join(_clean_label_jl(p) for p in label.split(" vs "))
            label = _re_jl.sub(r'_(?:[mMPpCc]|CR|cr|Cr|YN|yn|Yn)$', '', label)
            label = label.replace('_', ' ')
            label = _re_jl.sub(r'\s+', ' ', label)
            return label.strip()

        def _clean_index_jl(idx):
            if isinstance(idx, pd.MultiIndex):
                return pd.MultiIndex.from_tuples(
                    [tuple(_clean_label_jl(x) for x in t) for t in idx],
                    names=[_clean_label_jl(n) if n else n for n in idx.names])
            return pd.Index([_clean_label_jl(x) for x in idx],
                            name=_clean_label_jl(idx.name) if idx.name else idx.name)

        for _t in (_ct, _ctg):
            _t.index = _clean_index_jl(_t.index)
            _t.columns = _clean_index_jl(_t.columns)

        for _dir in (OUTPUT_DIR_JALNA, OUTPUT_DIR_JALNA_2):
            _anc_file = os.path.join(_dir, 'ANC_60d_by_birthweight.xlsx')
            with pd.ExcelWriter(_anc_file, engine='openpyxl') as _writer:
                _ct.to_excel(_writer, sheet_name='BW_Categories')
                _ctg.to_excel(_writer, sheet_name='LBW Normal Overweight')
            print(f"  Exported: {_anc_file} (2 sheets)")
    except Exception as _e:
        print(f"  Warning: Could not generate ANC_60d multi-level crosstab: {_e}")

    print("\n" + "="*60)
    print("JALNA PIPELINE COMPLETE")
    print("="*60)
    print("Column-wise % files ->", OUTPUT_DIR_JALNA)
    print("Row-wise    % files ->", OUTPUT_DIR_JALNA_2)



############################################################
JALNA PIPELINE -> output_jalna (column-wise %) + output_jalna_2 (row-wise %)
############################################################

--- [JALNA] S1: All MCD Before Exclusion ---
  Data shape: (590, 1694) | Pairs: 21 | File: crosstab_all_data_JL_140626_V(170626).xlsx


  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/output_jalna\crosstab_all_data_JL_140626_V(170626).xlsx (15 tables)


  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/output_jalna_2\crosstab_all_data_JL_140626_V(170626).xlsx (15 tables)

--- [JALNA] S2: Included/Excluded All ---
  Data shape: (590, 1694) | Pairs: 50 | File: included_excluded_CT_JL_140626_V(170626).xlsx


  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/output_jalna\included_excluded_CT_JL_140626_V(170626).xlsx (49 tables)


  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/output_jalna_2\included_excluded_CT_JL_140626_V(170626).xlsx (49 tables)

--- [JALNA] S3a: Learners Before Exclusion ---
  Data shape: (192, 1694) | Pairs: 14 | File: learner_based_ct_before_exclusion_JL_140626_V(170626).xlsx


...

  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/output_jalna_2\anc_activity_and_outcome_CT_after_exclusion_JL_140626_V(170626).xlsx (87 tables)

--- [JALNA] S16d: PNCL5M Only ---
  Data shape: (125, 1694) | Pairs: 87 | File: pncl5m_activity_and_outcome_CT_after_exclusion_JL_140626_V(170626).xlsx


  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/output_jalna\pncl5m_activity_and_outcome_CT_after_exclusion_JL_140626_V(170626).xlsx (87 tables)


  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/output_jalna_2\pncl5m_activity_and_outcome_CT_after_exclusion_JL_140626_V(170626).xlsx (87 tables)

--- [JALNA] S16e: PNCG5M Only ---
  Data shape: (89, 1694) | Pairs: 87 | File: pncg5m_activity_and_outcome_CT_after_exclusion_JL_140626_V(170626).xlsx


  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/output_jalna\pncg5m_activity_and_outcome_CT_after_exclusion_JL_140626_V(170626).xlsx (87 tables)


  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/output_jalna_2\pncg5m_activity_and_outcome_CT_after_exclusion_JL_140626_V(170626).xlsx (87 tables)

--- [JALNA] SECTION EXTRA: ANC_60d by Birthweight ---
  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/output_jalna\ANC_60d_by_birthweight.xlsx (2 sheets)
  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/output_jalna_2\ANC_60d_by_birthweight.xlsx (2 sheets)

JALNA PIPELINE COMPLETE
Column-wise % files -> C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/output_jalna
Row-wise    % files -> C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/output_jalna_2


In [24]:
# ==============================================================================
# GESTATIONAL-WEEK MISSING-REASON ADD-ON (situational, append-only)
# Whole-data tables built on 'gestational week classification' cover only the
# few adoptions where BOTH the LMP date and the child's date of birth are
# recorded (classification = weeks between LMP and DOB). This cell flags, for
# EVERY adoption, exactly why the classification is unavailable (LMP missing /
# DOB missing / both / DOB before LMP), so data gaps -- particularly LMP
# capture -- can be corrected early in the project.
# It only WRITES NEW FILES; every existing table, file and folder is untouched.
# Set GENERATE_GW_MISSING_REASON_CT = False to switch the add-on off entirely.
# ==============================================================================
GENERATE_GW_MISSING_REASON_CT = True

if GENERATE_GW_MISSING_REASON_CT:
    import re as _re_gw

    def _gw_present(col):
        return data1[col].notna() if col in data1.columns else pd.Series(False, index=data1.index)

    _gw_lmp = _gw_present('Last Menstrual Period (LMP)_CR') | _gw_present('Last Menstrual Period (LMP)_M')
    _gw_dob = _gw_present('Date of birth of baby_CR') | _gw_present('Date of birth_C')
    _gw_cls = _gw_present('gestational week classification')

    _GW_AVAILABLE    = 'Gestational week available'
    _GW_LMP_MISSING  = 'LMP date not available (DOB available)'
    _GW_DOB_MISSING  = 'DOB not available (LMP available) - child not born yet/child not followed up'
    _GW_BOTH_MISSING = 'Both LMP and DOB not available'
    _GW_IMPLAUSIBLE  = 'LMP & DOB available but DOB before LMP - suggestive of data entry error'

    _gw_col = 'gestational week availability'
    _df_gw = data1.copy()
    _df_gw[_gw_col] = np.select(
        [_gw_cls, _gw_lmp & _gw_dob, (~_gw_lmp) & (~_gw_dob), ~_gw_lmp],
        [_GW_AVAILABLE, _GW_IMPLAUSIBLE, _GW_BOTH_MISSING, _GW_LMP_MISSING],
        default=_GW_DOB_MISSING)

    print("Gestational week availability (all adoptions):")
    print(_df_gw[_gw_col].value_counts().to_string())

    _gw_order = [_GW_AVAILABLE, _GW_LMP_MISSING, _GW_DOB_MISSING,
                 _GW_BOTH_MISSING, _GW_IMPLAUSIBLE]
    _gw_row_order = {**row_order, _gw_col: _gw_order}
    _gw_col_order = {**column_order, _gw_col: _gw_order}

    _gw_pairs = [(_gw_col, 'Included excluded'),
                 (_gw_col, 'Type of adoption')]

    _gw_base = os.path.basename(INPUT_FILE)
    _gw_code = _re_gw.search(r'([A-Za-z]{2}_\d{6})', _gw_base)
    _gw_ver = _re_gw.search(r'(V\(\d{6}\))', _gw_base)
    _gw_fname = ('gestational_week_missing_reason_CT'
                 + (f'_{_gw_code.group(1)}' if _gw_code else '')
                 + (f'_{_gw_ver.group(1)}' if _gw_ver else '') + '.xlsx')

    _gw_res_col = generate_specific_crosstabs(
        data=_df_gw, crosstab_pairs=_gw_pairs,
        row_order=_gw_row_order, col_order=_gw_col_order)
    _gw_res_row = generate_specific_crosstabs_rowwise(
        data=_df_gw, crosstab_pairs=_gw_pairs,
        row_order=_gw_row_order, col_order=_gw_col_order)

    _gw_targets = [(OUTPUT_DIR, _gw_res_col), (OUTPUT_DIR_2, _gw_res_row)]
    if IS_JALNA:
        _gw_targets += [(OUTPUT_DIR_JALNA, _gw_res_col), (OUTPUT_DIR_JALNA_2, _gw_res_row)]
    for _gw_dir, _gw_res in _gw_targets:
        export_tables_to_single_sheet_to(_gw_res, _gw_fname, _gw_dir)


Gestational week availability (all adoptions):
gestational week availability
LMP date not available (DOB available)                                          391
DOB not available (LMP available) - child not born yet/child not followed up    175
Both LMP and DOB not available                                                   16
Gestational week available                                                        7
LMP & DOB available but DOB before LMP - suggestive of data entry error           1


  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/outputs\gestational_week_missing_reason_CT_JL_140626_V(170626).xlsx (2 tables)


  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/output_2\gestational_week_missing_reason_CT_JL_140626_V(170626).xlsx (2 tables)
  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/output_jalna\gestational_week_missing_reason_CT_JL_140626_V(170626).xlsx (2 tables)
  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/output_jalna_2\gestational_week_missing_reason_CT_JL_140626_V(170626).xlsx (2 tables)


In [25]:
# ==============================================================================
# EXCLUSION-DETAIL DIAGNOSTIC ADD-ON (situational, append-only)
# Extracts WHAT DATA IS PRESENT/ABSENT behind the two data-availability
# exclusion reasons, straight from the raw CR/CM fields, so this no longer
# needs one-off manual investigation:
#   'Adoption visit anthropometry data(Wt,Ht,zscore) not available'
#       -> birth details are there, the adoption-visit block was never entered
#          (split by whether birth anthropometry itself is present)
#   'Last visit Date not available'
#       -> last-visit anthropometry is there, only the date cell is blank in
#          the CR sheet; split by whether the CM visit log shows follow-up
#          visits beyond the adoption visit (date recoverable from CM log)
# Every other adoption keeps its original primary label, so each table still
# totals the full dataset. Writes NEW files only; nothing existing changes.
# Set GENERATE_EXCLUSION_DETAIL_CT = False to switch the add-on off entirely.
# ==============================================================================
GENERATE_EXCLUSION_DETAIL_CT = True

if GENERATE_EXCLUSION_DETAIL_CT and 'Primary Exclusion PNC 2' in data1.columns:
    import re as _re_xd

    _xd_primary = 'Primary Exclusion PNC 2'
    _xd_col = 'Primary Exclusion PNC 2 detailed'
    _XD_ANTHRO = 'Adoption visit anthropometry data(Wt,Ht,zscore) not available'
    _XD_LVDATE = 'Last visit Date not available'

    _XD_BIRTH_OK   = 'Birth details available (incl. birth anthropometry); no adoption details available'
    _XD_BIRTH_PART = 'Birth visit recorded but birth anthropometry also missing; no adoption details available'
    _XD_LV_FOLLOW  = 'Last visit date blank in CR sheet; CM log shows follow-up visits after adoption (date recoverable)'
    _XD_LV_ADOPT   = 'Last visit date blank in CR sheet; last visit is the adoption visit itself (date recoverable)'
    _XD_LV_NOCM    = 'Last visit date blank in CR sheet; no dated visit found in CM log'

    # highest dated visit number per row in the CM visit log
    _xd_visit_cols = sorted((c for c in data1.columns if _re_xd.fullmatch(r'Visit Date \d+', c)),
                            key=lambda c: int(c.split()[-1]))
    _xd_last_cm = pd.Series(0, index=data1.index)
    for _c in _xd_visit_cols:
        _xd_last_cm = _xd_last_cm.where(data1[_c].isna(), int(_c.split()[-1]))

    def _xd_ok(col):
        return data1[col].notna() if col in data1.columns else pd.Series(False, index=data1.index)

    _xd_birth_anthro = _xd_ok('Weight 1') & _xd_ok('Height 1')

    _xd_detail = data1[_xd_primary].copy()
    _m = data1[_xd_primary] == _XD_ANTHRO
    _xd_detail[_m & _xd_birth_anthro] = _XD_BIRTH_OK
    _xd_detail[_m & ~_xd_birth_anthro] = _XD_BIRTH_PART
    _m = data1[_xd_primary] == _XD_LVDATE
    _xd_detail[_m & (_xd_last_cm >= 3)] = _XD_LV_FOLLOW
    _xd_detail[_m & (_xd_last_cm == 2)] = _XD_LV_ADOPT
    _xd_detail[_m & (_xd_last_cm < 2)] = _XD_LV_NOCM

    _df_xd = data1.copy()
    _df_xd[_xd_col] = _xd_detail

    print("Exclusion detail derived for the two data-gap reasons:")
    print(_df_xd.loc[data1[_xd_primary].isin([_XD_ANTHRO, _XD_LVDATE]), _xd_col]
          .value_counts().to_string())

    # keep the primary table's row order, with sub-reasons slotted in place
    _xd_expanded = []
    for _x in row_order.get(_xd_primary, []):
        if _x == _XD_ANTHRO:
            _xd_expanded += [_XD_BIRTH_OK, _XD_BIRTH_PART]
        elif _x == _XD_LVDATE:
            _xd_expanded += [_XD_LV_FOLLOW, _XD_LV_ADOPT, _XD_LV_NOCM]
        else:
            _xd_expanded.append(_x)
    _xd_row_order = {**row_order, _xd_col: _xd_expanded}
    _xd_col_order = {**column_order, _xd_col: _xd_expanded}

    _xd_pairs = [(_xd_col, 'Included excluded'),
                 (_xd_col, 'District'),
                 (_xd_col, 'Role Group')]

    _xd_base = os.path.basename(INPUT_FILE)
    _xd_code = _re_xd.search(r'([A-Za-z]{2}_\d{6})', _xd_base)
    _xd_ver = _re_xd.search(r'(V\(\d{6}\))', _xd_base)
    _xd_fname = ('exclusion_reason_detail_CT'
                 + (f'_{_xd_code.group(1)}' if _xd_code else '')
                 + (f'_{_xd_ver.group(1)}' if _xd_ver else '') + '.xlsx')

    _xd_res_col = generate_specific_crosstabs(
        data=_df_xd, crosstab_pairs=_xd_pairs,
        row_order=_xd_row_order, col_order=_xd_col_order)
    _xd_res_row = generate_specific_crosstabs_rowwise(
        data=_df_xd, crosstab_pairs=_xd_pairs,
        row_order=_xd_row_order, col_order=_xd_col_order)

    _xd_targets = [(OUTPUT_DIR, _xd_res_col), (OUTPUT_DIR_2, _xd_res_row)]
    if IS_JALNA:
        _xd_targets += [(OUTPUT_DIR_JALNA, _xd_res_col), (OUTPUT_DIR_JALNA_2, _xd_res_row)]
    for _xd_dir, _xd_res in _xd_targets:
        export_tables_to_single_sheet_to(_xd_res, _xd_fname, _xd_dir)
elif GENERATE_EXCLUSION_DETAIL_CT:
    print("'Primary Exclusion PNC 2' column not found -> exclusion-detail add-on skipped.")


Exclusion detail derived for the two data-gap reasons:
Primary Exclusion PNC 2 detailed
Birth details available (incl. birth anthropometry); no adoption details available                    61
Last visit date blank in CR sheet; CM log shows follow-up visits after adoption (date recoverable)    10
Last visit date blank in CR sheet; last visit is the adoption visit itself (date recoverable)          8
Birth visit recorded but birth anthropometry also missing; no adoption details available               2


  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/outputs\exclusion_reason_detail_CT_JL_140626_V(170626).xlsx (3 tables)


  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/output_2\exclusion_reason_detail_CT_JL_140626_V(170626).xlsx (3 tables)
  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/output_jalna\exclusion_reason_detail_CT_JL_140626_V(170626).xlsx (3 tables)
  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/output_jalna_2\exclusion_reason_detail_CT_JL_140626_V(170626).xlsx (3 tables)


In [26]:
# ==============================================================================
# LAST-VISIT-DATE RECOVERY ADD-ON (situational, append-only)
# For adoptions excluded as 'Last visit Date not available', the last-visit
# anthropometry exists in the CR sheet and the CM visit log holds DATED visits
# -- so the missing date IS recoverable from raw data. This cell extracts, for
# every such case, the latest dated visit in the CM log and writes a per-case
# correction list the data team can act on (and use to fix the source records).
# NOTE: the exclusion cascade and z-scores are computed upstream in the _Exc
# extract, so recovered dates cannot re-include cases inside this notebook;
# the list feeds the correction upstream. Writes NEW files only.
# Set GENERATE_LV_DATE_RECOVERY_LIST = False to switch the add-on off entirely.
# ==============================================================================
GENERATE_LV_DATE_RECOVERY_LIST = True

if GENERATE_LV_DATE_RECOVERY_LIST and 'Primary Exclusion PNC 2' in data1.columns:
    import re as _re_lvr

    _lvr_visit_cols = sorted((c for c in data1.columns if _re_lvr.fullmatch(r'Visit Date \d+', c)),
                             key=lambda c: int(c.split()[-1]))
    _lvr_date = pd.Series(pd.NaT, index=data1.index)
    _lvr_num = pd.Series(0, index=data1.index)
    for _c in _lvr_visit_cols:
        _lvr_date = _lvr_date.where(data1[_c].isna(), data1[_c])
        _lvr_num = _lvr_num.where(data1[_c].isna(), int(_c.split()[-1]))

    _lvr_mask = data1['Primary Exclusion PNC 2'] == 'Last visit Date not available'
    _lvr_cols = [c for c in ['Case ID', 'User Acc ID all', 'User Name all', 'Role Group',
                             'Blocks', 'Mother Adoption date_CR', 'Baby Adoption date_CR',
                             'Date of birth of baby_CR', 'Last Visit Date_CR',
                             'Last Weight_CR', 'Last Height_CR',
                             'Last Weight Zscore_CR', 'Last Height Zscore_CR']
                 if c in data1.columns]
    _lvr = data1.loc[_lvr_mask, _lvr_cols].copy()
    _lvr['Last visit date recovered from CM sheet'] = _lvr_date[_lvr_mask]
    _lvr['Recovered from CM visit number'] = _lvr_num[_lvr_mask]
    _lvr['Follow-up visits beyond adoption in CM log'] = np.where(_lvr_num[_lvr_mask] >= 3, 'Yes', 'No')
    _lvr['Expected status once date corrected'] = np.where(
        _lvr_num[_lvr_mask] >= 3,
        'Re-check for inclusion (real follow-up visits exist)',
        'Only adoption visit occurred (still excluded)')

    print(f"Last-visit-date recovery: {len(_lvr)} case(s); CM date recovered for "
          f"{int(_lvr['Last visit date recovered from CM sheet'].notna().sum())} of them.")

    _lvr_base = os.path.basename(INPUT_FILE)
    _lvr_code = _re_lvr.search(r'([A-Za-z]{2}_\d{6})', _lvr_base)
    _lvr_ver = _re_lvr.search(r'(V\(\d{6}\))', _lvr_base)
    _lvr_fname = ('last_visit_date_recovery_list'
                  + (f'_{_lvr_code.group(1)}' if _lvr_code else '')
                  + (f'_{_lvr_ver.group(1)}' if _lvr_ver else '') + '.xlsx')

    # mutually-exclusive routing: Jalna run -> Jalna folders only; else normal folders
    if MUTUALLY_EXCLUSIVE_OUTPUTS and IS_JALNA:
        _lvr_dirs = [OUTPUT_DIR_JALNA, OUTPUT_DIR_JALNA_2]
    else:
        _lvr_dirs = [OUTPUT_DIR, OUTPUT_DIR_2] + ([OUTPUT_DIR_JALNA, OUTPUT_DIR_JALNA_2] if IS_JALNA else [])
    for _lvr_dir in _lvr_dirs:
        _lvr_path = os.path.join(_lvr_dir, _lvr_fname)
        with pd.ExcelWriter(_lvr_path, engine='openpyxl') as _lvr_w:
            _lvr.to_excel(_lvr_w, sheet_name='LV date recovery', index=False)
        print(f"  Exported: {_lvr_path} ({len(_lvr)} rows)")
elif GENERATE_LV_DATE_RECOVERY_LIST:
    print("'Primary Exclusion PNC 2' column not found -> recovery add-on skipped.")


Last-visit-date recovery: 18 case(s); CM date recovered for 18 of them.
  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/outputs\last_visit_date_recovery_list_JL_140626_V(170626).xlsx (18 rows)
  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/output_2\last_visit_date_recovery_list_JL_140626_V(170626).xlsx (18 rows)
  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/output_jalna\last_visit_date_recovery_list_JL_140626_V(170626).xlsx (18 rows)
  Exported: C:/Users/AayushmanSingh/Desktop/Crosstables individual/Crosstables individual/output_jalna_2\last_visit_date_recovery_list_JL_140626_V(170626).xlsx (18 rows)


In [ ]:
# ==============================================================================
# ANTHRO STATUS CROSSTAB ADD-ON (situational, append-only)
# Requested Jul 2026: WFA / HFA / WFH at all 3 timelines (BV / AV / LV), in both
# column variants -- the 4-category one (Normal / Mild / MUW-MST-MAM /
# SUW-SST-SAM) and the 3-category '... 2' one (Normal / severe pair / Zscore
# not available) -- crossed with the same partner variables Section 16 uses for
# its WFA status tables. EVERY crosstab is written in TWO forms:
#   (1) the full table, built exactly like every other pipeline table;
#   (2) an '(excluding Normal)' twin immediately after it: the Normal row is
#       removed from DISPLAY ONLY -- counts, percentages and the Total row are
#       untouched and stay computed out of the FULL N including Normal
#       (same convention as the existing z_NA display rule).
# Runs on the 5 Section-16 subsets plus one before-exclusion job (all data vs
# 'Included excluded', mirroring Section 2's anthro tables) and writes NEW
# dedicated files (anthro_status_CT_<subset>_<code>_<V>.xlsx) to the same
# folders as everything else (column-wise + row-wise %, Jalna routing incl.).
# Existing tables and files are untouched.
# Set GENERATE_ANTHRO_STATUS_CT = False to switch the add-on off entirely.
# ==============================================================================
GENERATE_ANTHRO_STATUS_CT = True

if GENERATE_ANTHRO_STATUS_CT:
    import re as _re_an

    _AN_INDICATORS = ['WFA', 'HFA', 'WFH']
    _AN_TIMELINES = ['BV', 'AV', 'LV']

    # all 18 status columns: for each indicator+timeline the 4-category plain
    # column first, then its 3-category ' 2' twin
    _an_status_cols = []
    for _ind in _AN_INDICATORS:
        for _tl in _AN_TIMELINES:
            _an_status_cols += [f'{_ind} status at {_tl}', f'{_ind} status at {_tl} 2']

    # partner variables = the ones Section 16 already crosses WFA status with
    # ('District' auto-swaps to 'Blocks' at district level via apply_geo_level)
    _AN_PARTNERS = [
        'District', 'Role Group', 'Member Tag_CR', 'Type of adoption',
        'Adoption type 1', 'Mother Age Group',
        "Mother's education level_M",
        "Color of mother's family ration card_M",
        "Mother's social category_M",
        'gestational week classification', 'adoption age classification',
        'adoption duration baby', 'visit category',
        'bf assessment category', 'Activity Score',
    ]

    # row/column orders for the 4-category columns (the ' 2' ones already exist
    # in row_order); merged locally so the global dicts stay untouched
    _AN_SEV = {'WFA': ('MUW', 'SUW'), 'HFA': ('MST', 'SST'), 'WFH': ('MAM', 'SAM')}
    _an_orders = {}
    for _ind in _AN_INDICATORS:
        for _tl in _AN_TIMELINES:
            _m_cat, _s_cat = _AN_SEV[_ind]
            _an_orders[f'{_ind} status at {_tl}'] = ['Normal', 'Mild', _m_cat, _s_cat]
    _an_row_order = {**row_order, **_an_orders}
    _an_col_order = {**column_order, **_an_orders}

    def _an_with_excl_normal(tables):
        """Interleave: after every table insert its '(excluding Normal)' twin.
        Display-only removal -- counts, % and Totals keep the full N."""
        out = {}
        for _name, _tbl in tables.items():
            out[_name] = _tbl
            if ' vs ' in _name:
                _left, _right = _name.split(' vs ', 1)
                _twin_name = f'{_left} (excluding Normal) vs {_right}'
            else:
                _twin_name = f'{_name} (excluding Normal)'
            out[_twin_name] = _tbl.drop(index=['Normal'], errors='ignore')
        return out

    _an_base = os.path.basename(INPUT_FILE)
    _an_code = _re_an.search(r'([A-Za-z]{2}_\d{6})', _an_base)
    _an_ver = _re_an.search(r'(V\(\d{6}\))', _an_base)

    def _an_fname(tag):
        return ('anthro_status_CT_' + tag
                + (f'_{_an_code.group(1)}' if _an_code else '')
                + (f'_{_an_ver.group(1)}' if _an_ver else '') + '.xlsx')

    # the 5 Section-16 subsets (v1 generators, like Section 16) + the
    # before-exclusion job (v2 generators, like Section 2)
    _an_jobs = [
        ('all_included', all_incl,          'v1', _AN_PARTNERS),
        ('anc_pncl5m',   anc_pncl5m_s16,    'v1', _AN_PARTNERS),
        ('anc',          anc_s16,           'v1', _AN_PARTNERS),
        ('pncl5m',       pncl5m_s16,        'v1', _AN_PARTNERS),
        ('pncg5m',       pncg5m_s16,        'v1', _AN_PARTNERS),
        ('all_data_before_exclusion', data1, 'v2', ['Included excluded']),
    ]

    print("\n" + "#"*60)
    print("ANTHRO STATUS CT ADD-ON (WFA/HFA/WFH x BV/AV/LV, 2 forms each)")
    print("#"*60)

    for _tag, _df_an, _gen_kind, _partners in _an_jobs:
        _an_pairs = [(_sc, _p) for _sc in _an_status_cols for _p in _partners]
        _g_col = generate_specific_crosstabs if _gen_kind == 'v1' else generate_specific_crosstabs_v2
        _g_row = generate_specific_crosstabs_rowwise if _gen_kind == 'v1' else generate_specific_crosstabs_v2_rowwise

        _res_col = _an_with_excl_normal(_g_col(
            data=_df_an, crosstab_pairs=_an_pairs,
            row_order=_an_row_order, col_order=_an_col_order))
        _res_row = _an_with_excl_normal(_g_row(
            data=_df_an, crosstab_pairs=_an_pairs,
            row_order=_an_row_order, col_order=_an_col_order))

        _fname = _an_fname(_tag)
        print(f"\n--- [ANTHRO] {_tag}: data {_df_an.shape} | pairs {len(_an_pairs)} "
              f"| tables {len(_res_col)} -> {_fname}")

        _an_targets = [(OUTPUT_DIR, _res_col), (OUTPUT_DIR_2, _res_row)]
        if IS_JALNA:
            _an_targets += [(OUTPUT_DIR_JALNA, _res_col), (OUTPUT_DIR_JALNA_2, _res_row)]
        for _an_dir, _an_res in _an_targets:
            export_tables_to_single_sheet_to(_an_res, _fname, _an_dir)

    print("\n" + "="*60)
    print("ANTHRO STATUS CT ADD-ON COMPLETE")
    print("="*60)


In [ ]:
# ==============================================================================
# NO-ROLE-GROUP CASE LIST ADD-ON (requested Jul 2026, append-only)
# Companion to ADD-ON DATA STEP 3 (config cell): the cases removed from the
# analysis because NO role / role group is defined are listed here AFTER the
# analysis has ended -- one sheet per level:
#   'No-role learners' : one row per learner (deduplicated on 'User Acc ID all')
#                        with the count of their case rows in the dataset
#   'No-role cases'    : every removed case row with identifying fields
# Writes NEW files only (no existing table / file changes); same folder routing
# as every other add-on (normal folders; Jalna folders on a JL run).
# Set GENERATE_NO_ROLE_GROUP_LIST = False to switch the add-on off entirely.
# ==============================================================================
GENERATE_NO_ROLE_GROUP_LIST = True

if GENERATE_NO_ROLE_GROUP_LIST and EXCLUDE_UNDEFINED_ROLE_GROUP:
    import re as _re_nr

    print("\n" + "="*60)
    print("NO-ROLE-GROUP CASE LIST (cases removed from the analysis)")
    print("="*60)

    _nr_learner_cols = [c for c in [
        'User Acc ID all', 'User Reg Id all', 'User Name all',
        'User Role', 'User Role_CR', 'Role_M', 'Role Group', 'Department',
        'Member Tag_CR', 'Facility/NGO all', 'Phc Taluka_CR', 'Blocks', 'District']
        if c in NO_ROLE_CASES.columns]
    _nr_case_cols = [c for c in [
        'Case ID', 'User Acc ID all', 'User Reg Id all', 'User Name all',
        'User Role', 'User Role_CR', 'Role_M', 'Role Group', 'Department',
        'Member Tag_CR', 'Facility/NGO all', 'Phc Taluka_CR', 'Blocks', 'District',
        'Mother Name all', 'Child Name all', 'Type of adoption',
        'Mother Adoption date_CR', 'Baby Adoption date_CR',
        'Included excluded', 'Primary Exclusion PNC 2']
        if c in NO_ROLE_CASES.columns]

    _nr_cases = NO_ROLE_CASES[_nr_case_cols].copy()

    if 'User Acc ID all' in NO_ROLE_CASES.columns:
        _nr_tmp = NO_ROLE_CASES.copy()
        _nr_tmp['_nr_key'] = _nr_tmp['User Acc ID all'].astype(str)
        _nr_tmp['N case rows in data'] = _nr_tmp.groupby('_nr_key')['_nr_key'].transform('count')
        _nr_learners = (_nr_tmp.drop_duplicates(subset=['_nr_key'])
                        [_nr_learner_cols + ['N case rows in data']].copy())
    else:
        _nr_learners = _nr_cases.copy()

    print(f"Cases removed (no role / role group defined): {len(_nr_cases)} case row(s)"
          f" across {len(_nr_learners)} learner(s)")
    if len(_nr_learners):
        _nr_show = [c for c in ['User Acc ID all', 'User Name all', 'Facility/NGO all',
                                'Blocks', 'District', 'Member Tag_CR', 'N case rows in data']
                    if c in _nr_learners.columns]
        print(_nr_learners[_nr_show].to_string(index=False))

    _nr_base = os.path.basename(INPUT_FILE)
    _nr_code = _re_nr.search(r'([A-Za-z]{2}_\d{6})', _nr_base)
    _nr_ver = _re_nr.search(r'(V\(\d{6}\))', _nr_base)
    _nr_fname = ('no_role_group_case_list'
                 + (f'_{_nr_code.group(1)}' if _nr_code else '')
                 + (f'_{_nr_ver.group(1)}' if _nr_ver else '') + '.xlsx')

    # mutually-exclusive routing: Jalna run -> Jalna folders only; else normal folders
    if MUTUALLY_EXCLUSIVE_OUTPUTS and IS_JALNA:
        _nr_dirs = [OUTPUT_DIR_JALNA, OUTPUT_DIR_JALNA_2]
    else:
        _nr_dirs = [OUTPUT_DIR, OUTPUT_DIR_2] + ([OUTPUT_DIR_JALNA, OUTPUT_DIR_JALNA_2] if IS_JALNA else [])
    for _nr_dir in _nr_dirs:
        _nr_path = os.path.join(_nr_dir, _nr_fname)
        with pd.ExcelWriter(_nr_path, engine='openpyxl') as _nr_w:
            _nr_learners.to_excel(_nr_w, sheet_name='No-role learners', index=False)
            _nr_cases.to_excel(_nr_w, sheet_name='No-role cases', index=False)
        print(f"  Exported: {_nr_path} ({len(_nr_learners)} learner(s), {len(_nr_cases)} case row(s))")
elif GENERATE_NO_ROLE_GROUP_LIST:
    print("EXCLUDE_UNDEFINED_ROLE_GROUP is False -> no-role-group case list skipped.")


In [ ]:
# ==============================================================================
# NO-BLOCK CASE LIST ADD-ON (requested 29-07-2026, append-only)
# Companion to ADD-ON DATA STEP 3 (d) (config cell): the cases removed from the
# analysis because NO block (Phc Taluka_CR) is defined are listed here AFTER the
# analysis has ended -- one sheet per level:
#   'No-block learners' : one row per learner (deduplicated on 'User Acc ID all')
#                         with how many of their case rows had no block and how
#                         many case rows they have in total
#   'No-block cases'    : every removed case row with identifying fields
# Writes NEW files only (no existing table / file changes); same folder routing
# as every other add-on (normal folders; Jalna folders on a JL run).
# Set GENERATE_NO_BLOCK_LIST = False to switch the add-on off entirely.
# ==============================================================================
GENERATE_NO_BLOCK_LIST = True

# Source of the list depends on the mode:
#   EXCLUDE_UNDEFINED_BLOCK = True  -> the cases that were REMOVED (NO_BLOCK_CASES)
#   EXCLUDE_UNDEFINED_BLOCK = False -> the cases still IN the analysis that STILL
#                                      have no block after the MASD fill
if GENERATE_NO_BLOCK_LIST and IS_DISTRICT_LEVEL:
    if EXCLUDE_UNDEFINED_BLOCK:
        NO_BLOCK_CASES = NO_BLOCK_CASES
        _nb_mode = 'removed from the analysis'
    else:
        NO_BLOCK_CASES = data1[data1[BLOCK_COL].astype(str).eq('z_NA')].copy()
        _nb_mode = 'kept in the analysis but still without a block'
    import re as _re_nb

    print("\n" + "="*60)
    print(f"NO-BLOCK CASE LIST (cases {_nb_mode})")
    print("="*60)

    _nb_learner_cols = [c for c in [
        'User Acc ID all', 'User Reg Id all', 'User Name all',
        'User Role', 'User Role_CR', 'Role_M', 'Role Group', 'Department',
        'Member Tag_CR', 'Facility/NGO all', 'Phc Taluka_CR', 'Blocks', 'District']
        if c in NO_BLOCK_CASES.columns]
    _nb_case_cols = [c for c in [
        'Case ID', 'User Acc ID all', 'User Reg Id all', 'User Name all',
        'User Role', 'User Role_CR', 'Role_M', 'Role Group', 'Department',
        'Member Tag_CR', 'Facility/NGO all', 'Phc Taluka_CR', 'Blocks', 'District',
        'Mother Name all', 'Child Name all', 'Type of adoption',
        'Mother Adoption date_CR', 'Baby Adoption date_CR',
        'Included excluded', 'Primary Exclusion PNC 2']
        if c in NO_BLOCK_CASES.columns]

    _nb_cases = NO_BLOCK_CASES[_nb_case_cols].copy()

    if 'User Acc ID all' in NO_BLOCK_CASES.columns:
        _nb_tmp = NO_BLOCK_CASES.copy()
        _nb_tmp['_nb_key'] = _nb_tmp['User Acc ID all'].astype(str)
        _nb_tmp['N case rows with no block'] = _nb_tmp.groupby('_nb_key')['_nb_key'].transform('count')
        _nb_learners = (_nb_tmp.drop_duplicates(subset=['_nb_key'])
                        [_nb_learner_cols + ['N case rows with no block']].copy())
        # how many case rows the learner still has left in the analysis, so it is
        # visible whether the learner dropped out of the analysis altogether
        if 'User Acc ID all' in data1.columns:
            _nb_left = data1['User Acc ID all'].astype(str).value_counts()
            _nb_learners['N case rows still in analysis'] = (
                _nb_learners['User Acc ID all'].astype(str).map(_nb_left).fillna(0).astype(int))
            _nb_learners['Learner dropped from analysis'] = (
                _nb_learners['N case rows still in analysis'].eq(0).map({True: 'Yes', False: 'No'}))
    else:
        _nb_learners = _nb_cases.copy()

    print(f"Cases with no block after the MASD fill: {len(_nb_cases)} case row(s)"
          f" across {len(_nb_learners)} learner(s)")
    if 'Learner dropped from analysis' in _nb_learners.columns:
        print("  Learners dropped from the analysis entirely: "
              f"{int(_nb_learners['Learner dropped from analysis'].eq('Yes').sum())}"
              f" | still present via other cases: "
              f"{int(_nb_learners['Learner dropped from analysis'].eq('No').sum())}")
    if len(_nb_learners):
        _nb_show = [c for c in ['User Acc ID all', 'User Name all', 'Role Group', 'Department',
                                'Facility/NGO all', 'N case rows with no block',
                                'N case rows still in analysis', 'Learner dropped from analysis']
                    if c in _nb_learners.columns]
        print(_nb_learners[_nb_show].to_string(index=False))

    _nb_base = os.path.basename(INPUT_FILE)
    _nb_code = _re_nb.search(r'([A-Za-z]{2}_\d{6})', _nb_base)
    _nb_ver = _re_nb.search(r'(V\(\d{6}\))', _nb_base)
    _nb_fname = ('no_block_case_list'
                 + (f'_{_nb_code.group(1)}' if _nb_code else '')
                 + (f'_{_nb_ver.group(1)}' if _nb_ver else '') + '.xlsx')

    # mutually-exclusive routing: Jalna run -> Jalna folders only; else normal folders
    if MUTUALLY_EXCLUSIVE_OUTPUTS and IS_JALNA:
        _nb_dirs = [OUTPUT_DIR_JALNA, OUTPUT_DIR_JALNA_2]
    else:
        _nb_dirs = [OUTPUT_DIR, OUTPUT_DIR_2] + ([OUTPUT_DIR_JALNA, OUTPUT_DIR_JALNA_2] if IS_JALNA else [])
    for _nb_dir in _nb_dirs:
        _nb_path = os.path.join(_nb_dir, _nb_fname)
        with pd.ExcelWriter(_nb_path, engine='openpyxl') as _nb_w:
            _nb_learners.to_excel(_nb_w, sheet_name='No-block learners', index=False)
            _nb_cases.to_excel(_nb_w, sheet_name='No-block cases', index=False)
        print(f"  Exported: {_nb_path} ({len(_nb_learners)} learner(s), {len(_nb_cases)} case row(s))")
elif GENERATE_NO_BLOCK_LIST:
    print("ANALYSIS_LEVEL is not 'district' -> no-block case list skipped.")


In [ ]:
# ==============================================================================
# REPORT CROSSTABS ADD-ON (requested 04-08-2026, append-only)
# Builds the exact crosstab list the supervisor specified in
#   "Input sheet fro report HST Ujjian to aayushman.xlsx"
# and writes ONE workbook whose TABS are that sheet's short codes
# (BE_LC, BE_MCD, AE_MCD, AE_LC, AE_ANC, AE_PNCBW, AE_ANC ADDNL,
#  AE-PNCL5M, AE-PNCG5M, AE-PNCG5M BFCF), each tab holding only the tables
# listed for that code, in the sheet's own row order.
#
# The spec sheet's columns are: code | source workbook | Row | Column | Included | label
# Only rows with Included == 1 are built. Variable names in the spec use
# underscores ('Mother's_education_level_M'); they are matched to the derived
# sheet's real column names ("Mother's education level_M") by normalising
# underscores, '?' and spacing, so the spec can be written either way.
#
# 'District' is left as written and passes through the pipeline's existing
# apply_geo_level(), which substitutes the Blocks variable at district level --
# this is the spec sheet's note "Wherever district is the variable in MP replace
# it with blocks for ujjain".
#
# Tables are generated ONE PAIR AT A TIME with the pipeline's own generators, so
# formatting, category ordering, z_NA handling and percentages are identical to
# every other output. The 'Learner Category 3' twin add-on is suspended for the
# duration so the supervisor's list is reproduced exactly, with nothing inserted.
#
# Writes NEW files only: column-wise % into OUTPUT_DIR, row-wise % into
# OUTPUT_DIR_2 (Jalna routing respected). Nothing existing is changed.
# Auto-inert: if the spec file is not found the cell prints a note and skips.
# Set GENERATE_REPORT_CROSSTABS = False to switch the add-on off entirely.
# ==============================================================================
GENERATE_REPORT_CROSSTABS = True
REPORT_SPEC_GLOB = "Input sheet*.xlsx"
REPORT_SPEC_SHEET = "Sheet1"

if GENERATE_REPORT_CROSSTABS:
    import glob as _glob_rpt
    import re as _re_rpt
    from collections import OrderedDict as _OD_rpt

    # ---- locate the spec file -------------------------------------------------
    _rpt_search = [os.getcwd(),
                   os.path.dirname(os.path.abspath(str(INPUT_FILE))),
                   os.path.join(os.path.dirname(os.path.abspath(str(INPUT_FILE))), ".."),
                   os.path.join(os.path.dirname(os.path.abspath(str(INPUT_FILE))), "..", "..")]
    _rpt_spec_path = None
    for _d in _rpt_search:
        _hits = sorted(_glob_rpt.glob(os.path.join(_d, REPORT_SPEC_GLOB)))
        if _hits:
            _rpt_spec_path = _hits[0]
            break

    if _rpt_spec_path is None:
        print(f"REPORT CROSSTABS: no file matching '{REPORT_SPEC_GLOB}' found -> add-on skipped.")
    else:
        print("\n" + "#" * 70)
        print("REPORT CROSSTABS ADD-ON")
        print("#" * 70)
        print(f"  Spec file: {_rpt_spec_path}")

        _rpt_spec = pd.read_excel(_rpt_spec_path, sheet_name=REPORT_SPEC_SHEET)
        _rpt_spec.columns = [str(c).strip().lower() for c in _rpt_spec.columns]
        # positional read: code | workbook | row | column | included | label
        _rpt_spec = _rpt_spec.iloc[:, :6]
        _rpt_spec.columns = ["code", "workbook", "row", "col", "included", "label"]
        _rpt_spec = _rpt_spec.dropna(subset=["code", "row", "col"])
        _rpt_spec["code"] = _rpt_spec["code"].astype(str).str.strip()
        _n_all = len(_rpt_spec)
        _rpt_spec = _rpt_spec[pd.to_numeric(_rpt_spec["included"], errors="coerce").fillna(1) == 1]
        print(f"  Spec rows: {_n_all} total, {len(_rpt_spec)} with Included = 1")

        # ---- which analysis subset each code is built on ----------------------
        # (the same DataFrames the corresponding sections of this notebook use)
        _RPT_GROUP_DATA = _OD_rpt([
            ("BE_LC",          ("df_learners",      "learner level, before exclusion")),
            ("BE_MCD",         ("data1",            "all case rows, before exclusion")),
            ("AE_MCD",         ("all_data_s4",      "case rows, after exclusion")),
            ("AE_LC",          ("df_learners_incl", "learner level, after exclusion")),
            ("AE_ANC",         ("df_anc_incl",      "ANC only, after exclusion")),
            ("AE_PNCBW",       ("df_pnc_both_incl", "PNC proxy birthweight, after exclusion")),
            ("AE_ANC ADDNL",   ("anc_s16",          "ANC activity & outcome, after exclusion")),
            ("AE-PNCL5M",      ("df_pncl5m_incl",   "PNCL5M, after exclusion")),
            ("AE-PNCG5M",      ("df_pncg5m_incl",   "PNCG5M, after exclusion")),
            ("AE-PNCG5M BFCF", ("pncg5m_s16",       "PNCG5M activity & outcome, after exclusion")),
        ])

        # ---- spec variable name -> real derived column ------------------------
        def _rpt_norm(name):
            s = str(name).replace("_", " ").replace("?", "")
            return _re_rpt.sub(r"\s+", " ", s).strip().lower()

        def _rpt_resolve(name, columns, _cache={}):
            key = (str(name), id(columns))
            if key in _cache:
                return _cache[key]
            n = str(name).strip()
            hit = None
            if n in columns:
                hit = n
            else:
                target = _rpt_norm(n)
                for c in columns:
                    if _rpt_norm(c) == target:
                        hit = c
                        break
            _cache[key] = hit
            return hit

        # ---- reproduce the supervisor's list EXACTLY: no LC3 twins inserted ---
        _rpt_lc3_saved = LEARNER_CAT3_TWIN
        LEARNER_CAT3_TWIN = False
        try:
            _rpt_groups_col = _OD_rpt()
            _rpt_groups_row = _OD_rpt()
            _rpt_missing = []

            for _code in _RPT_GROUP_DATA:
                _rows = _rpt_spec[_rpt_spec["code"] == _code]
                if _rows.empty:
                    continue
                _var_name, _desc = _RPT_GROUP_DATA[_code]
                _df_rpt = globals().get(_var_name)
                if _df_rpt is None:
                    print(f"  [!] {_code}: source frame '{_var_name}' not defined -> skipped")
                    continue

                _tab_col, _tab_row = _OD_rpt(), _OD_rpt()
                for _, _r in _rows.iterrows():
                    _rv = _rpt_resolve(_r["row"], _df_rpt.columns)
                    _cv = _rpt_resolve(_r["col"], _df_rpt.columns)
                    if _rv is None or _cv is None:
                        _rpt_missing.append((_code, _r["row"], _r["col"],
                                             "row" if _rv is None else "column"))
                        continue
                    _pair = [(_rv, _cv)]
                    _res_c = generate_specific_crosstabs(
                        data=_df_rpt, crosstab_pairs=_pair,
                        row_order=row_order, col_order=column_order)
                    _res_r = generate_specific_crosstabs_rowwise(
                        data=_df_rpt, crosstab_pairs=_pair,
                        row_order=row_order, col_order=column_order)
                    for _src, _dst in ((_res_c, _tab_col), (_res_r, _tab_row)):
                        for _k, _v in _src.items():
                            _kk, _i = _k, 2
                            while _kk in _dst:          # keep every requested table
                                _kk = f"{_k} ({_i})"
                                _i += 1
                            _dst[_kk] = _v

                _rpt_groups_col[_code] = _tab_col
                _rpt_groups_row[_code] = _tab_row
                print(f"  {_code:<16} {_desc:<42} data {_df_rpt.shape[0]:>5} rows | "
                      f"{len(_rows):>3} requested -> {len(_tab_col):>3} tables")

            if _rpt_missing:
                print(f"\n  [!] {len(_rpt_missing)} requested table(s) skipped "
                      f"(variable not present in the derived sheet):")
                for _c, _rw, _cl, _which in _rpt_missing:
                    print(f"      {_c}: {_rw} vs {_cl}  ({_which} variable not found)")
        finally:
            LEARNER_CAT3_TWIN = _rpt_lc3_saved

        # ---- writer: ONE workbook, one tab per code --------------------------
        def _rpt_clean_label(label):
            """Identical to the export-time label cleaning used everywhere else."""
            if not isinstance(label, str):
                return label
            if " vs " in label:
                return " vs ".join(_rpt_clean_label(p) for p in label.split(" vs "))
            label = _re_rpt.sub(r'_(?:[mMPpCc]|CR|cr|Cr|YN|yn|Yn)$', '', label)
            label = label.replace('_', ' ')
            label = _re_rpt.sub(r'\s+', ' ', label)
            label = label.strip()
            return DISPLAY_LABEL_RENAMES.get(label, label)

        def _rpt_export_workbook(groups, file_name, out_dir):
            wb = Workbook()
            wb.remove(wb.active)
            _n_tables = 0
            for _sheet_name, _tables in groups.items():
                ws = wb.create_sheet(title=str(_sheet_name)[:31])
                _cur = 1
                for _tname, _tdf in _tables.items():
                    _tname = _rpt_clean_label(_tname)
                    _tdf = _tdf.copy()
                    _tdf.columns = [_rpt_clean_label(c) for c in _tdf.columns]
                    _tdf.index = [_rpt_clean_label(i) for i in _tdf.index]
                    if _tdf.index.name:
                        _tdf.index.name = _rpt_clean_label(_tdf.index.name)

                    _tc = ws.cell(row=_cur, column=1, value=_tname)
                    _tc.font = Font(name='Calibri', size=24, bold=True)
                    _tc.alignment = Alignment(horizontal='center')
                    _cur += 1

                    _drows = list(dataframe_to_rows(_tdf, index=True, header=True))
                    for _ri, _row in enumerate(_drows):
                        for _ci, _val in enumerate(_row):
                            _cell = ws.cell(row=_cur + _ri, column=_ci + 1, value=_val)
                            _is_hdr = _ri == 0
                            _is_last = _ri == len(_drows) - 1
                            _cell.font = Font(name='Calibri', size=24,
                                              bold=_is_hdr or _is_last)
                            if _ri == 0 or _ci == 0:
                                _cell.alignment = Alignment(horizontal='center')
                            else:
                                _cell.alignment = Alignment(horizontal='right')
                    _cur += len(_drows) + 2
                    _n_tables += 1

                _widths = {}
                for _row in ws.iter_rows():
                    for _cell in _row:
                        if _cell.value:
                            _L = _cell.column_letter
                            _widths[_L] = max(_widths.get(_L, 0),
                                              estimate_column_width(_cell.value, font_size=24))
                for _L, _w in _widths.items():
                    ws.column_dimensions[_L].width = _w

            _path = os.path.join(out_dir, file_name)
            wb.save(_path)
            print(f"  Exported: {_path} ({len(groups)} tabs, {_n_tables} tables)")

        _rpt_fname = ('report_crosstabs'
                      + (f'_{_dyn_code.group(1)}' if _dyn_code else '')
                      + (f'_V({_dyn_ver.group(1)})' if _dyn_ver else '') + '.xlsx')

        if MUTUALLY_EXCLUSIVE_OUTPUTS and IS_JALNA:
            _rpt_targets = [(OUTPUT_DIR_JALNA, _rpt_groups_col),
                            (OUTPUT_DIR_JALNA_2, _rpt_groups_row)]
        else:
            _rpt_targets = [(OUTPUT_DIR, _rpt_groups_col), (OUTPUT_DIR_2, _rpt_groups_row)]
            if IS_JALNA:
                _rpt_targets += [(OUTPUT_DIR_JALNA, _rpt_groups_col),
                                 (OUTPUT_DIR_JALNA_2, _rpt_groups_row)]

        print()
        for _rpt_dir, _rpt_groups in _rpt_targets:
            _rpt_export_workbook(_rpt_groups, _rpt_fname, _rpt_dir)

        print("=" * 70)
        print("REPORT CROSSTABS ADD-ON COMPLETE")
        print("=" * 70)
